# Radar Financeiro — V3_DASHBOARD

Motor funcional preservado da v2 com a experiência final aprovada aplicada exclusivamente à apresentação HTML.


In [ ]:
from traceback import format_exc

try:
    # Gerenciador corporativo local para criação da sessão Spark com suporte a DB2 e Hive
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao='radar-financeiro-v3-dashboard',
        exibir_configuracao=False,
        ativar_logs=True,
    )
    spark = gerenciador_local.criar_sessao_spark(db2=True)
    print('[V3_DASHBOARD] Sessão Spark inicializada com sucesso pelo padrão corporativo.')
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise

### Utilitários Corporativos
Carregamento dos gerenciadores corporativos no kernel local.


In [ ]:
# Carrega conectores e utilitários de ambiente corporativo
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb

---
# Bloco 1 — Configuração Inicial & Tabela Estática CATEGORIAS (Spark Remoto)
Definição de constantes de ambiente, entrada do cliente, limpeza de catálogo e criação da view SQL `vw_categorias`.


In [ ]:
%%spark

import calendar
import datetime
import hashlib
import html
import json
import os
import re
import time
from decimal import Decimal, ROUND_HALF_UP
from datetime import timedelta

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, ShortType,
    StringType, DateType, TimestampType, DecimalType, ArrayType
)
from pyspark.storagelevel import StorageLevel

# --- Nomes das Tabelas Físicas nas Fontes Corporativas ---
FONTE_TRAN = 'DB2GFP.TRAN_RLZD_INST_PCT'   # Movimentações e transações realizadas
FONTE_CICLO = 'DB2GFP.CT_GRDR_FNCO'        # Ciclo e dia de fechamento do balanço da conta
FONTE_RENDA = 'DB2DFE.REN_AVLD_PF'         # Renda presumida e avaliada da pessoa física (Hive)
FONTE_PERFIL = 'DB2D1D.DVS_GRDR_FNCO_PF'   # Perfil financeiro do cliente (Macro/Micro)
VIEW_RESULTADO = 'vw_radar_financeiro_cliente_mvp' # Nome contratual da view temporária publicada

# --- Parâmetros Técnicos de Execução e Limites ---
FETCHSIZE = 10_000                         # Tamanho de lote para leitura DB2
QUERY_TIMEOUT_SECONDS = 900                # Timeout de 15 minutos para consultas DB2
DIAS_CONTEXTO_RECONCILIACAO = 5            # Janela de contexto temporal (±5 dias corridos)
LIMITE_PAYLOAD_BYTES = 2 * 1024 * 1024     # Limite de segurança de 2 MiB para payload HTML

# --- Determinação das Datas da Janela de Formação do Público ---
# DATA_EXECUCAO é obtida da variável de ambiente HOJE (ou data atual)
DATA_EXECUCAO = datetime.date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])

def recuar_um_mes_calendario(data):
    """Calcula o mesmo dia no mês anterior, ajustando para o último dia caso o mês anterior seja mais curto."""
    total = data.year * 12 + data.month - 2
    ano, mes_zero = divmod(total, 12)
    mes = mes_zero + 1
    return datetime.date(ano, mes, min(data.day, calendar.monthrange(ano, mes)[1]))

# Janela de formação do público: 1 mês calendário fechado anterior a DATA_EXECUCAO
DATA_INICIAL_PUBLICO = recuar_um_mes_calendario(DATA_EXECUCAO)
DATA_FINAL_EXCLUSIVA_PUBLICO = DATA_EXECUCAO
DT_MES_EXEA = DATA_EXECUCAO.replace(day=1)

# Inicializa conector JDBC DB2 corporativo
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))

print(f'[V3_DASHBOARD] DATA_EXECUCAO={DATA_EXECUCAO}')
print(f'[V3_DASHBOARD] JANELA_PUBLICO={DATA_INICIAL_PUBLICO} <= TS_INCL_TRAN < {DATA_FINAL_EXCLUSIVA_PUBLICO}')

### Seleção do Cliente e Quantidade de Ciclos
`CD_CLI` tem prioridade. Se estiver `None`, o notebook procura pelo `CPF`; se ambos estiverem `None`, seleciona um cliente elegível aleatoriamente. `periodo` define de 1 a 6 ciclos financeiros fechados e mantém o comportamento original quando vale 1. O bloco de seleção do cliente está delimitado para remoção simples.


In [ ]:
%%spark

# Parâmetros de entrada para o diagnóstico
CD_CLI = None
CPF = None
periodo = 1

def validar_periodo(valor):
    if isinstance(valor, bool) or not isinstance(valor, int):
        raise TypeError('periodo deve ser um número inteiro entre 1 e 6.')
    if not 1 <= valor <= 6:
        raise ValueError('periodo deve estar entre 1 e 6 ciclos fechados.')
    return valor

periodo = validar_periodo(periodo)
print(f'[V3_DASHBOARD] Quantidade de ciclos fechados: {periodo}')

# >>> INÍCIO DO BLOCO TEMPORÁRIO DE SELEÇÃO — remover este trecho para voltar à baseline v2 <<<
if CD_CLI is None:
    if CPF is not None:
        sql_selecao_cliente = f"""
SELECT CD_CLI
FROM {FONTE_TRAN}
WHERE NR_CPF_CNPJ_TITR = {CPF}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
ORDER BY TS_INCL_TRAN DESC
FETCH FIRST 1 ROW ONLY
"""
        CD_CLI = conector_db2.sql(
            sql_selecao_cliente,
            fetchsize=FETCHSIZE,
            query_timeout=QUERY_TIMEOUT_SECONDS,
        ).first()['CD_CLI']
        print(f'[V3_DASHBOARD] CPF={CPF} -> CD_CLI={CD_CLI}')
    else:
        from pyspark.sql.functions import rand
        sql_selecao_cliente = f"""
SELECT DISTINCT CD_CLI
FROM {FONTE_TRAN}
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
"""
        CD_CLI = conector_db2.sql(
            sql_selecao_cliente,
            fetchsize=FETCHSIZE,
            query_timeout=QUERY_TIMEOUT_SECONDS,
        ).orderBy(rand()).first()['CD_CLI']
        print(f'[V3_DASHBOARD] CD_CLI aleatório selecionado: {CD_CLI}')
# <<< FIM DO BLOCO TEMPORÁRIO DE SELEÇÃO >>>

# Validações estritas de tipo e intervalo para evitar injeção ou estouro de tipo físico
if CD_CLI is None:
    raise RuntimeError('BLOQUEADO: informe um único CD_CLI inteiro nesta célula.')
if isinstance(CD_CLI, bool):
    raise TypeError('CD_CLI deve ser inteiro, não booleano.')
texto_cd_cli = str(CD_CLI).strip()
if not re.fullmatch(r'[+-]?[0-9]+', texto_cd_cli):
    raise TypeError('CD_CLI deve possuir representação inteira exata.')
CD_CLI = int(texto_cd_cli)
if not (-2147483648 <= CD_CLI <= 2147483647):
    raise ValueError('CD_CLI não cabe no tipo físico INT (INT32).')

print(f'[V3_DASHBOARD] CD_CLI validado: {CD_CLI}')

In [ ]:
%%spark

# Limpeza preventiva de todas as views temporárias do Radar no catálogo Spark
# Garante que execuções anteriores não deixem resíduos que possam afetar o processamento atual
def limpar_views_temporarias_radar():
    prefixos = [
        'vw_q1_', 'vw_q2_', 'vw_q3_', 'vw_q4_', 'vw_q5_', 'vw_mov_',
        'vw_pares_', 'vw_ids_', 'vw_cliente_', 'vw_ciclo_', 'vw_renda_',
        'vw_perfil_', 'vw_conta_', 'vw_agregacoes_', 'vw_orcamento_',
        'vw_percentuais_', 'vw_pontuacoes_', 'vw_tema_', 'vw_resultado_', 'vw_dashboard_',
        VIEW_RESULTADO
    ]
    for obj in spark.catalog.listTables():
        if obj.isTemporary:
            for p in prefixos:
                if obj.name.startswith(p) or obj.name == p:
                    spark.catalog.dropTempView(obj.name)
                    break

limpar_views_temporarias_radar()
print('[V3_DASHBOARD] Catálogo limpo de views anteriores.')

### Mapa Estático CATEGORIAS Registrado como View SQL
Tabela de referência estática de 70 linhas e 12 atributos que mapeia categoria e natureza contábil para classe temática Radar e regras de participação orçamentária.


In [ ]:
%%spark

# Definição dos 12 atributos e das 70 linhas contratuais de mapeamento de categorias
# Chave de casamento no Spark SQL: (CD_CATEGORIA, TIPO)
COLUNAS_CATEGORIAS = [
    'TIPO', 'CD_GRUPO', 'TX_GRUPO', 'CD_CATEGORIA', 'TX_CATEGORIA',
    'CD_IR', 'TX_IR', 'CD_CLASS_RADAR', 'TX_CLASS_RADAR',
    'IN_AGRO', 'IN_PARTICIPA_CALCULO', 'IN_PARTICIPA_ORCAMENTO'
]

LINHAS_CATEGORIAS = [
    (None, 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    (None, 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'N'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S', 'N'),
]

schema_categorias = StructType([
    StructField('TIPO', StringType(), True),
    StructField('CD_GRUPO', IntegerType(), False),
    StructField('TX_GRUPO', StringType(), False),
    StructField('CD_CATEGORIA', IntegerType(), False),
    StructField('TX_CATEGORIA', StringType(), False),
    StructField('CD_IR', IntegerType(), False),
    StructField('TX_IR', StringType(), False),
    StructField('CD_CLASS_RADAR', IntegerType(), False),
    StructField('TX_CLASS_RADAR', StringType(), False),
    StructField('IN_AGRO', StringType(), False),
    StructField('IN_PARTICIPA_CALCULO', StringType(), False),
    StructField('IN_PARTICIPA_ORCAMENTO', StringType(), False),
])

# Criação e registro da view temporária SQL
df_categorias = spark.createDataFrame(LINHAS_CATEGORIAS, schema_categorias)
df_categorias.createOrReplaceTempView('vw_categorias')

# Validação SQL do Mapa: garante 70 linhas exatas e unicidade da chave composta (TIPO, CD_CATEGORIA)
resumo_cat = spark.sql("""
SELECT 
    COUNT(1) AS QT_TOTAL,
    COUNT(DISTINCT CONCAT(COALESCE(TIPO, 'X'), '-', CD_CATEGORIA)) AS QT_CHAVES_DISTINTAS
FROM vw_categorias
""").first()

if resumo_cat['QT_TOTAL'] != 70 or resumo_cat['QT_CHAVES_DISTINTAS'] != 70:
    raise RuntimeError(f'Mapa CATEGORIAS inválido: total={resumo_cat["QT_TOTAL"]}, chaves_distintas={resumo_cat["QT_CHAVES_DISTINTAS"]}.')

print('[V3_DASHBOARD] View SQL vw_categorias registrada com 70 entradas validadas.')

---
# Bloco 2 — Leituras e Derivações (Cadeia de Dependências em SQL)
Cada query SQL executada inline em seu respectivo lugar, seguida de transformações em Spark SQL.


### Consulta Externa Q1 — Formação do Cliente (DB2)
Recupera os registros de formação do cliente no período de 1 mês fechado. Filtros: `CD_EST_TRAN_INST = 0` (transação realizada), `CD_TIP_PSS = 1` (pessoa física).


In [ ]:
%%spark

# Consulta SQL inline enviada diretamente ao DB2
# Objetivo: identificar se o cliente é elegível na janela de formação e extrair titularidade/contas
sql_q1_db2 = f"""
SELECT
    CD_CLI,
    TS_INCL_TRAN,
    NR_CPF_CNPJ_TITR,
    NR_AG_TITR,
    CD_CT_TITR,
    NR_MCA_PCT_OPB,
    CD_PRD
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
"""

df_q1_raw = conector_db2.sql(sql_q1_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
df_q1_raw = df_q1_raw.persist(StorageLevel.MEMORY_AND_DISK)
df_q1_raw.createOrReplaceTempView('vw_q1_cliente_raw')

qt_q1 = spark.sql('SELECT COUNT(1) AS QT FROM vw_q1_cliente_raw').first()['QT']
if qt_q1 <= 0:
    raise RuntimeError(f'O CD_CLI {CD_CLI} não pertence à janela de formação do público ({DATA_INICIAL_PUBLICO} a {DATA_FINAL_EXCLUSIVA_PUBLICO}).')

print(f'[V3_DASHBOARD] Q1 executada no DB2: {qt_q1} registros na view vw_q1_cliente_raw.')

### Derivações Q1 em SQL — Cliente, CPF e Conta Elegível


In [ ]:
%%spark

# 1. Derivação do Timestamp de Referência e Unicidade do CPF
# - TS_INCL_TRAN_REF: maior timestamp de inclusão de transação no período de formação
# - FL_CPF_UNICO = 'S' se e somente se houver exatamente 1 CPF distinto associado às transações
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_cliente_derivado AS
SELECT
    CD_CLI,
    MAX(TS_INCL_TRAN) AS TS_INCL_TRAN_REF,
    CASE WHEN COUNT(DISTINCT NR_CPF_CNPJ_TITR) = 1 THEN 'S' ELSE 'N' END AS FL_CPF_UNICO,
    CASE WHEN COUNT(DISTINCT NR_CPF_CNPJ_TITR) = 1 THEN CAST(MAX(NR_CPF_CNPJ_TITR) AS DECIMAL(14,0)) ELSE NULL END AS CD_CPF
FROM vw_q1_cliente_raw
GROUP BY CD_CLI
""")

# 2. Identificação das Contas Correntes Elegíveis do Cliente
# - NR_MCA_PCT_OPB = 999999999 (pacote operacional de conta corrente padrão)
# - CD_PRD = 6 (código de produto correspondente a conta corrente ativa)
# - Agência e conta preenchidas e não vazias
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_contas_elegiveis_q1 AS
SELECT
    NR_AG_TITR,
    CD_CT_TITR,
    COUNT(1) AS QT_OCORRENCIAS
FROM vw_q1_cliente_raw
WHERE NR_MCA_PCT_OPB = 999999999
  AND CD_PRD = 6
  AND NR_AG_TITR IS NOT NULL
  AND CD_CT_TITR IS NOT NULL
  AND TRIM(CAST(CD_CT_TITR AS STRING)) != ''
GROUP BY NR_AG_TITR, CD_CT_TITR
""")

# 3. Resumo da Conta Elegível
# - FL_CONTA_ELEGIVEL_UNICA = 'S' se houver exatamente 1 conta corrente distinta elegível
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_conta_elegivel_resumo AS
SELECT
    CASE WHEN COUNT(1) = 1 THEN 'S' ELSE 'N' END AS FL_CONTA_ELEGIVEL_UNICA,
    CASE WHEN COUNT(1) = 1 THEN MAX(NR_AG_TITR) ELSE NULL END AS NR_AG_TITR,
    CASE WHEN COUNT(1) = 1 THEN MAX(CD_CT_TITR) ELSE NULL END AS CD_CT_TITR
FROM vw_contas_elegiveis_q1
""")

res_cli = spark.sql('SELECT * FROM vw_cliente_derivado').first()
res_cta = spark.sql('SELECT * FROM vw_conta_elegivel_resumo').first()

# Liberação de memória de Q1 logo após a extração das variáveis essenciais
df_q1_raw.unpersist()

print(f'[V3_DASHBOARD] Q1 Derivações SQL: TS_REF={res_cli["TS_INCL_TRAN_REF"]}, CPF_UNICO={res_cli["FL_CPF_UNICO"]}, CONTA_UNICA={res_cta["FL_CONTA_ELEGIVEL_UNICA"]}')

### Normalização da Conta Elegível em SQL


In [ ]:
%%spark

def criar_view_conta_normalizada_v6(view_origem, view_destino):
    """Normaliza a chave da conta aplicando TRIM antes de validar e converter."""
    for nome in (view_origem, view_destino):
        if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', nome):
            raise ValueError(f'Nome de view inválido para normalização V6: {nome!r}.')

    spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW {view_destino} AS
    WITH conta_textual AS (
        SELECT
            FL_CONTA_ELEGIVEL_UNICA,
            NR_AG_TITR,
            CD_CT_TITR,
            TRIM(CAST(NR_AG_TITR AS STRING)) AS NR_AG_TITR_TXT,
            TRIM(CAST(CD_CT_TITR AS STRING)) AS CD_CT_TITR_TXT
        FROM {view_origem}
    ), conta_significativa AS (
        SELECT
            *,
            LTRIM('0', CD_CT_TITR_TXT) AS NR_CC_SIGNIFICATIVA
        FROM conta_textual
    )
    SELECT
        FL_CONTA_ELEGIVEL_UNICA,
        NR_AG_TITR,
        CD_CT_TITR,
        CASE
            WHEN FL_CONTA_ELEGIVEL_UNICA = 'S'
             AND NR_AG_TITR_TXT RLIKE '^[0-9]+$'
             AND CD_CT_TITR_TXT RLIKE '^[0-9]+$'
             AND CAST(NR_AG_TITR_TXT AS BIGINT) BETWEEN -2147483648 AND 2147483647
             AND LENGTH(NR_CC_SIGNIFICATIVA) <= 11
            THEN CAST(NR_AG_TITR_TXT AS INT)
            ELSE NULL
        END AS CD_UOR_CC_NORM,
        CASE
            WHEN FL_CONTA_ELEGIVEL_UNICA = 'S'
             AND NR_AG_TITR_TXT RLIKE '^[0-9]+$'
             AND CD_CT_TITR_TXT RLIKE '^[0-9]+$'
             AND CAST(NR_AG_TITR_TXT AS BIGINT) BETWEEN -2147483648 AND 2147483647
             AND LENGTH(NR_CC_SIGNIFICATIVA) <= 11
            THEN CAST(
                CASE
                    WHEN NR_CC_SIGNIFICATIVA = '' THEN '0'
                    ELSE NR_CC_SIGNIFICATIVA
                END AS DECIMAL(11,0)
            )
            ELSE NULL
        END AS NR_CC_NORM
    FROM conta_significativa
    """)


criar_view_conta_normalizada_v6(
    'vw_conta_elegivel_resumo',
    'vw_conta_normalizada',
)

cta_norm_row = spark.sql('SELECT * FROM vw_conta_normalizada').first()
tem_conta_norm = cta_norm_row['CD_UOR_CC_NORM'] is not None and cta_norm_row['NR_CC_NORM'] is not None

print(f'[V3_DASHBOARD] Conta normalizada em SQL: UOR={cta_norm_row["CD_UOR_CC_NORM"]}, NR_CC={cta_norm_row["NR_CC_NORM"]}')


### Consulta Externa Q2 — Ciclo do Cartão/Conta (DB2)
Recupera o histórico do dia de fechamento do balanço da conta corrente normalizada em `DB2GFP.CT_GRDR_FNCO`.


In [ ]:
%%spark

# Executa Q2 no DB2 somente se houver uma conta corrente única devidamente normalizada
if not tem_conta_norm:
    schema_ciclo_raw = StructType([
        StructField('CD_UOR_CC', IntegerType(), True),
        StructField('NR_CC', DecimalType(11, 0), True),
        StructField('DD_INC_MM_CLC_BLC', ShortType(), True),
        StructField('TS_ULT_EXEA_PSQ', TimestampType(), True),
    ])
    df_q2_raw = spark.createDataFrame([], schema_ciclo_raw)
    df_q2_raw.createOrReplaceTempView('vw_q2_ciclo_raw')
    print('[V3_DASHBOARD] Q2: SKIPPED (conta normalizada indisponível).')
else:
    uor_val = cta_norm_row['CD_UOR_CC_NORM']
    nr_val = cta_norm_row['NR_CC_NORM']
    sql_q2_db2 = f"""
SELECT
    CD_UOR_CC,
    NR_CC,
    DD_INC_MM_CLC_BLC,
    TS_ULT_EXEA_PSQ
FROM {FONTE_CICLO}
WHERE CD_UOR_CC = {uor_val}
  AND NR_CC = {nr_val}
"""
    df_q2_raw = conector_db2.sql(sql_q2_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
    df_q2_raw.createOrReplaceTempView('vw_q2_ciclo_raw')
    qt_q2 = spark.sql('SELECT COUNT(1) AS QT FROM vw_q2_ciclo_raw').first()['QT']
    print(f'[V3_DASHBOARD] Q2 executada no DB2: {qt_q2} registros na view vw_q2_ciclo_raw.')

### Derivações Q2 em SQL — Ciclo, Fallback e Janela Financeira


In [ ]:
%%spark

# 1. Seleção do Ciclo Mais Recente via Window Function em SQL
# Desempate estrito por TS_ULT_EXEA_PSQ DESC
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ciclo_selecionado AS
WITH rankeado AS (
    SELECT
        CD_UOR_CC,
        NR_CC,
        DD_INC_MM_CLC_BLC,
        TS_ULT_EXEA_PSQ,
        ROW_NUMBER() OVER (PARTITION BY CD_UOR_CC, NR_CC ORDER BY TS_ULT_EXEA_PSQ DESC) AS RN
    FROM vw_q2_ciclo_raw
)
SELECT
    TS_ULT_EXEA_PSQ AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC
FROM rankeado
WHERE RN = 1
""")

# 2. Resolução do Dia de Ciclo com Fallback
# - Se o cliente não possui conta única: DD_INC_MM_CLC_BLC_FALLBACK = NULL
# - Se possui conta única mas não há registro na fonte Q2: assume Fallback = 1
# - Se há registro na fonte: assume o valor do dia retornado
row_cli_ts = spark.sql('SELECT TS_INCL_TRAN_REF FROM vw_cliente_derivado').first()['TS_INCL_TRAN_REF']
row_ciclo = spark.sql('SELECT * FROM vw_ciclo_selecionado').first()

dd_ciclo_val = row_ciclo['DD_INC_MM_CLC_BLC'] if row_ciclo else None
if not tem_conta_norm:
    dd_fallback_val = None
elif dd_ciclo_val is None:
    dd_fallback_val = 1
else:
    dd_fallback_val = int(dd_ciclo_val)

# 3. Cálculo da Janela Financeira Fechada [DT_REF_INI, DT_REF_FIM]
# DT_REF_FIM permanece no último dia anterior ao ciclo aberto atual.
# DT_REF_INI recua a quantidade de ciclos fechados definida em periodo.
def calcular_inicio_periodo_fechado(inicio_ciclo_aberto, dia_ciclo, quantidade_ciclos):
    mes_alvo_total = inicio_ciclo_aberto.year * 12 + inicio_ciclo_aberto.month - 1 - quantidade_ciclos
    ano_alvo, mes_alvo_zero = divmod(mes_alvo_total, 12)
    mes_alvo = mes_alvo_zero + 1
    dia_alvo = min(dia_ciclo, calendar.monthrange(ano_alvo, mes_alvo)[1])
    return datetime.date(ano_alvo, mes_alvo, dia_alvo)

if dd_fallback_val is None:
    dt_ref_ini_val = None
    dt_ref_fim_val = None
else:
    if not (1 <= dd_fallback_val <= 31):
        raise RuntimeError(f'Dia de ciclo fora do domínio 1..31: {dd_fallback_val}.')
    ts_ref = row_cli_ts
    dia_mes_ref = min(dd_fallback_val, calendar.monthrange(ts_ref.year, ts_ref.month)[1])
    candidato = datetime.datetime(ts_ref.year, ts_ref.month, dia_mes_ref)
    if ts_ref >= candidato:
        inicio_aberto = candidato.date()
    else:
        total = ts_ref.year * 12 + ts_ref.month - 2
        ano_ant, mes_ant_zero = divmod(total, 12)
        mes_ant = mes_ant_zero + 1
        inicio_aberto = datetime.date(ano_ant, mes_ant, min(dd_fallback_val, calendar.monthrange(ano_ant, mes_ant)[1]))
    dt_ref_fim_val = inicio_aberto - timedelta(days=1)
    dt_ref_ini_val = calcular_inicio_periodo_fechado(inicio_aberto, dd_fallback_val, periodo)

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_ciclo_janela AS
SELECT
    CAST('{row_ciclo['TS_DD_INC_MM_CLC_BLC_REF']}' AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST({('NULL' if dd_ciclo_val is None else dd_ciclo_val)} AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST({('NULL' if dd_fallback_val is None else dd_fallback_val)} AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK,
    CAST({('NULL' if dt_ref_ini_val is None else f"'{dt_ref_ini_val.isoformat()}'")} AS DATE) AS DT_REF_INI,
    CAST({('NULL' if dt_ref_fim_val is None else f"'{dt_ref_fim_val.isoformat()}'")} AS DATE) AS DT_REF_FIM
""" if row_ciclo else f"""
CREATE OR REPLACE TEMPORARY VIEW vw_ciclo_janela AS
SELECT
    CAST(NULL AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST(NULL AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST({('NULL' if dd_fallback_val is None else dd_fallback_val)} AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK,
    CAST({('NULL' if dt_ref_ini_val is None else f"'{dt_ref_ini_val.isoformat()}'")} AS DATE) AS DT_REF_INI,
    CAST({('NULL' if dt_ref_fim_val is None else f"'{dt_ref_fim_val.isoformat()}'")} AS DATE) AS DT_REF_FIM
""")

res_janela = spark.sql('SELECT * FROM vw_ciclo_janela').first()
print(f'[V3_DASHBOARD] Janela Financeira SQL: periodo={periodo}; DT_REF_INI={res_janela["DT_REF_INI"]} .. DT_REF_FIM={res_janela["DT_REF_FIM"]}')

### Consulta Externa Q3 — Renda Presumida (Hive)
Recupera a série histórica de renda avaliada do CPF único na tabela Hive `DB2DFE.REN_AVLD_PF`.


In [ ]:
%%spark

cpf_row = spark.sql('SELECT FL_CPF_UNICO, CD_CPF FROM vw_cliente_derivado').first()

# Executa Q3 no Hive somente se o CPF for estritamente único
if cpf_row['FL_CPF_UNICO'] != 'S' or cpf_row['CD_CPF'] is None:
    schema_q3 = StructType([
        StructField('NR_CPF', DecimalType(11, 0), True),
        StructField('DT_INCL_REN_AVLD', DateType(), True),
        StructField('VL_REN', DecimalType(17, 2), True),
    ])
    df_q3_raw = spark.createDataFrame([], schema_q3)
    df_q3_raw.createOrReplaceTempView('vw_q3_renda_raw')
    print('[V3_DASHBOARD] Q3: SKIPPED (CPF único indisponível).')
else:
    cpf_num = int(cpf_row['CD_CPF'])
    sql_q3_hive = f"""
SELECT
    NR_CPF_BASE_SRF AS NR_CPF,
    DT_INCL_REN_AVLD,
    VL_REN
FROM {FONTE_RENDA}
WHERE NR_CPF_BASE_SRF = {cpf_num}
"""
    df_q3_raw = spark.sql(sql_q3_hive)
    df_q3_raw.createOrReplaceTempView('vw_q3_renda_raw')
    qt_q3 = spark.sql('SELECT COUNT(1) AS QT FROM vw_q3_renda_raw').first()['QT']
    print(f'[V3_DASHBOARD] Q3 executada no Hive: {qt_q3} registros na view vw_q3_renda_raw.')

### Derivações Q3 em SQL — Seleção na Maior Data de Renda


In [ ]:
%%spark

# Seleção da renda mais recente associada ao CPF único
# Ordenação estrita por DT_INCL_REN_AVLD DESC
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_renda_derivada AS
WITH rankeado AS (
    SELECT
        CAST(DT_INCL_REN_AVLD AS DATE) AS DT_REN_PRES_REF,
        CAST(VL_REN * {periodo} AS DECIMAL(17,2)) AS VL_REN_PRES,
        ROW_NUMBER() OVER (PARTITION BY NR_CPF ORDER BY DT_INCL_REN_AVLD DESC) AS RN
    FROM vw_q3_renda_raw
)
SELECT
    DT_REN_PRES_REF,
    VL_REN_PRES
FROM rankeado
WHERE RN = 1
""")

res_renda = spark.sql('SELECT * FROM vw_renda_derivada').first()
if res_renda is None:
    spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW vw_renda_derivada AS
    SELECT CAST(NULL AS DATE) AS DT_REN_PRES_REF, CAST(NULL AS DECIMAL(17,2)) AS VL_REN_PRES
    """)
    res_renda = spark.sql('SELECT * FROM vw_renda_derivada').first()

print(f'[V3_DASHBOARD] Renda Selecionada SQL: DT_REF={res_renda["DT_REN_PRES_REF"]}, VL_REN={res_renda["VL_REN_PRES"]}')

### Consulta Externa Q4 — Perfil Financeiro (DB2)
Recupera o histórico de macro/microperfil do cliente em `DB2D1D.DVS_GRDR_FNCO_PF` com corte até `DATA_EXECUCAO`.


In [ ]:
%%spark

# Consulta SQL inline no DB2 para o perfil do cliente
# Filtro de corte temporal: DT_REF <= DATA_EXECUCAO (garante estabilidade temporal)
sql_q4_db2 = f"""
SELECT
    CD_CLI,
    DT_REF,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI
FROM {FONTE_PERFIL}
WHERE CD_CLI = {CD_CLI}
  AND DT_REF <= DATE('{DATA_EXECUCAO.isoformat()}')
"""

df_q4_raw = conector_db2.sql(sql_q4_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
df_q4_raw.createOrReplaceTempView('vw_q4_perfil_raw')

qt_q4 = spark.sql('SELECT COUNT(1) AS QT FROM vw_q4_perfil_raw').first()['QT']
print(f'[V3_DASHBOARD] Q4 executada no DB2: {qt_q4} registros na view vw_q4_perfil_raw.')

### Derivações Q4 em SQL — Seleção do Perfil na Maior Data Elegível


In [ ]:
%%spark

# Seleção do perfil do cliente na data mais recente elegível (MAX(DT_REF) <= DATA_EXECUCAO)
# Validação: se houver mais de um perfil na mesma data máxima, bloqueia por ambiguidade
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_perfil_derivado AS
WITH max_data AS (
    SELECT MAX(DT_REF) AS MAX_DT_REF FROM vw_q4_perfil_raw
),
elegiveis AS (
    SELECT p.*
    FROM vw_q4_perfil_raw p
    INNER JOIN max_data m ON p.DT_REF = m.MAX_DT_REF
)
SELECT
    DT_REF AS DT_REF_PRFL,
    CAST(CD_MAC_PRFL_CLI AS INT) AS CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CAST(CD_MIC_PRFL_CLI AS INT) AS CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI
FROM elegiveis
""")

linhas_perfil = spark.sql('SELECT * FROM vw_perfil_derivado').collect()
if len(linhas_perfil) > 1:
    raise RuntimeError('Q4 retornou mais de uma linha na maior DT_REF elegível.')
if len(linhas_perfil) == 1 and linhas_perfil[0]['DT_REF_PRFL'] > DATA_EXECUCAO:
    raise RuntimeError(f'DT_REF_PRFL posterior à DATA_EXECUCAO: {linhas_perfil[0]["DT_REF_PRFL"]} > {DATA_EXECUCAO}.')

if len(linhas_perfil) == 0:
    spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW vw_perfil_derivado AS
    SELECT
        CAST(NULL AS DATE) AS DT_REF_PRFL,
        CAST(NULL AS INT) AS CD_MAC_PRFL_CLI,
        CAST(NULL AS STRING) AS NM_MAC_PRFL_CLI,
        CAST(NULL AS INT) AS CD_MIC_PRFL_CLI,
        CAST(NULL AS STRING) AS NM_MIC_PRFL_CLI
    """)
    res_prfl = spark.sql('SELECT * FROM vw_perfil_derivado').first()
else:
    res_prfl = linhas_perfil[0]

print(f'[V3_DASHBOARD] Perfil Selecionado SQL: CD_MAC={res_prfl["CD_MAC_PRFL_CLI"]} ({res_prfl["NM_MAC_PRFL_CLI"]})')

### Consulta Externa Q5 — Movimentações no Contexto ±5 Dias (DB2)
Recupera as transações do cliente no intervalo estendido `[DT_REF_INI - 5d, DT_REF_FIM + 5d]` para permitir a reconciliação das bordas temporais.


In [ ]:
%%spark

inicio_q5_periodo = time.perf_counter()

janela_row = spark.sql('SELECT DT_REF_INI, DT_REF_FIM FROM vw_ciclo_janela').first()
dt_ini_j = janela_row['DT_REF_INI']
dt_fim_j = janela_row['DT_REF_FIM']

if (dt_ini_j is None) != (dt_fim_j is None):
    raise RuntimeError('DT_REF_INI e DT_REF_FIM devem existir conjuntamente.')

# Contrato funcional da Q5 consumido pelo motor da v2. Nomes, ordem e tipos
# físicos são bloqueantes; nullable e metadata pertencem ao transporte JDBC.
COLUNAS_Q5_FUNCIONAIS = [
    'NR_TRAN_INST_PCT',
    'CD_CLI',
    'DT_TRAN',
    'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL',
    'CD_TIP_MOE_CRR',
    'VL_TRAN',
]

SCHEMA_Q5_FUNCIONAL_V2 = StructType([
    StructField('NR_TRAN_INST_PCT', LongType(), True),
    StructField('CD_CLI', IntegerType(), True),
    StructField('DT_TRAN', DateType(), True),
    StructField('CD_NTZ_CTB_TRAN', StringType(), True),
    StructField('CD_CTGR_TRAN_OGNL', IntegerType(), True),
    StructField('CD_TIP_MOE_CRR', StringType(), True),
    StructField('VL_TRAN', DecimalType(15, 2), True),
])

# A consulta externa continua sendo a Q5 original, com os mesmos filtros. As
# duas colunas adicionais existem somente na sidecar de apresentação.
if dt_ini_j is None:
    schema_q5_apresentacao = StructType(
        list(SCHEMA_Q5_FUNCIONAL_V2.fields) + [
            StructField('TX_DCR_TRAN_OGNL', StringType(), True),
            StructField('NR_MCA_PCT_OPB', StringType(), True),
        ]
    )
    df_q5_contexto_apresentacao = spark.createDataFrame([], schema_q5_apresentacao)
    qt_q5 = 0
    print('[V3_DASHBOARD] Q5: SKIPPED (janela financeira indisponível).')
else:
    dt_q5_ini = (dt_ini_j - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)).isoformat()
    dt_q5_fim = (dt_fim_j + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)).isoformat()
    sql_q5_db2 = f"""
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    DT_TRAN,
    CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL,
    CD_TIP_MOE_CRR,
    VL_TRAN,
    TX_DCR_TRAN_OGNL,
    NR_MCA_PCT_OPB
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND DT_TRAN >= DATE('{dt_q5_ini}')
  AND DT_TRAN <= DATE('{dt_q5_fim}')
  AND (
      CD_NTZ_CTB_TRAN = 'C'
      OR (CD_NTZ_CTB_TRAN = 'D' AND IN_VSLO_CSM = 'S')
  )
"""
    df_q5_contexto_apresentacao = conector_db2.sql(
        sql_q5_db2,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT_SECONDS,
    ).persist(StorageLevel.MEMORY_AND_DISK)
    qt_q5 = df_q5_contexto_apresentacao.count()
    print(f'[V3_DASHBOARD] Q5 executada no DB2: {qt_q5} registros.')

df_q5_contexto_apresentacao.createOrReplaceTempView('vw_q5_mov_contexto_apresentacao')

# Fronteira rígida: o motor recebe exatamente a projeção funcional da v2.
df_q5_contexto = df_q5_contexto_apresentacao.select(*COLUNAS_Q5_FUNCIONAIS)
df_q5_contexto.createOrReplaceTempView('vw_q5_mov_contexto')

def assinatura_estrutural_q5(schema):
    """Compara nomes, ordem e tipos, isolando detalhes variáveis do JDBC."""
    return [
        (campo.name, campo.dataType.jsonValue())
        for campo in schema.fields
    ]


schema_q5_obtido = df_q5_contexto.schema
assinatura_q5_obtida = assinatura_estrutural_q5(schema_q5_obtido)
assinatura_q5_esperada = assinatura_estrutural_q5(SCHEMA_Q5_FUNCIONAL_V2)
if assinatura_q5_obtida != assinatura_q5_esperada:
    raise RuntimeError(
        'BLOQUEADO: a projeção funcional da Q5 divergiu da v2. '
        f'obtido={schema_q5_obtido.json()}; '
        f'esperado={SCHEMA_Q5_FUNCIONAL_V2.json()}.'
    )

if df_q5_contexto.columns != COLUNAS_Q5_FUNCIONAIS:
    raise RuntimeError('BLOQUEADO: nomes ou ordem da projeção funcional da Q5 foram alterados.')

tempo_q5_periodo_seg = time.perf_counter() - inicio_q5_periodo
print(f'[V3_DASHBOARD][PERFORMANCE_Q5] periodo={periodo}; linhas={qt_q5}; tempo_seg={tempo_q5_periodo_seg:.6f}')


---
# Bloco 3 — Transformações Financeiras em Spark SQL Puro
Marcação da janela, reconciliação em SQL, classificação com CATEGORIAS, agregações, orçamento e pontuações.


In [ ]:
%%spark

# 1. Marcação da Flag IN_JANELA
# - 'S': transação realizada dentro da janela oficial [DT_REF_INI, DT_REF_FIM]
# - 'N': transação realizada na borda temporal externa (contexto ±5 dias)
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_marcado AS
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    DT_TRAN,
    CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL,
    CD_TIP_MOE_CRR,
    VL_TRAN,
    CASE 
        WHEN DT_TRAN >= DATE('{dt_ini_j.isoformat() if dt_ini_j else '1900-01-01'}') 
         AND DT_TRAN <= DATE('{dt_fim_j.isoformat() if dt_fim_j else '1900-01-01'}') 
        THEN 'S' 
        ELSE 'N' 
    END AS IN_JANELA
FROM vw_q5_mov_contexto
""")

# 2. Universo Oficial Inicial (vw_mov_raw)
# Contém todas as transações oficiais antes de qualquer anulação por reconciliação
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_raw AS
SELECT * FROM vw_mov_marcado WHERE IN_JANELA = 'S'
""")

qt_raw = spark.sql('SELECT COUNT(1) AS QT FROM vw_mov_raw').first()['QT']
print(f'[V3_DASHBOARD] Universo oficial na janela (vw_mov_raw): {qt_raw} transações.')

### Reconciliação em SQL — Pares Exatos (Mesma Data, Valor, Moeda e Naturezas Opostas)


In [ ]:
%%spark

# 1. Identificação dos Pares Exatos Candidatos
# Anula créditos e débitos idênticos na mesma data: número de pares = LEAST(QT_C, QT_D)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pares_exatos_candidatos AS
SELECT
    CD_CLI,
    DT_TRAN,
    VL_TRAN,
    CD_TIP_MOE_CRR,
    IN_JANELA,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_C,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_D,
    LEAST(
        SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END),
        SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END)
    ) AS QT_PARES_EXATOS
FROM vw_mov_marcado
WHERE NR_TRAN_INST_PCT IS NOT NULL
  AND CD_NTZ_CTB_TRAN IN ('C', 'D')
  AND DT_TRAN IS NOT NULL
  AND VL_TRAN IS NOT NULL
  AND CD_TIP_MOE_CRR IS NOT NULL
GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
HAVING QT_PARES_EXATOS > 0
""")

# 2. Seleção dos IDs Consumidos por Par Exato
# Consome os primeiros IDs em ordem crescente (NR_TRAN_INST_PCT ASC) para cada natureza
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_exatos AS
WITH rankeado AS (
    SELECT
        m.NR_TRAN_INST_PCT,
        m.IN_JANELA,
        m.CD_NTZ_CTB_TRAN,
        p.QT_PARES_EXATOS,
        ROW_NUMBER() OVER (
            PARTITION BY m.CD_CLI, m.DT_TRAN, m.VL_TRAN, m.CD_TIP_MOE_CRR, m.IN_JANELA, m.CD_NTZ_CTB_TRAN
            ORDER BY m.NR_TRAN_INST_PCT ASC
        ) AS RN
    FROM vw_mov_marcado m
    INNER JOIN vw_pares_exatos_candidatos p
       ON m.CD_CLI = p.CD_CLI
      AND m.DT_TRAN = p.DT_TRAN
      AND m.VL_TRAN = p.VL_TRAN
      AND m.CD_TIP_MOE_CRR = p.CD_TIP_MOE_CRR
      AND m.IN_JANELA = p.IN_JANELA
    WHERE m.CD_NTZ_CTB_TRAN IN ('C', 'D')
)
SELECT
    NR_TRAN_INST_PCT,
    CASE WHEN IN_JANELA = 'S' THEN 'EXATO_OFICIAL' ELSE 'EXATO_CONTEXTO' END AS TIPO_CONSUMO
FROM rankeado
WHERE RN <= QT_PARES_EXATOS
""")

# 3. Universo Residual
# Transações não consumidas por par exato que avançam para a reconciliação de bordas temporais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_contexto_residual AS
SELECT m.*
FROM vw_mov_marcado m
LEFT ANTI JOIN vw_ids_consumidos_exatos e ON m.NR_TRAN_INST_PCT = e.NR_TRAN_INST_PCT
""")

res_exatos = spark.sql("""
SELECT
    SUM(CASE WHEN TIPO_CONSUMO = 'EXATO_OFICIAL' THEN 1 ELSE 0 END) AS QT_OFICIAIS,
    SUM(CASE WHEN TIPO_CONSUMO = 'EXATO_CONTEXTO' THEN 1 ELSE 0 END) AS QT_CONTEXTO
FROM vw_ids_consumidos_exatos
""").first()

qt_pares_exatos_oficiais = int((res_exatos['QT_OFICIAIS'] or 0) / 2)
print(f'[V3_DASHBOARD] Pares Exatos em SQL: Oficiais={qt_pares_exatos_oficiais}, Linhas consumidas={res_exatos["QT_OFICIAIS"] or 0}')

### Reconciliação em SQL — Bordas Temporais (Matching nos Resíduos)


In [ ]:
%%spark

# UDF SQL para o matching determinístico de bordas temporais (±5 dias)
# Critérios de Otimização Contratuais:
# 1. Maximizar quantidade total de pares (cardinalidade)
# 2. Minimizar distância absoluta total em dias (proximidade)
# 3. Desempatar deterministicamente por IDs ascendentes
SCHEMA_PAR_BORDA = ArrayType(StructType([
    StructField('NR_TRAN_DENTRO', LongType(), False),
    StructField('NR_TRAN_FORA', LongType(), False),
    StructField('DT_TRAN_DENTRO', DateType(), False),
    StructField('DT_TRAN_FORA', DateType(), False),
    StructField('DIF_DIAS', IntegerType(), False),
]))

def parear_listas_residuais_sql_impl(lista_dentro, lista_fora):
    dentro = sorted(
        [(r['DT_TRAN'], int(r['NR_TRAN_INST_PCT'])) for r in (lista_dentro or [])],
        key=lambda item: (item[0], item[1])
    )
    fora = sorted(
        [(r['DT_TRAN'], int(r['NR_TRAN_INST_PCT'])) for r in (lista_fora or [])],
        key=lambda item: (item[0], item[1])
    )
    n = len(dentro)
    m = len(fora)
    vazio = (0, 0, ())
    dp = [[vazio for _ in range(m + 1)] for _ in range(n + 1)]

    def chave_solucao(solucao):
        quantidade, custo, pares = solucao
        assinatura_ids = tuple((par[0], par[1]) for par in pares)
        return (-quantidade, custo, assinatura_ids)

    for i in range(n - 1, -1, -1):
        for j in range(m - 1, -1, -1):
            candidatos = [dp[i + 1][j], dp[i][j + 1]]
            dt_dentro, id_dentro = dentro[i]
            dt_fora, id_fora = fora[j]
            dif_dias = abs((dt_dentro - dt_fora).days)
            if 1 <= dif_dias <= DIAS_CONTEXTO_RECONCILIACAO:
                quantidade, custo, pares = dp[i + 1][j + 1]
                par = (id_dentro, id_fora, dt_dentro, dt_fora, dif_dias)
                candidatos.append((quantidade + 1, custo + dif_dias, (par,) + pares))
            dp[i][j] = min(candidatos, key=chave_solucao)

    return list(dp[0][0][2])

spark.udf.register('parear_borda_udf', parear_listas_residuais_sql_impl, SCHEMA_PAR_BORDA)

# 1. Agrupamento das listas de dentro e fora da janela com valores e moedas iguais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pares_borda_calculados AS
WITH dentro_agg AS (
    SELECT
        CD_CLI,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        CD_NTZ_CTB_TRAN AS NTZ_DENTRO,
        COLLECT_LIST(NAMED_STRUCT('DT_TRAN', DT_TRAN, 'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT)) AS LISTA_DENTRO
    FROM vw_mov_contexto_residual
    WHERE IN_JANELA = 'S'
      AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN
),
fora_agg AS (
    SELECT
        CD_CLI,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END AS NTZ_DENTRO,
        COLLECT_LIST(NAMED_STRUCT('DT_TRAN', DT_TRAN, 'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT)) AS LISTA_FORA
    FROM vw_mov_contexto_residual
    WHERE IN_JANELA = 'N'
      AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END
),
pares_array AS (
    SELECT
        d.CD_CLI,
        d.VL_TRAN,
        d.CD_TIP_MOE_CRR,
        EXPLODE(parear_borda_udf(d.LISTA_DENTRO, f.LISTA_FORA)) AS PAR
    FROM dentro_agg d
    INNER JOIN fora_agg f
       ON d.CD_CLI = f.CD_CLI
      AND d.VL_TRAN = f.VL_TRAN
      AND d.CD_TIP_MOE_CRR = f.CD_TIP_MOE_CRR
      AND d.NTZ_DENTRO = f.NTZ_DENTRO
)
SELECT
    PAR.NR_TRAN_DENTRO AS NR_TRAN_DENTRO,
    PAR.NR_TRAN_FORA AS NR_TRAN_FORA,
    PAR.DT_TRAN_DENTRO AS DT_TRAN_DENTRO,
    PAR.DT_TRAN_FORA AS DT_TRAN_FORA,
    PAR.DIF_DIAS AS DIF_DIAS
FROM pares_array
""")

# 2. Identificação dos IDs Consumidos por Borda Temporal
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_borda AS
SELECT NR_TRAN_DENTRO AS NR_TRAN_INST_PCT, 'BORDA_OFICIAL' AS TIPO_CONSUMO FROM vw_pares_borda_calculados
UNION ALL
SELECT NR_TRAN_FORA AS NR_TRAN_INST_PCT, 'BORDA_CONTEXTO' AS TIPO_CONSUMO FROM vw_pares_borda_calculados
""")

qt_pares_borda = spark.sql('SELECT COUNT(1) AS QT FROM vw_pares_borda_calculados').first()['QT']
print(f'[V3_DASHBOARD] Pares de Borda em SQL: {qt_pares_borda}')

### Universo Efetivo e Verificação de Invariantes em SQL


In [ ]:
%%spark

# União de todos os IDs consumidos (Pares Exatos + Bordas)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_todos AS
SELECT * FROM vw_ids_consumidos_exatos
UNION ALL
SELECT * FROM vw_ids_consumidos_borda
""")

# IDs oficiais que devem ser expurgados da janela oficial
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_removidos_oficiais AS
SELECT * FROM vw_ids_consumidos_todos WHERE TIPO_CONSUMO IN ('EXATO_OFICIAL', 'BORDA_OFICIAL')
""")

# Verificação das Invariantes Contratuais de Reconciliação:
# 1. Nenhuma transação pode ser consumida mais de uma vez (TOTAL == DISTINTOS)
# 2. Total de remoções oficiais deve satisfazer a fórmula: 2 * PARES_EXATOS + PARES_BORDA
inv_rec = spark.sql("""
SELECT
    (SELECT COUNT(1) FROM vw_ids_consumidos_todos) AS TOTAL_CONSUMIDOS,
    (SELECT COUNT(DISTINCT NR_TRAN_INST_PCT) FROM vw_ids_consumidos_todos) AS DISTINTOS_CONSUMIDOS,
    (SELECT COUNT(1) FROM vw_ids_removidos_oficiais) AS TOTAL_REMOVIDOS_OFICIAIS,
    (SELECT COUNT(DISTINCT NR_TRAN_INST_PCT) FROM vw_ids_removidos_oficiais) AS DISTINTOS_REMOVIDOS_OFICIAIS
""").first()

if inv_rec['TOTAL_CONSUMIDOS'] != inv_rec['DISTINTOS_CONSUMIDOS']:
    raise RuntimeError('Invariante violada em SQL: transação consumida em mais de um par.')
if inv_rec['TOTAL_REMOVIDOS_OFICIAIS'] != inv_rec['DISTINTOS_REMOVIDOS_OFICIAIS']:
    raise RuntimeError('Invariante violada em SQL: COUNT(removidos) != COUNT(DISTINCT removidos).')

qt_transacoes_oficiais_removidas = inv_rec['TOTAL_REMOVIDOS_OFICIAIS']
esperado_removidas = 2 * qt_pares_exatos_oficiais + qt_pares_borda
if qt_transacoes_oficiais_removidas != esperado_removidas:
    raise RuntimeError(f'Invariante violada: remoções oficiais={qt_transacoes_oficiais_removidas}; esperado={esperado_removidas}.')

# Construção do Universo Efetivo (vw_mov_efetivo):
# Remove as transações anuladas via LEFT ANTI JOIN
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_efetivo AS
SELECT
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN
FROM vw_mov_raw m
LEFT ANTI JOIN vw_ids_removidos_oficiais r ON m.NR_TRAN_INST_PCT = r.NR_TRAN_INST_PCT
""")

qt_efetivo = spark.sql('SELECT COUNT(1) AS QT FROM vw_mov_efetivo').first()['QT']
esperado_efetivo = qt_raw - esperado_removidas
if qt_efetivo != esperado_efetivo:
    raise RuntimeError(f'Invariante violada em SQL: QT_EFETIVO={qt_efetivo}; esperado={esperado_efetivo}.')

print(f'[V3_DASHBOARD] Universo Efetivo SQL (vw_mov_efetivo): {qt_efetivo} transações.')

### Classificação com CATEGORIAS, Universo de Moeda e Agro em SQL


In [ ]:
%%spark

# 1. Classificação das Transações Efetivas com o Mapa CATEGORIAS
# LEFT JOIN pela chave composta (CD_CTGR_TRAN_OGNL == CD_CATEGORIA AND CD_NTZ_CTB_TRAN == TIPO)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_classificado AS
SELECT
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    c.CD_GRUPO,
    c.TX_GRUPO,
    COALESCE(c.TX_CATEGORIA, 'Sem Categoria') AS TX_CATEGORIA,
    c.CD_IR,
    c.TX_IR,
    COALESCE(c.CD_CLASS_RADAR, 0) AS CD_CLASS_RADAR,
    COALESCE(c.TX_CLASS_RADAR, 'Outras Entradas') AS TX_CLASS_RADAR,
    COALESCE(c.IN_AGRO, 'N') AS IN_AGRO,
    COALESCE(c.IN_PARTICIPA_CALCULO, 'N') AS IN_PARTICIPA_CALCULO,
    COALESCE(c.IN_PARTICIPA_ORCAMENTO, 'N') AS IN_PARTICIPA_ORCAMENTO
FROM vw_mov_efetivo m
LEFT JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
""")

# 2. Universo em Moeda Corrente Nacional (BRL)
# Filtra apenas transações em BRL para alimentação das métricas financeiras e orçamentárias
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_brl AS
SELECT * FROM vw_mov_classificado WHERE CD_TIP_MOE_CRR = 'BRL'
""")

# 3. Derivação das Flags de Moeda e Agro
# - FL_SOMENTE_BRL: 'S' se todas as transações efetivas forem BRL, 'N' caso haja moedas estrangeiras
# - FL_TEM_MOV_AGRO: 'S' se houver pelo menos uma transação com indicador agropecuário
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_flags_moeda_agro AS
SELECT
    CASE 
        WHEN (SELECT COUNT(1) FROM vw_mov_classificado) = 0 THEN NULL
        WHEN (SELECT COUNT(DISTINCT CD_TIP_MOE_CRR) FROM vw_mov_classificado) = 1 
         AND (SELECT MAX(CD_TIP_MOE_CRR) FROM vw_mov_classificado) = 'BRL' THEN 'S'
        ELSE 'N'
    END AS FL_SOMENTE_BRL,
    CASE 
        WHEN (SELECT COUNT(1) FROM vw_mov_brl) = 0 THEN NULL
        WHEN (SELECT COUNT(1) FROM vw_mov_brl WHERE IN_AGRO = 'S') > 0 THEN 'S'
        ELSE 'N'
    END AS FL_TEM_MOV_AGRO
""")

res_flags = spark.sql('SELECT * FROM vw_flags_moeda_agro').first()
print(f'[V3_DASHBOARD] Flags SQL: FL_SOMENTE_BRL={res_flags["FL_SOMENTE_BRL"]}, FL_TEM_MOV_AGRO={res_flags["FL_TEM_MOV_AGRO"]}')

### Agregações Temáticas e Totais Orçamentários em SQL


In [ ]:
%%spark

# Agregações Financeiras em SQL:
# - Entradas Temáticas (0 a 4): Renda (1), Estorno (2), Resgate (3), Outras (0), Crédito (4) com IN_PARTICIPA_CALCULO = 'S'
# - Saídas Temáticas (5 a 9): Indeterminado (5), Essenciais (6), Não Essenciais (7), Futuro (8), Obrigações (9) com IN_PARTICIPA_CALCULO = 'S'
# - Totais Orçamentários: VL_ENT_TOTAL e VL_SAI_TOTAL considerando apenas transações com IN_PARTICIPA_ORCAMENTO = 'S'
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_agregacoes_financeiras AS
SELECT
    -- Quantidades
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    -- Entradas Temáticas (Classes 0 a 4 com IN_PARTICIPA_CALCULO = 'S')
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 1 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 2 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 3 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 0 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 4 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    -- Saídas Temáticas (Classes 5 a 9 com IN_PARTICIPA_CALCULO = 'S')
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 5 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 6 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 7 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 8 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 9 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    -- Totais Orçamentários (IN_PARTICIPA_ORCAMENTO = 'S')
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_TOTAL,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_TOTAL
FROM vw_mov_brl
""")

# Validação das invariantes de agregações em SQL
inv_agg = spark.sql("""
SELECT
    QT_TRANS_TOTAL,
    QT_TRANS_ENT,
    QT_TRANS_SAI,
    VL_ENT_TOTAL,
    VL_SAI_TOTAL
FROM vw_agregacoes_financeiras
""").first()

# NULL nas quantidades parciais representa universo vazio; normalize apenas para a checagem.
qt_ent_invariante_v5 = 0 if inv_agg['QT_TRANS_ENT'] is None else inv_agg['QT_TRANS_ENT']
qt_sai_invariante_v5 = 0 if inv_agg['QT_TRANS_SAI'] is None else inv_agg['QT_TRANS_SAI']
if inv_agg['QT_TRANS_TOTAL'] != (qt_ent_invariante_v5 + qt_sai_invariante_v5):
    raise RuntimeError('Invariante violada em SQL: QT_TRANS_TOTAL != QT_TRANS_ENT + QT_TRANS_SAI.')

print(f'[V3_DASHBOARD] Agregações SQL: Entradas={inv_agg["VL_ENT_TOTAL"]}, Saídas={inv_agg["VL_SAI_TOTAL"]}')

### Análise Orçamentária e Faixas em SQL


In [ ]:
%%spark

# 1. Cálculo do Saldo Orçamentário e Razão Saídas / Entradas
# - VL_RES_ORC = VL_ENT_TOTAL - VL_SAI_TOTAL
# - PC_SAI_ENT = VL_SAI_TOTAL / VL_ENT_TOTAL (arredondado em 6 casas decimais)
# 2. Enquadramento nas 5 Faixas Orçamentárias Contratuais:
# - Faixa 0: Neutro (0.95 <= PC <= 1.05)
# - Faixa 1: Deficitário Moderado (1.05 < PC <= 1.25)
# - Faixa 2: Deficitário Acentuado (PC > 1.25)
# - Faixa 3: Superavitário Moderado (0.75 <= PC < 0.95)
# - Faixa 4: Superavitário Acentuado (PC < 0.75)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_orcamento_derivado AS
WITH base AS (
    SELECT
        VL_ENT_TOTAL,
        VL_SAI_TOTAL,
        QT_TRANS_TOTAL,
        CAST(VL_ENT_TOTAL - VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_RES_ORC,
        CASE 
            WHEN QT_TRANS_TOTAL = 0 OR VL_ENT_TOTAL = 0 THEN NULL
            ELSE CAST(ROUND(VL_SAI_TOTAL / VL_ENT_TOTAL, 6) AS DECIMAL(9,6))
        END AS PC_SAI_ENT
    FROM vw_agregacoes_financeiras
),
faixas AS (
    SELECT
        *,
        CASE 
            WHEN PC_SAI_ENT IS NULL THEN NULL
            WHEN PC_SAI_ENT >= 0.950000 AND PC_SAI_ENT <= 1.050000 THEN 0
            WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
            WHEN PC_SAI_ENT > 1.250000 THEN 2
            WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
            ELSE 4
        END AS CD_FAIXA_ORC
    FROM base
)
SELECT
    VL_RES_ORC,
    PC_SAI_ENT,
    CD_FAIXA_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 0
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 2
        ELSE 1
    END AS CD_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 'Deficitário'
        ELSE 'Superavitário'
    END AS TX_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL
        WHEN CD_FAIXA_ORC IN (1, 3) THEN 'Moderado'
        ELSE 'Acentuado'
    END AS TX_STS_RES,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado'
        WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado'
        WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado'
        WHEN CD_FAIXA_ORC = 4 THEN 'Superavitário Acentuado'
    END AS TX_STS_FINAL
FROM faixas
""")

res_orc = spark.sql('SELECT * FROM vw_orcamento_derivado').first()
print(f'[V3_DASHBOARD] Orçamento SQL: VL_RES_ORC={res_orc["VL_RES_ORC"]}, FAIXA={res_orc["CD_FAIXA_ORC"]}, STATUS={res_orc["TX_STS_FINAL"]}')

### Percentuais sobre Renda Presumida e Referências em SQL


In [ ]:
%%spark

# 1. Percentuais de Referência Contratuais (Constantes fixas):
# - IND: 75% | ESS: 50% | NAO_ESS: 30% | FUT: 20% | OBR: 30%
# 2. Percentuais Observados sobre a Renda Presumida:
# - PC_SAI_* = VL_SAI_* / VL_REN_PRES (arredondado em 6 casas decimais)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_percentuais_renda AS
WITH renda_base AS (
    SELECT VL_REN_PRES FROM vw_renda_derivada
),
agregados AS (
    SELECT VL_SAI_IND, VL_SAI_ESS, VL_SAI_NAO_ESS, VL_SAI_FUT, VL_SAI_OBR FROM vw_agregacoes_financeiras
)
SELECT
    -- Constantes de Referência Contratuais
    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,
    -- Percentuais Observados sobre Renda Presumida
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_IND / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_NAO_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_FUT / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_OBR / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
FROM agregados a
CROSS JOIN renda_base r
""")

res_pc = spark.sql('SELECT * FROM vw_percentuais_renda').first()
print(f'[V3_DASHBOARD] % Renda SQL: ESS={res_pc["PC_SAI_ESS"]}, NAO_ESS={res_pc["PC_SAI_NAO_ESS"]}, FUT={res_pc["PC_SAI_FUT"]}, OBR={res_pc["PC_SAI_OBR"]}')

### Matrizes de Pontuação em SQL (Concentração, Orçamento, Perfil e Finais)


In [ ]:
%%spark

# 1. Dimensões de Pontuação:
# - Concentração: pontua o desvio de cada gasto temático em relação aos percentuais de referência
# - Orçamentária: pontua a pressão sobre o orçamento global de acordo com a faixa orçamentária
# - Perfil: pontua a aderência do comportamento financeiro ao macroperfil do cliente
# 2. Regra Especial de IND:
# - O tema 1 (Categorização dos Gastos / IND) recebe estritamente a pontuação de concentração (IND_FIM = CONC_IND)
# - Não soma orçamento nem perfil, mantendo o foco analítico exclusivo no volume de despesas sem categorização
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pontuacoes AS
WITH ctx AS (
    SELECT 
        (SELECT QT_TRANS_TOTAL FROM vw_agregacoes_financeiras) AS QT_TRANS_TOTAL,
        (SELECT VL_REN_PRES FROM vw_renda_derivada) AS VL_REN_PRES,
        (SELECT CD_FAIXA_ORC FROM vw_orcamento_derivado) AS CD_FAIXA_ORC,
        (SELECT CD_MAC_PRFL_CLI FROM vw_perfil_derivado) AS CD_MAC_PRFL_CLI,
        p.*
    FROM vw_percentuais_renda p
),
conc AS (
    SELECT
        ctx.*,
        -- Pontuação de Concentração por Tema
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_IND > 0.750000 THEN 99
            ELSE 0
        END AS NR_PONT_CONC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_ESS < 0.500000 THEN 0
            WHEN PC_SAI_ESS < 0.750000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.300000 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_FUT >= 0.300000 THEN 0
            WHEN PC_SAI_FUT >= 0.200000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_OBR < 0.300000 THEN 0
            WHEN PC_SAI_OBR < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_OBR
    FROM ctx
),
orc AS (
    SELECT
        conc.*,
        -- Pontuação Orçamentária por Tema (Matriz de Faixas)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 4 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_OBR
    FROM conc
),
prfl AS (
    SELECT
        orc.*,
        -- Pontuação de Perfil por Tema (Matriz de Macroperfis 1-Endividado, 2-Equilibrista, 3-Investidor)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 0
            ELSE 1
        END AS NR_PONT_PRFL_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 3 THEN 2
            WHEN CD_MAC_PRFL_CLI = 2 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 2
            ELSE 0
        END AS NR_PONT_PRFL_OBR
    FROM orc
)
SELECT
    *,
    -- REGRA ESPECIAL IND: IND_FIM = CONC_IND (ignora ORC e PRFL)
    NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
    -- Finais para os demais temas (soma CONC + ORC + PRFL, NULL se qualquer parcela for NULL)
    CASE WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS END AS NR_PONT_ESS_FIM,
    CASE WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS END AS NR_PONT_NAO_ESS_FIM,
    CASE WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT END AS NR_PONT_FUT_FIM,
    CASE WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR END AS NR_PONT_OBR_FIM,
    -- Flag de Pontuação Completa: 'S' se todos os 5 temas finais foram pontuados
    CASE 
        WHEN NR_PONT_CONC_IND IS NOT NULL 
         AND (NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT) IS NOT NULL 
         AND (NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR) IS NOT NULL 
        THEN 'S' 
        ELSE 'N' 
    END AS FL_PONTUACAO_COMPLETA
FROM prfl
""")

res_pont = spark.sql('SELECT NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM, FL_PONTUACAO_COMPLETA FROM vw_pontuacoes').first()
print(f'[V3_DASHBOARD] Pontuações Finais SQL: IND={res_pont["NR_PONT_IND_FIM"]}, ESS={res_pont["NR_PONT_ESS_FIM"]}, NAO_ESS={res_pont["NR_PONT_NAO_ESS_FIM"]}, FUT={res_pont["NR_PONT_FUT_FIM"]}, OBR={res_pont["NR_PONT_OBR_FIM"]} (COMPLETA={res_pont["FL_PONTUACAO_COMPLETA"]})')

### Tema Vencedor e Tratamento de Empate em SQL


In [ ]:
%%spark

# 1. Identificação da Maior Pontuação (GREATEST dos 5 temas finais)
# 2. Contagem de Vencedores (empates):
# - Se houver mais de um tema com a pontuação máxima: CD_TEMA_VENCEDOR = 9 ('Empate')
# - Se houver vencedor único: atribui o código do tema com maior pontuação (1 a 5)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_tema_vencedor AS
WITH base AS (
    SELECT * FROM vw_pontuacoes
),
max_calculado AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE GREATEST(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM)
        END AS NR_PONT_MAX
    FROM base
),
contagem_vencedores AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE (
                CASE WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 1 ELSE 0 END
            )
        END AS QT_TEMAS_PONT_MAX
    FROM max_calculado
)
SELECT
    *,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 9
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 3
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 5
    END AS CD_TEMA_VENCEDOR,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 'Empate'
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 'Categorização dos Gastos'
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 'Gestão de Orçamento'
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 'Consumo Planejado'
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 'Formação de Reserva'
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 'Uso Consciente do Crédito'
    END AS TX_TEMA_VENCEDOR
FROM contagem_vencedores
""")

res_venc = spark.sql('SELECT NR_PONT_MAX, QT_TEMAS_PONT_MAX, CD_TEMA_VENCEDOR, TX_TEMA_VENCEDOR FROM vw_tema_vencedor').first()
print(f'[V3_DASHBOARD] Vencedor SQL: CD={res_venc["CD_TEMA_VENCEDOR"]} ({res_venc["TX_TEMA_VENCEDOR"]}), PONT={res_venc["NR_PONT_MAX"]} (EMPATES={res_venc["QT_TEMAS_PONT_MAX"]})')

---
# Bloco 4 — Consolidação das 80 Colunas e Validação Contratual em SQL
Junção das views parciais para montar o registro final de 80 atributos físicos e validação com `raise`.


In [ ]:
%%spark

# Consolidação dos 80 Atributos do Contrato Físico Final
# Junção analítica via CROSS JOIN das views parciais estruturadas
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_resultado_80_colunas AS
SELECT
    -- 1 a 7: Cliente e CPF
    c.CD_CLI AS CD_CLI,
    DATE('{DATA_EXECUCAO.isoformat()}') AS DT_EXEA,
    DATE('{DT_MES_EXEA.isoformat()}') AS DT_MES_EXEA,
    c.TS_INCL_TRAN_REF AS TS_INCL_TRAN_REF,
    c.FL_CPF_UNICO AS FL_CPF_UNICO,
    c.CD_CPF AS CD_CPF,
    cta.FL_CONTA_ELEGIVEL_UNICA AS FL_CONTA_ELEGIVEL_UNICA,
    -- 8 a 10: Ciclo
    j.TS_DD_INC_MM_CLC_BLC_REF AS TS_DD_INC_MM_CLC_BLC_REF,
    j.DD_INC_MM_CLC_BLC AS DD_INC_MM_CLC_BLC,
    j.DD_INC_MM_CLC_BLC_FALLBACK AS DD_INC_MM_CLC_BLC_FALLBACK,
    -- 11 a 12: Renda
    r.DT_REN_PRES_REF AS DT_REN_PRES_REF,
    r.VL_REN_PRES AS VL_REN_PRES,
    -- 13 a 17: Perfil
    p.DT_REF_PRFL AS DT_REF_PRFL,
    p.CD_MAC_PRFL_CLI AS CD_MAC_PRFL_CLI,
    p.NM_MAC_PRFL_CLI AS NM_MAC_PRFL_CLI,
    p.CD_MIC_PRFL_CLI AS CD_MIC_PRFL_CLI,
    p.NM_MIC_PRFL_CLI AS NM_MIC_PRFL_CLI,
    -- 18 a 21: Janela, Moeda e Agro
    j.DT_REF_INI AS DT_REF_INI,
    j.DT_REF_FIM AS DT_REF_FIM,
    f.FL_SOMENTE_BRL AS FL_SOMENTE_BRL,
    f.FL_TEM_MOV_AGRO AS FL_TEM_MOV_AGRO,
    -- 22 a 26: Quantidades e Totais
    a.QT_TRANS_TOTAL AS QT_TRANS_TOTAL,
    a.QT_TRANS_ENT AS QT_TRANS_ENT,
    a.QT_TRANS_SAI AS QT_TRANS_SAI,
    a.VL_ENT_TOTAL AS VL_TRANS_ENT,
    a.VL_SAI_TOTAL AS VL_TRANS_SAI,
    -- 27 a 32: Entradas Temáticas e Total
    a.VL_ENT_REN AS VL_ENT_REN,
    a.VL_ENT_EST AS VL_ENT_EST,
    a.VL_ENT_RESG AS VL_ENT_RESG,
    a.VL_ENT_OUT AS VL_ENT_OUT,
    a.VL_ENT_CRED AS VL_ENT_CRED,
    a.VL_ENT_TOTAL AS VL_ENT_TOTAL,
    -- 33 a 38: Saídas Temáticas e Total
    a.VL_SAI_IND AS VL_SAI_IND,
    a.VL_SAI_ESS AS VL_SAI_ESS,
    a.VL_SAI_NAO_ESS AS VL_SAI_NAO_ESS,
    a.VL_SAI_FUT AS VL_SAI_FUT,
    a.VL_SAI_OBR AS VL_SAI_OBR,
    a.VL_SAI_TOTAL AS VL_SAI_TOTAL,
    -- 39 a 45: Orçamento
    o.VL_RES_ORC AS VL_RES_ORC,
    o.PC_SAI_ENT AS PC_SAI_ENT,
    o.CD_RES_ORC AS CD_RES_ORC,
    o.TX_RES_ORC AS TX_RES_ORC,
    o.CD_FAIXA_ORC AS CD_FAIXA_ORC,
    o.TX_STS_RES AS TX_STS_RES,
    o.TX_STS_FINAL AS TX_STS_FINAL,
    -- 46 a 50: Percentuais Saídas sobre Renda
    v.PC_SAI_IND AS PC_SAI_IND,
    v.PC_SAI_ESS AS PC_SAI_ESS,
    v.PC_SAI_NAO_ESS AS PC_SAI_NAO_ESS,
    v.PC_SAI_FUT AS PC_SAI_FUT,
    v.PC_SAI_OBR AS PC_SAI_OBR,
    -- 51 a 55: Percentuais de Referência
    v.PC_REF_IND AS PC_REF_IND,
    v.PC_REF_ESS AS PC_REF_ESS,
    v.PC_REF_NAO_ESS AS PC_REF_NAO_ESS,
    v.PC_REF_FUT AS PC_REF_FUT,
    v.PC_REF_OBR AS PC_REF_OBR,
    -- 56 a 60: Pontuação de Concentração
    v.NR_PONT_CONC_IND AS NR_PONT_CONC_IND,
    v.NR_PONT_CONC_ESS AS NR_PONT_CONC_ESS,
    v.NR_PONT_CONC_NAO_ESS AS NR_PONT_CONC_NAO_ESS,
    v.NR_PONT_CONC_FUT AS NR_PONT_CONC_FUT,
    v.NR_PONT_CONC_OBR AS NR_PONT_CONC_OBR,
    -- 61 a 65: Pontuação Orçamentária
    v.NR_PONT_ORC_IND AS NR_PONT_ORC_IND,
    v.NR_PONT_ORC_ESS AS NR_PONT_ORC_ESS,
    v.NR_PONT_ORC_NAO_ESS AS NR_PONT_ORC_NAO_ESS,
    v.NR_PONT_ORC_FUT AS NR_PONT_ORC_FUT,
    v.NR_PONT_ORC_OBR AS NR_PONT_ORC_OBR,
    -- 66 a 70: Pontuação de Perfil
    v.NR_PONT_PRFL_IND AS NR_PONT_PRFL_IND,
    v.NR_PONT_PRFL_ESS AS NR_PONT_PRFL_ESS,
    v.NR_PONT_PRFL_NAO_ESS AS NR_PONT_PRFL_NAO_ESS,
    v.NR_PONT_PRFL_FUT AS NR_PONT_PRFL_FUT,
    v.NR_PONT_PRFL_OBR AS NR_PONT_PRFL_OBR,
    -- 71 a 75: Pontuações Finais
    v.NR_PONT_IND_FIM AS NR_PONT_IND_FIM,
    v.NR_PONT_ESS_FIM AS NR_PONT_ESS_FIM,
    v.NR_PONT_NAO_ESS_FIM AS NR_PONT_NAO_ESS_FIM,
    v.NR_PONT_FUT_FIM AS NR_PONT_FUT_FIM,
    v.NR_PONT_OBR_FIM AS NR_PONT_OBR_FIM,
    -- 76 a 80: Status e Tema Vencedor
    v.FL_PONTUACAO_COMPLETA AS FL_PONTUACAO_COMPLETA,
    v.NR_PONT_MAX AS NR_PONT_MAX,
    v.QT_TEMAS_PONT_MAX AS QT_TEMAS_PONT_MAX,
    v.CD_TEMA_VENCEDOR AS CD_TEMA_VENCEDOR,
    v.TX_TEMA_VENCEDOR AS TX_TEMA_VENCEDOR
FROM vw_cliente_derivado c
CROSS JOIN vw_conta_elegivel_resumo cta
CROSS JOIN vw_ciclo_janela j
CROSS JOIN vw_renda_derivada r
CROSS JOIN vw_perfil_derivado p
CROSS JOIN vw_flags_moeda_agro f
CROSS JOIN vw_agregacoes_financeiras a
CROSS JOIN vw_orcamento_derivado o
CROSS JOIN vw_tema_vencedor v
""")

print('[V3_DASHBOARD] View SQL vw_resultado_80_colunas montada com sucesso.')

### Validação Contratual Estrita via SQL


In [ ]:
%%spark

df_res_80 = spark.sql('SELECT * FROM vw_resultado_80_colunas')
qt_final_sql = df_res_80.count()

# 1. Validação de grão (exatamente 1 linha para o CD_CLI solicitado)
if qt_final_sql != 1:
    raise RuntimeError(f'Contrato violado: resultado deve possuir exatamente 1 linha; encontrado {qt_final_sql}.')

linha_final_sql = df_res_80.first()
if linha_final_sql['CD_CLI'] != CD_CLI:
    raise RuntimeError(f'Contrato violado: CD_CLI divergente ({linha_final_sql["CD_CLI"]} != {CD_CLI}).')

# 2. Validação de largura física (exatamente 80 colunas)
if len(df_res_80.columns) != 80:
    raise RuntimeError(f'Contrato violado: esperado 80 colunas; encontrado {len(df_res_80.columns)}.')

# 3. Validação de Nulabilidade das Colunas NOT NULL
campos_not_null = [
    'CD_CLI', 'DT_EXEA', 'DT_MES_EXEA', 'TS_INCL_TRAN_REF', 'FL_CPF_UNICO',
    'FL_CONTA_ELEGIVEL_UNICA', 'PC_REF_IND', 'PC_REF_ESS', 'PC_REF_NAO_ESS',
    'PC_REF_FUT', 'PC_REF_OBR', 'FL_PONTUACAO_COMPLETA'
]
for cnn in campos_not_null:
    if linha_final_sql[cnn] is None:
        raise RuntimeError(f'Coluna contratual NOT NULL com valor nulo: {cnn}.')

print('[V3_DASHBOARD] Validação contratual em SQL aprovada: 1 linha, 80 atributos íntegros.')

### Publicação da View Final no Catálogo


In [ ]:
%%spark

# Publica a view oficial vw_radar_financeiro_cliente_mvp no catálogo da sessão Spark
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW {VIEW_RESULTADO} AS
SELECT * FROM vw_resultado_80_colunas
""")

# Confirma que a view foi devidamente registrada e está disponível como tabela temporária
views_catalogo = [obj for obj in spark.catalog.listTables() if obj.name == VIEW_RESULTADO]
if len(views_catalogo) != 1 or not views_catalogo[0].isTemporary:
    raise RuntimeError(f'A temporary view {VIEW_RESULTADO} não foi registrada corretamente no catálogo.')

print(f'[V3_DASHBOARD] View final publicada no catálogo: {VIEW_RESULTADO}')

### V5 — Cenários pré-calculados com base financeira alternável


In [ ]:
%%spark

from pyspark.sql import functions as F


# V5 deriva Entradas Realizadas somente do universo já efetivo/reconciliado.
# O INNER JOIN exige classificação válida; BRL e natureza C são explícitos.
SQL_ENTRADAS_REALIZADAS_V5 = """
CREATE OR REPLACE TEMPORARY VIEW vw_v5_entradas_realizadas_detalhe AS
SELECT
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    c.CD_GRUPO,
    c.TX_GRUPO,
    c.TX_CATEGORIA,
    c.CD_CLASS_RADAR,
    c.TX_CLASS_RADAR
FROM vw_mov_efetivo m
INNER JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
WHERE m.CD_NTZ_CTB_TRAN = 'C'
  AND m.CD_TIP_MOE_CRR = 'BRL'
"""
spark.sql(SQL_ENTRADAS_REALIZADAS_V5)

# Janela indisponível é NULL; zero só representa janela existente sem crédito válido.
SQL_TOTAL_ENTRADAS_REALIZADAS_V5 = """
CREATE OR REPLACE TEMPORARY VIEW vw_v5_entradas_realizadas AS
WITH total AS (
    SELECT CAST(SUM(VL_TRAN) AS DECIMAL(25,2)) AS SOMA
    FROM vw_v5_entradas_realizadas_detalhe
)
SELECT
    CASE
        WHEN j.DT_REF_INI IS NULL OR j.DT_REF_FIM IS NULL
            THEN CAST(NULL AS DECIMAL(25,2))
        ELSE CAST(
            COALESCE(t.SOMA, CAST(0.00 AS DECIMAL(25,2)))
            AS DECIMAL(25,2)
        )
    END AS ENTRADAS_REALIZADAS
FROM vw_ciclo_janela j
CROSS JOIN total t
"""
spark.sql(SQL_TOTAL_ENTRADAS_REALIZADAS_V5)

SQL_BASES_FINANCEIRAS_V5 = """
CREATE OR REPLACE TEMPORARY VIEW vw_v5_bases_financeiras AS
SELECT
    'RENDA_PRESUMIDA' AS CD_CENARIO,
    CASE
        WHEN j.DT_REF_INI IS NULL OR j.DT_REF_FIM IS NULL
            THEN CAST(NULL AS DECIMAL(25,2))
        ELSE CAST(r.VL_REN_PRES AS DECIMAL(25,2))
    END AS BASE_FINANCEIRA
FROM vw_ciclo_janela j
CROSS JOIN vw_renda_derivada r
UNION ALL
SELECT
    'ENTRADAS_REALIZADAS' AS CD_CENARIO,
    CAST(e.ENTRADAS_REALIZADAS AS DECIMAL(25,2)) AS BASE_FINANCEIRA
FROM vw_v5_entradas_realizadas e
"""
spark.sql(SQL_BASES_FINANCEIRAS_V5)

# Fonte única das regras: SQLs extraídos mecanicamente das células V4 congeladas.
SQL_CADEIA_V4_V5 = {
    'orcamento': r"""CREATE OR REPLACE TEMPORARY VIEW vw_orcamento_derivado AS
WITH base AS (
    SELECT
        VL_ENT_TOTAL,
        VL_SAI_TOTAL,
        QT_TRANS_TOTAL,
        CAST(VL_ENT_TOTAL - VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_RES_ORC,
        CASE 
            WHEN QT_TRANS_TOTAL = 0 OR VL_ENT_TOTAL = 0 THEN NULL
            ELSE CAST(ROUND(VL_SAI_TOTAL / VL_ENT_TOTAL, 6) AS DECIMAL(9,6))
        END AS PC_SAI_ENT
    FROM vw_agregacoes_financeiras
),
faixas AS (
    SELECT
        *,
        CASE 
            WHEN PC_SAI_ENT IS NULL THEN NULL
            WHEN PC_SAI_ENT >= 0.950000 AND PC_SAI_ENT <= 1.050000 THEN 0
            WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
            WHEN PC_SAI_ENT > 1.250000 THEN 2
            WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
            ELSE 4
        END AS CD_FAIXA_ORC
    FROM base
)
SELECT
    VL_RES_ORC,
    PC_SAI_ENT,
    CD_FAIXA_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 0
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 2
        ELSE 1
    END AS CD_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 'Deficitário'
        ELSE 'Superavitário'
    END AS TX_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL
        WHEN CD_FAIXA_ORC IN (1, 3) THEN 'Moderado'
        ELSE 'Acentuado'
    END AS TX_STS_RES,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado'
        WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado'
        WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado'
        WHEN CD_FAIXA_ORC = 4 THEN 'Superavitário Acentuado'
    END AS TX_STS_FINAL
FROM faixas""",
    'percentuais': r"""CREATE OR REPLACE TEMPORARY VIEW vw_percentuais_renda AS
WITH renda_base AS (
    SELECT VL_REN_PRES FROM vw_renda_derivada
),
agregados AS (
    SELECT VL_SAI_IND, VL_SAI_ESS, VL_SAI_NAO_ESS, VL_SAI_FUT, VL_SAI_OBR FROM vw_agregacoes_financeiras
)
SELECT
    -- Constantes de Referência Contratuais
    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,
    -- Percentuais Observados sobre Renda Presumida
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_IND / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_NAO_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_FUT / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_OBR / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
FROM agregados a
CROSS JOIN renda_base r""",
    'pontuacoes': r"""CREATE OR REPLACE TEMPORARY VIEW vw_pontuacoes AS
WITH ctx AS (
    SELECT 
        (SELECT QT_TRANS_TOTAL FROM vw_agregacoes_financeiras) AS QT_TRANS_TOTAL,
        (SELECT VL_REN_PRES FROM vw_renda_derivada) AS VL_REN_PRES,
        (SELECT CD_FAIXA_ORC FROM vw_orcamento_derivado) AS CD_FAIXA_ORC,
        (SELECT CD_MAC_PRFL_CLI FROM vw_perfil_derivado) AS CD_MAC_PRFL_CLI,
        p.*
    FROM vw_percentuais_renda p
),
conc AS (
    SELECT
        ctx.*,
        -- Pontuação de Concentração por Tema
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_IND > 0.750000 THEN 99
            ELSE 0
        END AS NR_PONT_CONC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_ESS < 0.500000 THEN 0
            WHEN PC_SAI_ESS < 0.750000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.300000 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_FUT >= 0.300000 THEN 0
            WHEN PC_SAI_FUT >= 0.200000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_OBR < 0.300000 THEN 0
            WHEN PC_SAI_OBR < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_OBR
    FROM ctx
),
orc AS (
    SELECT
        conc.*,
        -- Pontuação Orçamentária por Tema (Matriz de Faixas)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 4 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_OBR
    FROM conc
),
prfl AS (
    SELECT
        orc.*,
        -- Pontuação de Perfil por Tema (Matriz de Macroperfis 1-Endividado, 2-Equilibrista, 3-Investidor)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 0
            ELSE 1
        END AS NR_PONT_PRFL_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 3 THEN 2
            WHEN CD_MAC_PRFL_CLI = 2 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 2
            ELSE 0
        END AS NR_PONT_PRFL_OBR
    FROM orc
)
SELECT
    *,
    -- REGRA ESPECIAL IND: IND_FIM = CONC_IND (ignora ORC e PRFL)
    NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
    -- Finais para os demais temas (soma CONC + ORC + PRFL, NULL se qualquer parcela for NULL)
    CASE WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS END AS NR_PONT_ESS_FIM,
    CASE WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS END AS NR_PONT_NAO_ESS_FIM,
    CASE WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT END AS NR_PONT_FUT_FIM,
    CASE WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR END AS NR_PONT_OBR_FIM,
    -- Flag de Pontuação Completa: 'S' se todos os 5 temas finais foram pontuados
    CASE 
        WHEN NR_PONT_CONC_IND IS NOT NULL 
         AND (NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT) IS NOT NULL 
         AND (NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR) IS NOT NULL 
        THEN 'S' 
        ELSE 'N' 
    END AS FL_PONTUACAO_COMPLETA
FROM prfl""",
    'vencedor': r"""CREATE OR REPLACE TEMPORARY VIEW vw_tema_vencedor AS
WITH base AS (
    SELECT * FROM vw_pontuacoes
),
max_calculado AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE GREATEST(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM)
        END AS NR_PONT_MAX
    FROM base
),
contagem_vencedores AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE (
                CASE WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 1 ELSE 0 END
            )
        END AS QT_TEMAS_PONT_MAX
    FROM max_calculado
)
SELECT
    *,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 9
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 3
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 5
    END AS CD_TEMA_VENCEDOR,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 'Empate'
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 'Categorização dos Gastos'
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 'Gestão de Orçamento'
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 'Consumo Planejado'
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 'Formação de Reserva'
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 'Uso Consciente do Crédito'
    END AS TX_TEMA_VENCEDOR
FROM contagem_vencedores""",
}
SHA256_SQL_CADEIA_V4_V5 = {'orcamento': '2fb6a2214ef4806c634914b6a40ae55e8bb4b23a651ca55c24facc7ea304cf22', 'percentuais': '70027ba9289a66195e4ef3aa06ba698dcf6db47d225db6e61af9e00a1ebd8011', 'pontuacoes': '263b378a805114c9e7e3023ce38fda50cce7ed687e20267fea90610ae8a581fc', 'vencedor': 'f29811785f1ece181df4d3f8b1bcaa99939a09356e5b53895a3f32d95bf109ae'}


def _nome_view_cenario_v5(prefixo, nome_original):
    return f'vw_v5_{prefixo}_{nome_original[3:]}'


def _remapear_sql_cenario_v5(sql_original, prefixo):
    nomes = [
        'vw_agregacoes_financeiras', 'vw_renda_derivada',
        'vw_orcamento_derivado', 'vw_percentuais_renda',
        'vw_pontuacoes', 'vw_tema_vencedor',
    ]
    remapeado = sql_original
    for nome in nomes:
        remapeado = re.sub(
            rf'\b{re.escape(nome)}\b',
            _nome_view_cenario_v5(prefixo, nome),
            remapeado,
        )
    return remapeado


PREFIXOS_CENARIOS_V5 = {
    'RENDA_PRESUMIDA': 'renda_presumida',
    'ENTRADAS_REALIZADAS': 'entradas_realizadas',
}
execucoes_cenarios_v5 = []


def calcular_cenario_v5(chave, base_financeira):
    if chave not in PREFIXOS_CENARIOS_V5:
        raise ValueError(f'Cenário V5 desconhecido: {chave}.')
    prefixo = PREFIXOS_CENARIOS_V5[chave]

    # Os dois adaptadores recebem a mesma e única BASE_FINANCEIRA.
    base_col = F.lit(base_financeira).cast(DecimalType(25, 2))
    nome_agregacoes = _nome_view_cenario_v5(prefixo, 'vw_agregacoes_financeiras')
    nome_renda = _nome_view_cenario_v5(prefixo, 'vw_renda_derivada')
    spark.table('vw_agregacoes_financeiras').withColumn(
        'VL_ENT_TOTAL', base_col
    ).createOrReplaceTempView(nome_agregacoes)
    spark.table('vw_renda_derivada').withColumn(
        'VL_REN_PRES', base_col
    ).createOrReplaceTempView(nome_renda)

    for etapa in ('orcamento', 'percentuais', 'pontuacoes', 'vencedor'):
        spark.sql(_remapear_sql_cenario_v5(SQL_CADEIA_V4_V5[etapa], prefixo))

    nome_orcamento = _nome_view_cenario_v5(prefixo, 'vw_orcamento_derivado')
    nome_vencedor = _nome_view_cenario_v5(prefixo, 'vw_tema_vencedor')
    linhas_orcamento = spark.table(nome_orcamento).collect()
    linhas_vencedor = spark.table(nome_vencedor).collect()
    if len(linhas_orcamento) != 1 or len(linhas_vencedor) != 1:
        raise RuntimeError(
            f'[V5] Cenário {chave} não produziu exatamente uma linha em cada etapa.'
        )
    execucoes_cenarios_v5.append(chave)
    return {
        'base_financeira': base_financeira,
        'orcamento': linhas_orcamento[0].asDict(recursive=True),
        'resultado': linhas_vencedor[0].asDict(recursive=True),
    }


bases_cenarios_v5 = {
    row['CD_CENARIO']: row['BASE_FINANCEIRA']
    for row in spark.table('vw_v5_bases_financeiras').collect()
}
if set(bases_cenarios_v5) != {'RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS'}:
    raise RuntimeError(f'[V5] Bases financeiras inesperadas: {sorted(bases_cenarios_v5)}.')

calculos_cenarios_v5 = {
    chave: calcular_cenario_v5(chave, bases_cenarios_v5[chave])
    for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS')
}

# A view oficial V4 permanece publicada e imutável. Os snapshots laterais
# preservam seu schema de 80 colunas e substituem somente resultados da cadeia.
resultado_oficial_v4_v5 = linha_final_sql.asDict(recursive=True)
CAMPOS_ORCAMENTO_CENARIO_V5 = [
    'VL_RES_ORC', 'PC_SAI_ENT', 'CD_RES_ORC', 'TX_RES_ORC',
    'CD_FAIXA_ORC', 'TX_STS_RES', 'TX_STS_FINAL',
]
CAMPOS_MOTOR_CENARIO_V5 = [
    'PC_SAI_IND', 'PC_SAI_ESS', 'PC_SAI_NAO_ESS', 'PC_SAI_FUT', 'PC_SAI_OBR',
    'NR_PONT_CONC_IND', 'NR_PONT_CONC_ESS', 'NR_PONT_CONC_NAO_ESS',
    'NR_PONT_CONC_FUT', 'NR_PONT_CONC_OBR',
    'NR_PONT_ORC_ESS', 'NR_PONT_ORC_NAO_ESS',
    'NR_PONT_ORC_FUT', 'NR_PONT_ORC_OBR',
    'NR_PONT_IND_FIM', 'NR_PONT_ESS_FIM', 'NR_PONT_NAO_ESS_FIM',
    'NR_PONT_FUT_FIM', 'NR_PONT_OBR_FIM',
    'FL_PONTUACAO_COMPLETA', 'NR_PONT_MAX', 'QT_TEMAS_PONT_MAX',
    'CD_TEMA_VENCEDOR', 'TX_TEMA_VENCEDOR',
]
CAMPOS_VARIAVEIS_CENARIO_V5 = set(CAMPOS_ORCAMENTO_CENARIO_V5 + CAMPOS_MOTOR_CENARIO_V5)

resultados_cenarios_v5 = {}
dataframes_cenarios_v5 = {}
for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS'):
    snapshot = dict(resultado_oficial_v4_v5)
    calculo = calculos_cenarios_v5[chave]
    for campo in CAMPOS_ORCAMENTO_CENARIO_V5:
        snapshot[campo] = calculo['orcamento'][campo]
    for campo in CAMPOS_MOTOR_CENARIO_V5:
        snapshot[campo] = calculo['resultado'][campo]
    if list(snapshot) != list(resultado_oficial_v4_v5) or len(snapshot) != 80:
        raise RuntimeError(f'[V5] Snapshot {chave} alterou o contrato de 80 colunas.')
    dataframe = spark.createDataFrame([snapshot], schema=df_res_80.schema)
    if dataframe.schema != df_res_80.schema or dataframe.columns != df_res_80.columns:
        raise RuntimeError(f'[V5] Schema lateral divergente no cenário {chave}.')
    nome_snapshot = _nome_view_cenario_v5(
        PREFIXOS_CENARIOS_V5[chave], 'vw_resultado_80_colunas'
    )
    dataframe.createOrReplaceTempView(nome_snapshot)
    resultados_cenarios_v5[chave] = snapshot
    dataframes_cenarios_v5[chave] = dataframe

valor_entradas_realizadas_v5 = bases_cenarios_v5['ENTRADAS_REALIZADAS']
print(
    '[V5] Dois cenários pré-calculados pela mesma cadeia Spark: '
    f'Renda Presumida={bases_cenarios_v5["RENDA_PRESUMIDA"]}; '
    f'Entradas Realizadas={valor_entradas_realizadas_v5}.'
)


---
# Bloco 5 — Dashboard Visual HTML (doc_html_spark)
Montagem e renderização do dashboard autocontido a partir das views SQL.


In [ ]:
%%spark

# Bloco exclusivamente explicativo. Nenhuma view criada abaixo alimenta o motor.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_transacoes AS
SELECT
    m.NR_TRAN_INST_PCT,
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    m.IN_JANELA,
    CAST(p.TX_DCR_TRAN_OGNL AS STRING) AS TX_DCR_TRAN_OGNL,
    CAST(p.NR_MCA_PCT_OPB AS STRING) AS NR_MCA_PCT_OPB,
    c.CD_GRUPO,
    c.TX_GRUPO,
    c.CD_IR,
    c.TX_IR,
    COALESCE(c.TX_CATEGORIA, 'Sem Categoria') AS TX_CATEGORIA,
    c.CD_CLASS_RADAR AS CD_CLASS_RADAR,
    c.TX_CLASS_RADAR AS TX_CLASS_RADAR,
    CASE
      WHEN c.CD_CLASS_RADAR IS NOT NULL AND c.TX_CLASS_RADAR IS NOT NULL THEN 'S'
      ELSE 'N'
    END AS IN_CLASSIFICADA_APRESENTACAO,
    COALESCE(c.IN_AGRO, 'N') AS IN_AGRO,
    COALESCE(c.IN_PARTICIPA_CALCULO, 'N') AS IN_PARTICIPA_CALCULO,
    COALESCE(c.IN_PARTICIPA_ORCAMENTO, 'N') AS IN_PARTICIPA_ORCAMENTO,
    r.TIPO_CONSUMO
FROM vw_mov_marcado m
LEFT JOIN vw_q5_mov_contexto_apresentacao p
  ON m.NR_TRAN_INST_PCT = p.NR_TRAN_INST_PCT
LEFT JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
LEFT JOIN vw_ids_consumidos_todos r
  ON m.NR_TRAN_INST_PCT = r.NR_TRAN_INST_PCT
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_transacoes_efetivas AS
SELECT *
FROM vw_dashboard_transacoes
WHERE IN_JANELA = 'S'
  AND TIPO_CONSUMO IS NULL
  AND CD_TIP_MOE_CRR = 'BRL'
""")

# Pareamento explicativo dos IDs exatos que o motor já selecionou.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_pares_exatos AS
WITH consumidas AS (
    SELECT
        m.*,
        e.TIPO_CONSUMO,
        ROW_NUMBER() OVER (
            PARTITION BY m.CD_CLI, m.DT_TRAN, m.VL_TRAN,
                         m.CD_TIP_MOE_CRR, m.IN_JANELA, m.CD_NTZ_CTB_TRAN
            ORDER BY m.NR_TRAN_INST_PCT
        ) AS RN_PAR
    FROM vw_mov_marcado m
    INNER JOIN vw_ids_consumidos_exatos e
      ON m.NR_TRAN_INST_PCT = e.NR_TRAN_INST_PCT
),
pares AS (
    SELECT
        c.TIPO_CONSUMO,
        c.IN_JANELA,
        c.NR_TRAN_INST_PCT AS ID_CREDITO,
        d.NR_TRAN_INST_PCT AS ID_DEBITO,
        c.DT_TRAN,
        c.VL_TRAN,
        c.CD_TIP_MOE_CRR
    FROM consumidas c
    INNER JOIN consumidas d
      ON c.CD_CLI = d.CD_CLI
     AND c.DT_TRAN = d.DT_TRAN
     AND c.VL_TRAN = d.VL_TRAN
     AND c.CD_TIP_MOE_CRR = d.CD_TIP_MOE_CRR
     AND c.IN_JANELA = d.IN_JANELA
     AND c.RN_PAR = d.RN_PAR
    WHERE c.CD_NTZ_CTB_TRAN = 'C'
      AND d.CD_NTZ_CTB_TRAN = 'D'
)
SELECT
    x.*,
    CAST(pc.TX_DCR_TRAN_OGNL AS STRING) AS DESC_CREDITO,
    CAST(pc.NR_MCA_PCT_OPB AS STRING) AS BANCO_CREDITO,
    CAST(pd.TX_DCR_TRAN_OGNL AS STRING) AS DESC_DEBITO,
    CAST(pd.NR_MCA_PCT_OPB AS STRING) AS BANCO_DEBITO
FROM pares x
LEFT JOIN vw_q5_mov_contexto_apresentacao pc ON x.ID_CREDITO = pc.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao pd ON x.ID_DEBITO = pd.NR_TRAN_INST_PCT
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_pares_borda AS
SELECT
    b.NR_TRAN_DENTRO,
    b.NR_TRAN_FORA,
    b.DT_TRAN_DENTRO,
    b.DT_TRAN_FORA,
    b.DIF_DIAS,
    md.CD_NTZ_CTB_TRAN AS NTZ_DENTRO,
    mf.CD_NTZ_CTB_TRAN AS NTZ_FORA,
    md.VL_TRAN,
    md.CD_TIP_MOE_CRR,
    CAST(pd.TX_DCR_TRAN_OGNL AS STRING) AS DESC_DENTRO,
    CAST(pd.NR_MCA_PCT_OPB AS STRING) AS BANCO_DENTRO,
    CAST(pf.TX_DCR_TRAN_OGNL AS STRING) AS DESC_FORA,
    CAST(pf.NR_MCA_PCT_OPB AS STRING) AS BANCO_FORA
FROM vw_pares_borda_calculados b
INNER JOIN vw_mov_marcado md ON b.NR_TRAN_DENTRO = md.NR_TRAN_INST_PCT
INNER JOIN vw_mov_marcado mf ON b.NR_TRAN_FORA = mf.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao pd ON b.NR_TRAN_DENTRO = pd.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao pf ON b.NR_TRAN_FORA = pf.NR_TRAN_INST_PCT
""")

df_dashboard_pivot = spark.sql("""
SELECT
    CD_NTZ_CTB_TRAN AS ntz,
    CD_CLASS_RADAR AS cd_classe,
    TX_CLASS_RADAR AS tx_classe,
    IN_CLASSIFICADA_APRESENTACAO AS classificada,
    CD_CTGR_TRAN_OGNL AS cd_cat,
    TX_CATEGORIA AS tx_cat,
    IN_PARTICIPA_CALCULO AS part_calc,
    IN_PARTICIPA_ORCAMENTO AS part_orc,
    COUNT(1) AS qt,
    SUM(VL_TRAN) AS vl_mov,
    SUM(CASE WHEN IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN ELSE 0 END) AS vl_tematico,
    SUM(CASE WHEN IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN ELSE 0 END) AS vl_orcamentario
FROM vw_dashboard_transacoes_efetivas
GROUP BY CD_NTZ_CTB_TRAN, CD_CLASS_RADAR, TX_CLASS_RADAR,
         IN_CLASSIFICADA_APRESENTACAO,
         CD_CTGR_TRAN_OGNL, TX_CATEGORIA,
         IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO
ORDER BY CD_NTZ_CTB_TRAN, CD_CLASS_RADAR, CD_CTGR_TRAN_OGNL
""")

lista_dashboard_pivot = [r.asDict() for r in df_dashboard_pivot.collect()]
lista_dashboard_transacoes = [r.asDict() for r in spark.sql("""
SELECT * FROM vw_dashboard_transacoes_efetivas
ORDER BY CD_NTZ_CTB_TRAN, CD_CLASS_RADAR, CD_CTGR_TRAN_OGNL,
         DT_TRAN, NR_TRAN_INST_PCT
""").collect()]
lista_dashboard_exatos = [r.asDict() for r in spark.sql("""
SELECT * FROM vw_dashboard_pares_exatos
ORDER BY IN_JANELA DESC, DT_TRAN, ID_CREDITO, ID_DEBITO
""").collect()]
lista_dashboard_borda = [r.asDict() for r in spark.sql("""
SELECT * FROM vw_dashboard_pares_borda
ORDER BY DT_TRAN_DENTRO, NR_TRAN_DENTRO, NR_TRAN_FORA
""").collect()]
lista_dashboard_externas = [r.asDict() for r in spark.sql("""
SELECT
    t.*,
    CASE
      WHEN t.TIPO_CONSUMO = 'BORDA_CONTEXTO' THEN 'Contraparte de borda'
      WHEN t.TIPO_CONSUMO = 'EXATO_CONTEXTO' THEN 'Consumida em par exato externo'
      ELSE 'Não utilizada'
    END AS USO_CONTEXTO
FROM vw_dashboard_transacoes t
WHERE t.IN_JANELA = 'N'
ORDER BY t.DT_TRAN, t.NR_TRAN_INST_PCT
""").collect()]

res_dict = dict(resultados_cenarios_v5['RENDA_PRESUMIDA'])

def esc(valor):
    return html.escape('—' if valor is None else str(valor), quote=True)

def fmt_data(valor, curta=False):
    if valor is None:
        return '—'
    if isinstance(valor, str):
        try:
            valor = datetime.date.fromisoformat(valor[:10])
        except ValueError:
            return esc(valor)
    return valor.strftime('%d/%m' if curta else '%d/%m/%Y')

def fmt_moeda(valor, sinal=False):
    if valor is None:
        return '—'
    numero = Decimal(str(valor))
    prefixo = ''
    if sinal:
        prefixo = '+ ' if numero > 0 else ('− ' if numero < 0 else '')
    numero = abs(numero) if sinal else numero
    texto = f'{numero:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'{prefixo}R$ {texto}'

def fmt_percentual(valor):
    if valor is None:
        return '—'
    texto = f'{Decimal(str(valor)) * Decimal("100"):.2f}'.replace('.', ',')
    return f'{texto}%'

def fmt_inteiro(valor):
    return '—' if valor is None else str(int(valor))

def nome_natureza(codigo):
    return {'C': 'Crédito', 'D': 'Débito'}.get(codigo, 'Natureza não reconhecida')

def somar_metricas(linhas, campo):
    valores = [r[campo] for r in linhas if r[campo] is not None]
    return sum((Decimal(str(v)) for v in valores), Decimal('0')) if valores else None

def metricas_html(linhas):
    return (
        f'<b>{sum(int(r["qt"]) for r in linhas)}</b> tx · '
        f'Mov. {fmt_moeda(somar_metricas(linhas, "vl_mov"))} · '
        f'Tema {fmt_moeda(somar_metricas(linhas, "vl_tematico"))} · '
        f'Orç. {fmt_moeda(somar_metricas(linhas, "vl_orcamentario"))}'
    )

def rotulo_tratamento(transacao):
    tema = transacao.get('IN_PARTICIPA_CALCULO') == 'S'
    orcamento = transacao.get('IN_PARTICIPA_ORCAMENTO') == 'S'
    classificada = transacao.get('IN_CLASSIFICADA_APRESENTACAO') == 'S'
    if tema and orcamento:
        return 'Tema + Orçamento'
    if tema:
        return 'Tema / Fora do orçamento'
    if orcamento:
        return 'Fora do tema / Orçamento'
    if classificada:
        return 'Classificada / Fora tema e orçamento'
    return 'Sem classificação / Fora tema e orçamento'

def rotulo_classe(cd_classe, tx_classe, classificada):
    if classificada != 'S' or cd_classe is None or tx_classe is None:
        return 'Sem classificação'
    return str(tx_classe)

def tabela_transacoes_html(transacoes):
    if not transacoes:
        corpo = '<tr><td colspan="7">Nenhuma transação nesta categoria.</td></tr>'
    else:
        corpo = ''.join(
            '<tr>'
            f'<td>{fmt_data(t["DT_TRAN"])}</td>'
            f'<td>{esc(nome_natureza(t["CD_NTZ_CTB_TRAN"]))}</td>'
            f'<td class="desc">{esc(t["TX_DCR_TRAN_OGNL"])}</td>'
            f'<td><code>{esc(t["NR_MCA_PCT_OPB"])}</code></td>'
            f'<td>{fmt_moeda(t["VL_TRAN"])}</td>'
            f'<td>{esc(t["CD_TIP_MOE_CRR"])}</td>'
            f'<td>{esc(rotulo_tratamento(t))}</td>'
            '</tr>'
            for t in transacoes
        )
    return (
        '<div class="tx-wrap"><table class="tx-table"><thead><tr>'
        '<th>Data</th><th>Natureza</th>'
        '<th>Descrição original<br/><small>TX_DCR_TRAN_OGNL</small></th>'
        '<th>Banco / Marca<br/><small>NR_MCA_PCT_OPB</small></th>'
        '<th>Valor</th><th>Moeda</th><th>Tratamento</th>'
        f'</tr></thead><tbody>{corpo}</tbody></table></div>'
    )

def render_composicao():
    blocos_natureza = []
    for ntz, titulo in [('C', 'Entradas'), ('D', 'Saídas')]:
        linhas_ntz = [r for r in lista_dashboard_pivot if r['ntz'] == ntz]
        if not linhas_ntz:
            continue
        blocos_classe = []
        classes = []
        for r in linhas_ntz:
            chave = (r['cd_classe'], r['tx_classe'], r['classificada'])
            if chave not in classes:
                classes.append(chave)
        for cd_classe, tx_classe, classificada in classes:
            linhas_classe = [
                r for r in linhas_ntz
                if (r['cd_classe'], r['tx_classe'], r['classificada'])
                == (cd_classe, tx_classe, classificada)
            ]
            blocos_categoria = []
            for cat in linhas_classe:
                transacoes = [
                    t for t in lista_dashboard_transacoes
                    if t['CD_NTZ_CTB_TRAN'] == ntz
                    and t['CD_CLASS_RADAR'] == cd_classe
                    and t['IN_CLASSIFICADA_APRESENTACAO'] == classificada
                    and t['CD_CTGR_TRAN_OGNL'] == cat['cd_cat']
                ]
                codigo = '—' if cat['cd_cat'] is None else str(cat['cd_cat'])
                blocos_categoria.append(
                    '<details class="level category" data-depth="category">'
                    f'<summary><span>{esc(codigo)} — {esc(cat["tx_cat"])}</span>'
                    f'<span class="metrics">{metricas_html([cat])}</span></summary>'
                    '<div class="detail-body"><div class="mini-grid">'
                    f'<div><span>Valor movimentado</span><strong>{fmt_moeda(cat["vl_mov"])}</strong></div>'
                    f'<div><span>Valor temático</span><strong>{fmt_moeda(cat["vl_tematico"])}</strong></div>'
                    f'<div><span>Valor orçamentário</span><strong>{fmt_moeda(cat["vl_orcamentario"])}</strong></div>'
                    f'</div>{tabela_transacoes_html(transacoes)}</div></details>'
                )
            blocos_classe.append(
                '<details class="level class" data-depth="class">'
                f'<summary><span>{esc(rotulo_classe(cd_classe, tx_classe, classificada))}</span>'
                f'<span class="metrics">{metricas_html(linhas_classe)}</span></summary>'
                f'<div class="detail-body">{"".join(blocos_categoria)}</div></details>'
            )
        total_orc = res_dict.get('VL_ENT_TOTAL' if ntz == 'C' else 'VL_SAI_TOTAL')
        blocos_natureza.append(
            '<details class="level nature" data-depth="nature">'
            f'<summary><span>{titulo}</span><span class="metrics">'
            f'{sum(int(r["qt"]) for r in linhas_ntz)} tx · Orçamento {fmt_moeda(total_orc)}</span></summary>'
            f'<div class="detail-body">{"".join(blocos_classe)}</div></details>'
        )
    if not blocos_natureza:
        return '<div class="result-note">Nenhuma transação efetiva em BRL para compor.</div>'
    return ''.join(blocos_natureza)

def lado_reconciliacao(titulo, data, ntz, valor, moeda, descricao, banco, fora=False):
    classe = 'side out' if fora else 'side'
    return (
        f'<div class="{classe}"><div class="tx-title">{esc(nome_natureza(ntz))} · {esc(titulo)}</div>'
        f'<div>{fmt_data(data)}</div><div class="tx-value">{fmt_moeda(valor)}</div><div>{esc(moeda)}</div>'
        '<dl><dt>Descrição original<span class="field-tag">TX_DCR_TRAN_OGNL</span></dt>'
        f'<dd>{esc(descricao)}</dd><dt>Banco / Marca<span class="field-tag">NR_MCA_PCT_OPB</span></dt>'
        f'<dd><code>{esc(banco)}</code></dd></dl></div>'
    )

def render_eventos_reconciliacao():
    eventos = []
    for par in lista_dashboard_exatos:
        if par['IN_JANELA'] != 'S':
            continue
        eventos.append(
            '<details class="event"><summary>'
            f'Par exato · {fmt_moeda(par["VL_TRAN"])} · {fmt_data(par["DT_TRAN"])}</summary>'
            '<div class="event-body"><div class="pair">'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN'], 'D', par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_DEBITO'], par['BANCO_DEBITO'])
            + '<div class="link">↔</div>'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN'], 'C', par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_CREDITO'], par['BANCO_CREDITO'])
            + '</div><div class="checks">✓ mesma data · ✓ mesmo valor · ✓ mesma moeda · ✓ naturezas opostas</div>'
            '<div class="result-note">Resultado: as duas movimentações foram anuladas e não participaram dos cálculos.</div>'
            '</div></details>'
        )
    for par in lista_dashboard_borda:
        eventos.append(
            '<details class="event"><summary>'
            f'Reconciliação de borda · {fmt_moeda(par["VL_TRAN"])} · diferença de {par["DIF_DIAS"]} dias</summary>'
            '<div class="event-body"><div class="pair">'
            + lado_reconciliacao('Fora do ciclo', par['DT_TRAN_FORA'], par['NTZ_FORA'], par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_FORA'], par['BANCO_FORA'], True)
            + f'<div class="link">{par["DIF_DIAS"]} dias<br/>↔</div>'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN_DENTRO'], par['NTZ_DENTRO'], par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_DENTRO'], par['BANCO_DENTRO'])
            + '</div><div class="checks">✓ mesmo valor · ✓ mesma moeda · ✓ naturezas opostas · ✓ diferença dentro de 5 dias</div>'
            '<div class="result-note">A movimentação oficial foi anulada. A contraparte externa foi usada somente como evidência e não entrou nos cálculos financeiros.</div>'
            '</div></details>'
        )
    return ''.join(eventos) or '<div class="result-note">Nenhum evento de reconciliação removeu movimentações oficiais.</div>'

def render_contexto_externo():
    linhas = ''.join(
        '<tr>'
        f'<td>{fmt_data(t["DT_TRAN"])}</td><td>{esc(nome_natureza(t["CD_NTZ_CTB_TRAN"]))}</td>'
        f'<td class="desc">{esc(t["TX_DCR_TRAN_OGNL"])}</td><td><code>{esc(t["NR_MCA_PCT_OPB"])}</code></td>'
        f'<td>{fmt_moeda(t["VL_TRAN"])}</td><td>{esc(t["CD_TIP_MOE_CRR"])}</td><td>{esc(t["USO_CONTEXTO"])}</td>'
        '</tr>'
        for t in lista_dashboard_externas
    )
    if not linhas:
        linhas = '<tr><td colspan="7">Nenhuma movimentação fora do ciclo.</td></tr>'
    return (
        f'<details class="event"><summary>Contexto externo analisado · {len(lista_dashboard_externas)} movimentações</summary>'
        '<div class="event-body"><div class="ext-table"><table><thead><tr>'
        '<th>Data</th><th>Natureza</th><th>Descrição original<br/><small>TX_DCR_TRAN_OGNL</small></th>'
        '<th>Banco / Marca<br/><small>NR_MCA_PCT_OPB</small></th><th>Valor</th><th>Moeda</th><th>Uso no contexto</th>'
        f'</tr></thead><tbody>{linhas}</tbody></table></div>'
        '<div class="result-note" style="margin-top:12px">Nenhuma movimentação externa participou dos cálculos financeiros do ciclo.</div>'
        '</div></details>'
    )

temas = [
    (1, 'Categorização dos Gastos', 'VL_SAI_IND', 'PC_SAI_IND', 'PC_REF_IND', 'NR_PONT_CONC_IND', None, None, 'NR_PONT_IND_FIM'),
    (2, 'Gestão de Orçamento', 'VL_SAI_ESS', 'PC_SAI_ESS', 'PC_REF_ESS', 'NR_PONT_CONC_ESS', 'NR_PONT_ORC_ESS', 'NR_PONT_PRFL_ESS', 'NR_PONT_ESS_FIM'),
    (3, 'Consumo Planejado', 'VL_SAI_NAO_ESS', 'PC_SAI_NAO_ESS', 'PC_REF_NAO_ESS', 'NR_PONT_CONC_NAO_ESS', 'NR_PONT_ORC_NAO_ESS', 'NR_PONT_PRFL_NAO_ESS', 'NR_PONT_NAO_ESS_FIM'),
    (4, 'Formação de Reserva', 'VL_SAI_FUT', 'PC_SAI_FUT', 'PC_REF_FUT', 'NR_PONT_CONC_FUT', 'NR_PONT_ORC_FUT', 'NR_PONT_PRFL_FUT', 'NR_PONT_FUT_FIM'),
    (5, 'Uso Consciente do Crédito', 'VL_SAI_OBR', 'PC_SAI_OBR', 'PC_REF_OBR', 'NR_PONT_CONC_OBR', 'NR_PONT_ORC_OBR', 'NR_PONT_PRFL_OBR', 'NR_PONT_OBR_FIM'),
]

def render_linhas_pontuacao(resultado):
    linhas = []
    for codigo, nome, vl, pc, ref, conc, orc, prfl, final in temas:
        vencedor = (
            resultado.get('CD_TEMA_VENCEDOR') == codigo
            or (
                resultado.get('CD_TEMA_VENCEDOR') == 9
                and resultado.get(final) is not None
                and resultado.get(final) == resultado.get('NR_PONT_MAX')
            )
        )
        classe = ' class="winner-row"' if vencedor else ''
        valor_final = fmt_inteiro(resultado.get(final))
        celula_final = f'<span class="score-final-badge">{valor_final}</span>' if vencedor else valor_final
        regra_final = 'Final = Concentração' if codigo == 1 else 'Final = Concentração + Orçamento + Perfil'
        linhas.append(
            f'<tr{classe}><td class="theme-cell">{esc(nome)}<small>{esc(regra_final)}</small></td>'
            f'<td class="num">{fmt_moeda(resultado.get(vl))}</td>'
            f'<td class="num">{fmt_percentual(resultado.get(pc))}</td>'
            f'<td class="num">{fmt_percentual(resultado.get(ref))}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(conc))}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(orc)) if orc else "—"}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(prfl)) if prfl else "—"}</td>'
            f'<td class="final-cell">{celula_final}</td></tr>'
        )
    return ''.join(linhas)

def resolver_fechamento_pontuacao(resultado):
    fl_completa = resultado.get('FL_PONTUACAO_COMPLETA')
    pont_max = resultado.get('NR_PONT_MAX')
    qt_max = resultado.get('QT_TEMAS_PONT_MAX')
    cd_vencedor = resultado.get('CD_TEMA_VENCEDOR')
    tx_vencedor = resultado.get('TX_TEMA_VENCEDOR')
    finais = (pont_max, qt_max, cd_vencedor, tx_vencedor)
    nomes_por_codigo = {codigo: nome for codigo, nome, *_ in temas}

    if fl_completa == 'N':
        if any(valor is not None for valor in finais):
            raise RuntimeError('Estado final inconsistente: pontuação incompleta com vencedor preenchido.')
        return {
            'estado': 'incompleta',
            'kicker': 'Pontuação incompleta',
            'titulo': 'Sem tema vencedor',
            'pontos': '—',
            'mensagem': 'Um ou mais temas não possuem pontuação final; o contrato não define vencedor.',
        }

    if fl_completa != 'S':
        raise RuntimeError(f'Estado final inconsistente: FL_PONTUACAO_COMPLETA={fl_completa}.')
    if pont_max is None or qt_max is None:
        raise RuntimeError('Estado final inconsistente: pontuação completa sem máximo ou quantidade.')
    if 2 <= int(qt_max) <= 5:
        if cd_vencedor != 9 or tx_vencedor != 'Empate':
            raise RuntimeError('Estado final inconsistente: empate sem código 9 e texto Empate.')
        return {
            'estado': 'empate',
            'kicker': 'Empate',
            'titulo': 'Nenhum vencedor isolado',
            'pontos': f'{fmt_inteiro(pont_max)} pontos',
            'mensagem': f'{fmt_inteiro(qt_max)} temas atingiram a pontuação máxima.',
        }
    if int(qt_max) == 1:
        if cd_vencedor not in nomes_por_codigo or tx_vencedor != nomes_por_codigo[cd_vencedor]:
            raise RuntimeError('Estado final inconsistente: vencedor único não corresponde ao contrato.')
        return {
            'estado': 'vencedor',
            'kicker': 'Tema de destaque',
            'titulo': str(tx_vencedor),
            'pontos': f'{fmt_inteiro(pont_max)} pontos',
            'mensagem': 'Maior pontuação entre os cinco temas · Sem empate.',
        }
    raise RuntimeError(f'Estado final inconsistente: QT_TEMAS_PONT_MAX={qt_max}.')

def render_fechamento_pontuacao(resultado):
    fechamento = resolver_fechamento_pontuacao(resultado)
    campos = [
        ('FL_PONTUACAO_COMPLETA', resultado.get('FL_PONTUACAO_COMPLETA')),
        ('NR_PONT_MAX', resultado.get('NR_PONT_MAX')),
        ('QT_TEMAS_PONT_MAX', resultado.get('QT_TEMAS_PONT_MAX')),
        ('CD_TEMA_VENCEDOR', resultado.get('CD_TEMA_VENCEDOR')),
        ('TX_TEMA_VENCEDOR', resultado.get('TX_TEMA_VENCEDOR')),
    ]
    memoria = ''.join(
        f'<div><span><code>{esc(campo)}</code></span><strong>{esc(valor)}</strong></div>'
        for campo, valor in campos
    )
    return (
        '<section class="section" data-section="fechamento-pontuacao">'
        '<div class="section-head"><div><span class="section-label">Fechamento</span>'
        '<h2>Decisão da pontuação</h2></div></div><div class="score-layout">'
        '<article class="winner-card">'
        f'<div class="kicker">{esc(fechamento["kicker"])}</div>'
        f'<h3>{esc(fechamento["titulo"])}</h3><div class="big">{esc(fechamento["pontos"])}</div>'
        f'<p>{esc(fechamento["mensagem"])}</p></article>'
        '<article class="trace-main"><h3>Fechamento contratual</h3>'
        '<p>Campos finais produzidos pelo motor, sem regra visual substitutiva.</p>'
        f'<div class="mini-grid">{memoria}</div>'
        '<div class="formula"><span>IND: Final = Concentração</span><b>≠</b>'
        '<span>Demais temas: Final = Concentração + Orçamento + Perfil</span></div>'
        '<div class="rule-note">Regra especial de IND: o final de Categorização dos Gastos usa somente a pontuação de concentração.</div>'
        '</article></div></section>'
    )

linhas_auditoria = ''.join(
    '<tr>'
    f'<td>{i + 1}</td><td><code>{esc(campo)}</code></td>'
    f'<td>{esc(df_res_80.schema[campo].dataType.simpleString().upper())}</td>'
    f'<td>{esc(resultado_oficial_v4_v5.get(campo))}</td></tr>'
    for i, campo in enumerate(df_res_80.columns)
)

def tag_execucao(rotulo, valor, positivo='S'):
    classe = ' ok' if valor == positivo else ''
    prefixo = '✓ ' if classe else ''
    exibido = {'S': 'Sim', 'N': 'Não', None: '—'}.get(valor, str(valor))
    return f'<span class="run-tag{classe}">{prefixo}{esc(rotulo)}: {esc(exibido)}</span>'

dt_contexto_ini = dt_ini_j - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO) if dt_ini_j else None
dt_contexto_fim = dt_fim_j + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO) if dt_fim_j else None
fallback_usado = (
    '—' if res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK') is None
    else ('Sim' if res_dict.get('DD_INC_MM_CLC_BLC') is None else 'Não')
)
def texto_payload_v5(valor):
    return '—' if valor is None else str(valor)

def montar_payload_cenario_v5(chave, resultado):
    presumida = chave == 'RENDA_PRESUMIDA'
    base_financeira = bases_cenarios_v5[chave]
    pc_saida_entrada_cenario = resultado.get('PC_SAI_ENT')
    largura_cenario = (
        0 if pc_saida_entrada_cenario is None
        else max(0, min(100, float(pc_saida_entrada_cenario) * 100))
    )
    rotulo = 'Renda Presumida' if presumida else 'Entradas Realizadas'
    referencia_rotulo = 'Referência da renda' if presumida else 'Origem da base'
    referencia_valor = (
        fmt_data(resultado.get('DT_REN_PRES_REF'))
        if presumida
        else f'Entradas efetivas do ciclo · {fmt_data(resultado.get("DT_REF_INI"), True)} → {fmt_data(resultado.get("DT_REF_FIM"), True)}'
    )
    audit_text = (
        'Leitura V5 calculada com VL_REN_PRES; a tabela técnica abaixo permanece sendo a view oficial V4.'
        if presumida
        else 'Leitura V5 calculada com créditos efetivos, classificados e BRL; a view oficial V4 não é modificada.'
    )
    return {
        'base_kicker': 'Base financeira',
        'base_badge': f'{rotulo} ativa',
        'valor_base': fmt_moeda(base_financeira),
        'referencia_rotulo': referencia_rotulo,
        'referencia_valor': referencia_valor,
        'entrada_rotulo': 'Base financeira',
        'entrada_valor': fmt_moeda(base_financeira),
        'saldo_status': texto_payload_v5(resultado.get('TX_STS_FINAL')),
        'saldo_valor': fmt_moeda(resultado.get('VL_RES_ORC'), True),
        'razao_valor': fmt_percentual(pc_saida_entrada_cenario),
        'largura_medidor': f'{largura_cenario:.2f}%',
        'pontuacao_html': render_linhas_pontuacao(resultado),
        'fechamento_html': render_fechamento_pontuacao(resultado),
        'tag_base_html': f'<span class="run-tag base-active">Base ativa: {esc(rotulo)}</span>',
        'tag_pontuacao_html': tag_execucao('Pontuação completa', resultado.get('FL_PONTUACAO_COMPLETA')),
        'audit_titulo': f'Cenário exibido: {rotulo}',
        'audit_texto': audit_text,
    }

payload_cenarios_v5 = {
    chave: montar_payload_cenario_v5(chave, resultados_cenarios_v5[chave])
    for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS')
}
payload_padrao_v5 = payload_cenarios_v5['RENDA_PRESUMIDA']
payload_cenarios_json_v5 = (
    json.dumps(payload_cenarios_v5, ensure_ascii=False, separators=(',', ':'))
    .replace('&', '\\u0026')
    .replace('<', '\\u003c')
    .replace('>', '\\u003e')
)

# Compatibilidade com validações visuais herdadas.
linhas_pontuacao = payload_padrao_v5['pontuacao_html']
pc_saida_entrada = res_dict.get('PC_SAI_ENT')
largura_medidor = 0 if pc_saida_entrada is None else max(0, min(100, float(pc_saida_entrada) * 100))

CSS_COMPLEMENTO_V5 = r"""
[data-radar-root="v3"] .base-switch{display:grid;grid-template-columns:1fr 1fr;gap:4px;margin-top:12px;padding:4px;background:#f0f3f8;border:1px solid var(--line);border-radius:12px}
[data-radar-root="v3"] .base-switch-btn{border:0;background:transparent;color:var(--muted);border-radius:9px;padding:7px 6px;font-size:9px;font-weight:850;line-height:1.15;cursor:pointer}
[data-radar-root="v3"] .base-switch-btn.active{background:#fff;color:var(--bb-blue-deep);box-shadow:0 2px 8px rgba(27,36,74,.12)}
[data-radar-root="v3"] .base-switch-btn:focus-visible{outline:3px solid rgba(70,94,255,.2);outline-offset:1px}
[data-radar-root="v3"] .scenario-active-tag{display:inline-flex;align-items:center;border:1px solid #cce7da;background:var(--good-bg);color:var(--good);border-radius:999px;padding:5px 8px;font-size:9px;font-weight:850}
[data-radar-root="v3"] .scenario-audit{display:flex;align-items:center;justify-content:space-between;gap:14px;margin-bottom:10px;padding:12px 14px;border:1px solid #dce3ff;background:#f7f8ff;border-radius:14px}
[data-radar-root="v3"] .scenario-audit strong,[data-radar-root="v3"] .scenario-audit span{display:block}
[data-radar-root="v3"] .scenario-audit strong{font-size:11px;color:var(--bb-blue-deep)}
[data-radar-root="v3"] .scenario-audit span{font-size:9px;color:var(--muted);margin-top:2px}
[data-radar-root="v3"] .run-tag.base-active{background:#eef8f3;border-color:#cce7da;color:#176848}
@media(max-width:760px){[data-radar-root="v3"] .scenario-audit{align-items:flex-start;flex-direction:column}}
"""

CSS_APROVADO_V3 = '[data-radar-root="v3"]{\n  --bb-blue:#465eff;--bb-blue-deep:#252d84;--bb-yellow:#fcfc30;--ink:#13162b;--muted:#667085;\n  --bg:#f4f6fb;--card:#fff;--line:#e3e7ef;--soft:#f8f9fc;--good:#087a55;--good-bg:#e8f7f0;\n  --warn:#8c5a00;--warn-bg:#fff5d9;--danger:#b42318;--danger-bg:#fff0ee;--shadow:0 14px 40px rgba(27,36,74,.08);\n  --radius:22px;--radius-sm:14px;--max:1280px\n}\n[data-radar-root="v3"],[data-radar-root="v3"] *{box-sizing:border-box}\n[data-radar-root="v3"]{scroll-behavior:smooth}\n[data-radar-root="v3"]{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",Arial,sans-serif;line-height:1.45}\n[data-radar-root="v3"] button,[data-radar-root="v3"] input{font:inherit}\n[data-radar-root="v3"] a{color:inherit}\n[data-radar-root="v3"] .shell{max-width:var(--max);margin:auto;padding:0 24px 84px}\n[data-radar-root="v3"] .skip{position:absolute;left:-9999px;top:auto}\n[data-radar-root="v3"] .skip:focus{left:16px;top:16px;z-index:9999;background:#fff;padding:10px 14px;border-radius:10px}\n[data-radar-root="v3"] .topbar{position:sticky;top:0;z-index:50;background:rgba(244,246,251,.88);backdrop-filter:blur(18px);border-bottom:1px solid rgba(227,231,239,.8)}\n[data-radar-root="v3"] .topbar-inner{max-width:var(--max);margin:auto;padding:12px 24px;display:flex;gap:14px;align-items:center;justify-content:space-between}\n[data-radar-root="v3"] .brand{display:flex;align-items:center;gap:10px;font-weight:850;white-space:nowrap}\n[data-radar-root="v3"] .brand-mark{width:28px;height:28px;border-radius:9px;background:var(--bb-yellow);border:7px solid var(--bb-blue);transform:rotate(45deg);box-shadow:inset 0 0 0 2px #fff}\n[data-radar-root="v3"] .nav{display:flex;gap:4px;overflow:auto;scrollbar-width:none}\n[data-radar-root="v3"] .nav::-webkit-scrollbar{display:none}\n[data-radar-root="v3"] .nav a{text-decoration:none;color:#525b75;padding:8px 10px;border-radius:10px;font-size:12px;font-weight:720;white-space:nowrap}\n[data-radar-root="v3"] .nav a:hover,[data-radar-root="v3"] .nav a.active{background:#fff;color:var(--bb-blue-deep);box-shadow:0 1px 4px rgba(0,0,0,.06)}\n[data-radar-root="v3"] .privacy{border:1px solid var(--line);background:#fff;border-radius:999px;padding:8px 11px;font-weight:750;font-size:12px;cursor:pointer;white-space:nowrap}\n[data-radar-root="v3"] .hero{margin-top:24px;background:linear-gradient(135deg,var(--bb-blue-deep),#3342b6 52%,var(--bb-blue));color:#fff;border-radius:28px;padding:26px 28px;box-shadow:0 20px 60px rgba(37,45,132,.18);position:relative;overflow:hidden}\n[data-radar-root="v3"] .hero:after{content:"";position:absolute;width:320px;height:320px;border-radius:50%;background:var(--bb-yellow);right:-165px;top:-175px;opacity:.96}\n[data-radar-root="v3"] .hero-grid{display:grid;grid-template-columns:1.4fr .8fr;gap:28px;position:relative;z-index:1}\n[data-radar-root="v3"] .eyebrow{font-size:10px;font-weight:850;letter-spacing:.12em;text-transform:uppercase;opacity:.72}\n[data-radar-root="v3"] .hero h1{margin:7px 0 14px;font-size:clamp(30px,4vw,48px);line-height:1.02;letter-spacing:-.04em}\n[data-radar-root="v3"] .hero p{margin:0;color:rgba(255,255,255,.76);max-width:760px}\n[data-radar-root="v3"] .hero-status{align-self:end;background:rgba(255,255,255,.1);border:1px solid rgba(255,255,255,.18);padding:18px;border-radius:18px}\n[data-radar-root="v3"] .hero-status small{display:block;color:rgba(255,255,255,.7)}\n[data-radar-root="v3"] .hero-status strong{font-size:21px;display:block;margin:3px 0}\n[data-radar-root="v3"] .hero-status span{font-size:12px}\n[data-radar-root="v3"] .hero-chips{display:flex;gap:7px;flex-wrap:wrap;margin-top:0}\n[data-radar-root="v3"] .chip{border:1px solid rgba(255,255,255,.2);background:rgba(255,255,255,.1);padding:6px 9px;border-radius:999px;font-size:10px;font-weight:720}\n[data-radar-root="v3"] .context-strip{display:flex;align-items:stretch;gap:0;background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}\n[data-radar-root="v3"] .context-item{flex:1;min-width:0;padding:14px 16px;border-right:1px solid var(--line)}\n[data-radar-root="v3"] .context-item:last-child{border-right:0}\n[data-radar-root="v3"] .context-label{font-size:9px;color:var(--muted);font-weight:850;text-transform:uppercase;letter-spacing:.07em}\n[data-radar-root="v3"] .context-value{font-size:14px;font-weight:820;margin-top:4px;overflow-wrap:anywhere}\n[data-radar-root="v3"] .context-meta{font-size:9px;color:var(--muted);margin-top:2px}\n[data-radar-root="v3"] .money-grid{display:grid;grid-template-columns:1fr auto 1fr auto 1.15fr;gap:0;align-items:stretch;background:#fff;border:1px solid var(--line);border-radius:20px;overflow:hidden}\n[data-radar-root="v3"] .money-card{background:#fff;padding:20px 22px;border-right:1px solid var(--line)}\n[data-radar-root="v3"] .money-card:last-child{border-right:0}\n[data-radar-root="v3"] .money-card .label{font-size:10px;color:var(--muted);text-transform:uppercase;font-weight:850;letter-spacing:.08em}\n[data-radar-root="v3"] .money-card .value{font-size:clamp(24px,3vw,36px);font-weight:900;margin:5px 0;letter-spacing:-.04em}\n[data-radar-root="v3"] .money-card .meta{font-size:11px;color:var(--muted)}\n[data-radar-root="v3"] .operator{align-self:center;font-size:22px;color:#a7afc3;font-weight:300;padding:0 10px}\n[data-radar-root="v3"] .money-card.balance-card{background:#f6fbf8}\n[data-radar-root="v3"] .balance-card .value{color:var(--good)}\n[data-radar-root="v3"] .status-pill{display:inline-flex;margin-top:10px;background:var(--good-bg);color:var(--good);padding:7px 10px;border-radius:999px;font-size:11px;font-weight:850}\n[data-radar-root="v3"] .ratio-bar{margin-top:12px;height:8px;background:#e9edf4;border-radius:999px;overflow:hidden}\n[data-radar-root="v3"] .ratio-bar span{display:block;height:100%;width:72.39%;background:var(--bb-blue)}\n[data-radar-root="v3"] .trace-box{display:grid;grid-template-columns:1.25fr .75fr;gap:12px;margin-top:12px}\n[data-radar-root="v3"] .trace-main,[data-radar-root="v3"] .trace-side{background:#fff;border:1px solid var(--line);border-radius:20px;padding:18px}\n[data-radar-root="v3"] .trace-main h3,[data-radar-root="v3"] .trace-side h3{margin:0 0 7px;font-size:15px}\n[data-radar-root="v3"] .trace-main p,[data-radar-root="v3"] .trace-side p{margin:0;color:var(--muted);font-size:12px}\n[data-radar-root="v3"] .formula{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin-top:15px}\n[data-radar-root="v3"] .formula span{background:var(--soft);border:1px solid var(--line);border-radius:11px;padding:8px 10px;font-size:11px;font-weight:800}\n[data-radar-root="v3"] .formula b{color:var(--muted)}\n[data-radar-root="v3"] .controls{display:flex;gap:8px;flex-wrap:wrap;align-items:center}\n[data-radar-root="v3"] .control-btn{border:1px solid var(--line);background:#fff;border-radius:11px;padding:8px 11px;font-size:11px;font-weight:750;cursor:pointer}\n[data-radar-root="v3"] .control-btn:hover{border-color:#bbc3d4}\n[data-radar-root="v3"] .search{border:1px solid var(--line);background:#fff;border-radius:11px;padding:8px 11px;font-size:11px;min-width:220px;outline:none}\n[data-radar-root="v3"] .search:focus{border-color:var(--bb-blue);box-shadow:0 0 0 3px rgba(70,94,255,.12)}\n[data-radar-root="v3"] .explorer{background:#fff;border:1px solid var(--line);border-radius:22px;padding:10px;box-shadow:var(--shadow)}\n[data-radar-root="v3"] .level{background:#fff;border:1px solid var(--line);border-radius:14px;margin:8px 0;overflow:hidden}\n[data-radar-root="v3"] .level summary{cursor:pointer;list-style:none;padding:14px 15px;display:flex;justify-content:space-between;gap:16px;align-items:center}\n[data-radar-root="v3"] .level summary::-webkit-details-marker{display:none}\n[data-radar-root="v3"] .level summary:before{content:"+";display:grid;place-items:center;flex:0 0 24px;height:24px;border-radius:8px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:900}\n[data-radar-root="v3"] .level[open]>summary:before{content:"−"}\n[data-radar-root="v3"] .level>summary{font-weight:750}\n[data-radar-root="v3"] .nature>summary{background:#f4f5ff;font-size:15px}\n[data-radar-root="v3"] .class>summary{background:#fafbff}\n[data-radar-root="v3"] .category>summary{font-weight:650}\n[data-radar-root="v3"] .metrics{font-size:11px;color:var(--muted);font-weight:500;text-align:right;margin-left:auto}\n[data-radar-root="v3"] .detail-body{padding:0 13px 13px 26px}\n[data-radar-root="v3"] .mini-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:8px 0 12px}\n[data-radar-root="v3"] .mini-grid div{background:#f8f9fc;border-radius:11px;padding:10px}\n[data-radar-root="v3"] .mini-grid span{font-size:9px;text-transform:uppercase;color:var(--muted);display:block}\n[data-radar-root="v3"] .mini-grid strong{font-size:13px}\n[data-radar-root="v3"] .tx-wrap,[data-radar-root="v3"] .ext-table{overflow-x:auto;border-radius:12px;border:1px solid var(--line);background:#fff}\n[data-radar-root="v3"] table{width:100%;border-collapse:collapse}\n[data-radar-root="v3"] th,[data-radar-root="v3"] td{padding:10px;border-bottom:1px solid var(--line);font-size:11px;text-align:left;vertical-align:top}\n[data-radar-root="v3"] th{font-size:9px;text-transform:uppercase;letter-spacing:.04em;color:var(--muted);background:#f8f9fc;position:sticky;top:0}\n[data-radar-root="v3"] th small{display:block;font-size:8px;letter-spacing:0;text-transform:none;color:#98a2b3}\n[data-radar-root="v3"] td.desc{min-width:220px;max-width:360px;white-space:normal}\n[data-radar-root="v3"] td code{font-size:10px;white-space:nowrap}\n[data-radar-root="v3"] .sim-note{font-size:10px;color:var(--muted);font-style:italic}\n[data-radar-root="v3"] .score-layout{display:grid;grid-template-columns:1.35fr .65fr;gap:12px}\n[data-radar-root="v3"] .score-list{background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}\n[data-radar-root="v3"] .score-item{background:#fff;padding:15px 16px;border-bottom:1px solid var(--line)}\n[data-radar-root="v3"] .score-item:last-of-type{border-bottom:0}\n[data-radar-root="v3"] .score-head{display:flex;justify-content:space-between;gap:14px;align-items:start}\n[data-radar-root="v3"] .score-head strong{display:block;font-size:13px}\n[data-radar-root="v3"] .score-head span{display:block;font-size:10px;color:var(--muted);margin-top:2px}\n[data-radar-root="v3"] .score-head>b{font-size:21px}\n[data-radar-root="v3"] .score-track{height:7px;background:#edf0f5;border-radius:999px;overflow:hidden;margin:11px 0 8px}\n[data-radar-root="v3"] .score-track span{display:block;height:100%;background:var(--bb-blue);min-width:0}\n[data-radar-root="v3"] .score-memory{display:flex;gap:12px;flex-wrap:wrap;color:var(--muted);font-size:10px}\n[data-radar-root="v3"] .score-memory b{color:var(--ink)}\n[data-radar-root="v3"] .winner-card{background:var(--bb-yellow);border:1px solid #e2df4f;border-radius:18px;padding:20px;position:sticky;top:84px;align-self:start}\n[data-radar-root="v3"] .winner-card .kicker{color:#333}\n[data-radar-root="v3"] .winner-card h3{font-size:25px;margin:6px 0;letter-spacing:-.03em}\n[data-radar-root="v3"] .winner-card .big{font-size:34px;font-weight:900}\n[data-radar-root="v3"] .winner-card p{font-size:11px;margin:8px 0 0;max-width:300px}\n[data-radar-root="v3"] .rule-note{background:#fff8df;border:1px solid #f1df9b;color:#725000;padding:10px 12px;border-radius:11px;font-size:10px;margin-top:10px}\n[data-radar-root="v3"] .recon{background:#fff;border:1px solid var(--line);border-radius:22px;padding:16px;box-shadow:var(--shadow)}\n[data-radar-root="v3"] .funnel{display:flex;align-items:center;justify-content:center;gap:9px;flex-wrap:wrap;margin:4px 0 16px}\n[data-radar-root="v3"] .step{padding:10px 13px;border-radius:11px;background:#f5f7fb;text-align:center;font-size:10px}\n[data-radar-root="v3"] .step b{font-size:18px;display:block}\n[data-radar-root="v3"] .minus{color:var(--danger)}\n[data-radar-root="v3"] .event{border:1px solid var(--line);border-radius:14px;margin:9px 0;overflow:hidden}\n[data-radar-root="v3"] .event summary{cursor:pointer;padding:13px 15px;font-weight:750;background:#fafbfe}\n[data-radar-root="v3"] .event-body{padding:13px 15px}\n[data-radar-root="v3"] .pair{display:grid;grid-template-columns:1fr 58px 1fr;gap:9px;align-items:stretch}\n[data-radar-root="v3"] .side{background:#f7f9fc;border-radius:12px;padding:13px}\n[data-radar-root="v3"] .side.out{background:var(--warn-bg)}\n[data-radar-root="v3"] .link{text-align:center;color:var(--muted);font-weight:850;align-self:center}\n[data-radar-root="v3"] .side .tx-title{font-size:13px;font-weight:850;margin-bottom:6px}\n[data-radar-root="v3"] .side .tx-value{font-size:18px;font-weight:850;margin:4px 0}\n[data-radar-root="v3"] .side dl{display:grid;grid-template-columns:120px 1fr;gap:5px 8px;margin:11px 0 0;font-size:10px}\n[data-radar-root="v3"] .side dt{color:var(--muted)}\n[data-radar-root="v3"] .side dd{margin:0;font-weight:650;word-break:break-word}\n[data-radar-root="v3"] .field-tag{font-size:7px;color:#8b98aa;display:block}\n[data-radar-root="v3"] .checks{margin:11px 0;font-size:10px;color:var(--muted)}\n[data-radar-root="v3"] .result-note{background:var(--good-bg);color:#155e3c;padding:10px;border-radius:10px;font-size:10px}\n[data-radar-root="v3"] .tech{background:#fff;border:1px solid var(--line);border-radius:18px;padding:0;overflow:hidden}\n[data-radar-root="v3"] .tech>summary{cursor:pointer;font-weight:820;padding:15px 17px;background:#fbfcfe}\n[data-radar-root="v3"] .tech .scroll{max-height:620px;overflow:auto;margin:0;border-top:1px solid var(--line)}\n[data-radar-root="v3"] .privacy-on [data-sensitive="true"]{filter:blur(5px);user-select:none}\n[data-radar-root="v3"] .privacy-on .tech tbody tr[data-private="true"] td:last-child{filter:blur(5px);user-select:none}\n[data-radar-root="v3"] mark{background:var(--bb-yellow);color:inherit;padding:0 .08em}\n[data-radar-root="v3"] .section{margin-top:24px}\n[data-radar-root="v3"] .section-head{display:flex;align-items:center;justify-content:space-between;gap:14px;margin-bottom:10px}\n[data-radar-root="v3"] .section-head h2{font-size:18px;letter-spacing:-.02em;margin:0}\n[data-radar-root="v3"] .run-header{margin-top:24px;background:#fff;border:1px solid var(--line);border-radius:20px;padding:17px 19px;display:flex;align-items:center;justify-content:space-between;gap:18px;box-shadow:0 8px 28px rgba(27,36,74,.05)}\n[data-radar-root="v3"] .run-title{display:flex;align-items:baseline;gap:10px;white-space:nowrap}\n[data-radar-root="v3"] .run-title .eyebrow{color:var(--muted);opacity:1}\n[data-radar-root="v3"] .run-title h1{font-size:22px;line-height:1;margin:0;letter-spacing:-.035em}\n[data-radar-root="v3"] .run-tags{display:flex;justify-content:flex-end;gap:7px;flex-wrap:wrap}\n[data-radar-root="v3"] .run-tag{display:inline-flex;align-items:center;min-height:28px;padding:5px 9px;border-radius:999px;border:1px solid var(--line);background:var(--soft);font-size:10px;font-weight:780;color:#4c556f;white-space:nowrap}\n[data-radar-root="v3"] .run-tag.ok{background:#f2f8f5;border-color:#d6eadf;color:#176848}\n[data-radar-root="v3"] .context-grid{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="v3"] .context-card{position:relative;background:#fff;border:1px solid var(--line);border-radius:18px;padding:16px 17px;min-height:138px;overflow:hidden;box-shadow:0 5px 18px rgba(27,36,74,.035)}\n[data-radar-root="v3"] .context-card:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:#dfe3ff}\n[data-radar-root="v3"] .context-card:hover{border-color:#d4d9e6;box-shadow:0 10px 28px rgba(27,36,74,.065)}\n[data-radar-root="v3"] .identity-card{grid-column:span 4;min-height:174px}\n[data-radar-root="v3"] .identity-card:before{background:var(--bb-blue)}\n[data-radar-root="v3"] .cycle-card{grid-column:span 8;min-height:174px}\n[data-radar-root="v3"] .cycle-card:before{background:#8090ff}\n[data-radar-root="v3"] .income-card{grid-column:span 3}\n[data-radar-root="v3"] .income-card:before{background:#22a06b}\n[data-radar-root="v3"] .profile-card{grid-column:span 3}\n[data-radar-root="v3"] .profile-card:before{background:#8c7ae6}\n[data-radar-root="v3"] .coverage-card{grid-column:span 3}\n[data-radar-root="v3"] .coverage-card:before{background:#19a47b}\n[data-radar-root="v3"] .window-card{grid-column:span 3}\n[data-radar-root="v3"] .window-card:before{background:#5a84d6}\n[data-radar-root="v3"] .context-top{display:flex;align-items:center;justify-content:space-between;gap:10px}\n[data-radar-root="v3"] .context-kicker{font-size:11px;font-weight:880;letter-spacing:.08em;text-transform:uppercase;color:var(--muted)}\n[data-radar-root="v3"] .context-dot{width:8px;height:8px;border-radius:50%;background:var(--bb-yellow);box-shadow:0 0 0 4px #fffbd3}\n[data-radar-root="v3"] .context-badge{font-size:11px;font-weight:820;padding:4px 7px;border-radius:999px;background:#f0f2ff;color:var(--bb-blue-deep);white-space:nowrap}\n[data-radar-root="v3"] .context-badge.subtle{background:#f5f6f9;color:var(--muted)}\n[data-radar-root="v3"] .context-badge.good{background:var(--good-bg);color:var(--good)}\n[data-radar-root="v3"] .context-main{font-size:23px;font-weight:900;letter-spacing:-.035em;margin:12px 0 12px;line-height:1.05}\n[data-radar-root="v3"] .context-main small{font-size:11px;font-weight:720;letter-spacing:0;color:var(--muted);margin-left:3px}\n[data-radar-root="v3"] .date-range span{color:#a2a9ba;font-weight:500;margin:0 4px}\n[data-radar-root="v3"] .context-labels{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}\n[data-radar-root="v3"] .micro-label{display:inline-flex;align-items:center;gap:5px;padding:6px 8px;border:1px solid #edf0f5;border-radius:8px;background:#fafbfe;font-size:11px;color:#596276;font-weight:720}\n[data-radar-root="v3"] .micro-label b{color:var(--ink);font-weight:840}\n[data-radar-root="v3"] .context-data{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-top:12px}\n[data-radar-root="v3"] .context-data.single{grid-template-columns:1fr}\n[data-radar-root="v3"] .data-cell{padding-top:9px;border-top:1px solid #eef0f4}\n[data-radar-root="v3"] .data-cell span{display:block;font-size:10px;text-transform:uppercase;letter-spacing:.06em;color:#98a2b3;font-weight:820}\n[data-radar-root="v3"] .data-cell strong{display:block;margin-top:3px;font-size:11px;line-height:1.25;overflow-wrap:anywhere}\n[data-radar-root="v3"] .context-foot{margin-top:8px;color:#8a93a7;font-size:10px;white-space:normal}\n[data-radar-root="v3"] .section-head>div:first-child{min-width:0}\n[data-radar-root="v3"] .section-label{display:block;font-size:9px;font-weight:900;letter-spacing:.11em;text-transform:uppercase;color:#98a2b3;margin-bottom:2px}\n[data-radar-root="v3"] .context-card,[data-radar-root="v3"] .finance-card,[data-radar-root="v3"] .score-card-v6{padding:0}\n[data-radar-root="v3"] .context-card>summary,[data-radar-root="v3"] .finance-card>summary,[data-radar-root="v3"] .score-card-v6>summary{list-style:none;cursor:pointer;padding:15px 16px;display:flex;align-items:center;justify-content:space-between;gap:12px;min-height:76px}\n[data-radar-root="v3"] .context-card>summary::-webkit-details-marker,[data-radar-root="v3"] .finance-card>summary::-webkit-details-marker,[data-radar-root="v3"] .score-card-v6>summary::-webkit-details-marker{display:none}\n[data-radar-root="v3"] .context-card>summary:after,[data-radar-root="v3"] .finance-card>summary:after,[data-radar-root="v3"] .score-card-v6>summary:after{content:"+";display:grid;place-items:center;flex:0 0 25px;height:25px;border-radius:8px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:900}\n[data-radar-root="v3"] .context-card[open]>summary:after,[data-radar-root="v3"] .finance-card[open]>summary:after,[data-radar-root="v3"] .score-card-v6[open]>summary:after{content:"−"}\n[data-radar-root="v3"] .card-summary-main{min-width:0;flex:1}\n[data-radar-root="v3"] .card-summary-row{display:flex;align-items:center;gap:8px;flex-wrap:wrap}\n[data-radar-root="v3"] .card-summary-value{font-size:20px;font-weight:900;letter-spacing:-.035em;line-height:1.1;margin-top:5px;overflow-wrap:anywhere}\n[data-radar-root="v3"] .card-summary-value.good{color:var(--good)}\n[data-radar-root="v3"] .card-body{border-top:1px solid #edf0f4;padding:12px 16px 15px}\n[data-radar-root="v3"] .context-card .context-labels,[data-radar-root="v3"] .context-card .context-data,[data-radar-root="v3"] .context-card .context-foot{margin-top:0}\n[data-radar-root="v3"] .context-card .context-data{margin-top:10px}\n[data-radar-root="v3"] .context-card .context-foot{padding-top:9px}\n[data-radar-root="v3"] .finance-card .card-labels{margin-top:0}\n[data-radar-root="v3"] .finance-card .meter{margin-top:4px}\n[data-radar-root="v3"] .score-card-v6 .score-data{margin-top:0}\n[data-radar-root="v3"] .score-card-v6 .score-parts{margin-top:10px}\n[data-radar-root="v3"] .score-card-v6.featured>summary{background:linear-gradient(135deg,#fffef0,#fff)}\n[data-radar-root="v3"] .finance-grid{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="v3"] .finance-card,[data-radar-root="v3"] .score-card-v6{position:relative;background:#fff;border:1px solid var(--line);border-radius:18px;padding:16px 17px;overflow:hidden;box-shadow:0 5px 18px rgba(27,36,74,.035);transition:.18s ease}\n[data-radar-root="v3"] .finance-card:before,[data-radar-root="v3"] .score-card-v6:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:#dfe3ff}\n[data-radar-root="v3"] .finance-card:hover,[data-radar-root="v3"] .score-card-v6:hover{border-color:#d4d9e6;box-shadow:0 10px 28px rgba(27,36,74,.065);transform:translateY(-1px)}\n[data-radar-root="v3"] .finance-in{grid-column:span 3}\n[data-radar-root="v3"] .finance-out{grid-column:span 3}\n[data-radar-root="v3"] .finance-balance{grid-column:span 4;background:#f8fcfa}\n[data-radar-root="v3"] .finance-ratio{grid-column:span 2}\n[data-radar-root="v3"] .finance-balance:before{background:#8fd8bd}\n[data-radar-root="v3"] .finance-ratio:before{background:#b9c1ff}\n[data-radar-root="v3"] .card-top{display:flex;align-items:center;justify-content:space-between;gap:10px}\n[data-radar-root="v3"] .card-kicker{font-size:9px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted);font-weight:900}\n[data-radar-root="v3"] .card-badge{padding:4px 7px;border-radius:999px;background:#f1f3ff;color:var(--bb-blue-deep);font-size:8px;font-weight:850;white-space:nowrap}\n[data-radar-root="v3"] .card-badge.good{background:var(--good-bg);color:var(--good)}\n[data-radar-root="v3"] .card-value{font-size:clamp(24px,3vw,34px);font-weight:900;letter-spacing:-.04em;line-height:1.05;margin:14px 0 12px}\n[data-radar-root="v3"] .finance-balance .card-value{color:var(--good)}\n[data-radar-root="v3"] .card-labels{display:flex;gap:6px;flex-wrap:wrap}\n[data-radar-root="v3"] .card-label{display:inline-flex;gap:4px;align-items:center;background:#f7f8fb;border:1px solid #eceff4;border-radius:8px;padding:5px 7px;font-size:9px;color:var(--muted)}\n[data-radar-root="v3"] .card-label b{color:var(--ink)}\n[data-radar-root="v3"] .meter{height:7px;background:#edf0f5;border-radius:999px;overflow:hidden;margin:12px 0 7px}\n[data-radar-root="v3"] .meter span{display:block;height:100%;background:var(--bb-blue);width:72.39%}\n[data-radar-root="v3"] .score-grid-v6{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="v3"] .score-card-v6{grid-column:span 3;min-height:190px}\n[data-radar-root="v3"] .score-card-v6.featured{grid-column:span 6;background:linear-gradient(135deg,#fffef0,#fff);border-color:#e8e584}\n[data-radar-root="v3"] .score-card-v6.featured:before{background:var(--bb-yellow);width:5px}\n[data-radar-root="v3"] .score-card-v6.wide{grid-column:span 3}\n[data-radar-root="v3"] .score-title{font-size:13px;font-weight:850;line-height:1.2;max-width:220px}\n[data-radar-root="v3"] .score-final{font-size:36px;font-weight:950;letter-spacing:-.05em;line-height:1}\n[data-radar-root="v3"] .score-final small{font-size:9px;color:var(--muted);font-weight:850;letter-spacing:.04em;text-transform:uppercase;display:block;margin-bottom:3px}\n[data-radar-root="v3"] .score-data{display:grid;grid-template-columns:repeat(3,1fr);gap:6px;margin-top:14px}\n[data-radar-root="v3"] .score-data div{border-top:1px solid #edf0f4;padding-top:8px}\n[data-radar-root="v3"] .score-data span{display:block;font-size:8px;text-transform:uppercase;letter-spacing:.05em;color:#98a2b3;font-weight:850}\n[data-radar-root="v3"] .score-data b{display:block;font-size:11px;margin-top:2px}\n[data-radar-root="v3"] .score-parts{display:flex;gap:6px;flex-wrap:wrap;margin-top:12px}\n[data-radar-root="v3"] .score-part{background:#f7f8fb;border:1px solid #eceff4;border-radius:8px;padding:5px 7px;font-size:9px;color:var(--muted)}\n[data-radar-root="v3"] .score-part b{color:var(--ink)}\n[data-radar-root="v3"] .score-rule{margin-top:10px;font-size:9px;color:#725000;background:#fff8df;border:1px solid #f1df9b;border-radius:9px;padding:7px 8px}\n@media(max-width:1050px){[data-radar-root="v3"] .finance-in,[data-radar-root="v3"] .finance-out{grid-column:span 3}\n[data-radar-root="v3"] .finance-balance,[data-radar-root="v3"] .finance-ratio{grid-column:span 6}\n[data-radar-root="v3"] .score-card-v6,[data-radar-root="v3"] .score-card-v6.featured,[data-radar-root="v3"] .score-card-v6.wide{grid-column:span 6}\n[data-radar-root="v3"] .context-grid{grid-template-columns:repeat(6,minmax(0,1fr))}\n[data-radar-root="v3"] .identity-card,[data-radar-root="v3"] .cycle-card{grid-column:span 3}\n[data-radar-root="v3"] .income-card,[data-radar-root="v3"] .profile-card,[data-radar-root="v3"] .coverage-card,[data-radar-root="v3"] .window-card{grid-column:span 3}\n[data-radar-root="v3"] .run-header{align-items:flex-start}\n[data-radar-root="v3"] .run-tags{justify-content:flex-start}\n[data-radar-root="v3"] .money-grid{grid-template-columns:1fr 30px 1fr}\n[data-radar-root="v3"] .money-grid .operator:nth-of-type(2){display:none}\n[data-radar-root="v3"] .money-grid .balance-card{grid-column:1/-1}\n[data-radar-root="v3"] .score-layout{grid-template-columns:1fr}\n[data-radar-root="v3"] .winner-card{position:static}\n[data-radar-root="v3"] .hero-grid{grid-template-columns:1fr}}\n@media(max-width:760px){[data-radar-root="v3"] .finance-in,[data-radar-root="v3"] .finance-out,[data-radar-root="v3"] .finance-balance,[data-radar-root="v3"] .finance-ratio,[data-radar-root="v3"] .score-card-v6,[data-radar-root="v3"] .score-card-v6.featured,[data-radar-root="v3"] .score-card-v6.wide{grid-column:1/-1}\n[data-radar-root="v3"] .score-data{grid-template-columns:repeat(3,1fr)}\n[data-radar-root="v3"] .shell{padding:0 13px 60px}\n[data-radar-root="v3"] .topbar-inner{padding:10px 13px}\n[data-radar-root="v3"] .nav{display:none}\n[data-radar-root="v3"] .brand span:last-child{display:none}\n[data-radar-root="v3"] .run-header{margin-top:14px;display:block}\n[data-radar-root="v3"] .run-tags{margin-top:12px}\n[data-radar-root="v3"] .context-grid{grid-template-columns:repeat(2,minmax(0,1fr))}\n[data-radar-root="v3"] .identity-card,[data-radar-root="v3"] .cycle-card,[data-radar-root="v3"] .income-card,[data-radar-root="v3"] .profile-card,[data-radar-root="v3"] .coverage-card,[data-radar-root="v3"] .window-card{grid-column:span 1}\n[data-radar-root="v3"] .section-head{align-items:start;flex-direction:column}\n[data-radar-root="v3"] .pair{grid-template-columns:1fr}\n[data-radar-root="v3"] .link{transform:rotate(90deg)}\n[data-radar-root="v3"] .metrics{display:none}}\n@media(max-width:480px){[data-radar-root="v3"] .context-grid{grid-template-columns:1fr}\n[data-radar-root="v3"] .identity-card,[data-radar-root="v3"] .cycle-card,[data-radar-root="v3"] .income-card,[data-radar-root="v3"] .profile-card,[data-radar-root="v3"] .coverage-card,[data-radar-root="v3"] .window-card{grid-column:span 1}\n[data-radar-root="v3"] .run-title{display:block}\n[data-radar-root="v3"] .run-title .eyebrow{margin-bottom:5px}\n[data-radar-root="v3"] .run-tags{gap:5px}\n[data-radar-root="v3"] .run-tag{font-size:9px;padding:5px 8px}\n[data-radar-root="v3"] .money-grid{grid-template-columns:1fr}\n[data-radar-root="v3"] .operator{transform:rotate(90deg);text-align:center}\n[data-radar-root="v3"] .mini-grid{grid-template-columns:1fr}\n[data-radar-root="v3"] .context-card{min-height:auto}\n[data-radar-root="v3"] .context-main{font-size:22px}\n[data-radar-root="v3"] .score-memory{display:grid;grid-template-columns:1fr 1fr}}\n@media(prefers-reduced-motion:reduce){[data-radar-root="v3"]{scroll-behavior:auto}\n[data-radar-root="v3"],[data-radar-root="v3"] *{transition:none!important;animation:none!important}}\n[data-radar-root="v3"] .context-card,[data-radar-root="v3"] .finance-card,[data-radar-root="v3"] .score-card-v6{padding:16px 17px}\n[data-radar-root="v3"] .card-static-head{display:block;min-height:auto;padding:0}\n[data-radar-root="v3"] .card-static-head .card-summary-main{width:100%}\n[data-radar-root="v3"] .context-card .card-body,[data-radar-root="v3"] .finance-card .card-body,[data-radar-root="v3"] .score-card-v6 .card-body{padding:11px 0 0;margin-top:11px;border-top:1px solid #edf0f4}\n[data-radar-root="v3"] .context-card .context-labels{margin-top:0}\n[data-radar-root="v3"] .context-card .context-data{margin-top:10px}\n[data-radar-root="v3"] .context-card .context-foot{padding-top:0;margin-top:8px}\n[data-radar-root="v3"] .finance-card .card-labels{margin-top:0}\n[data-radar-root="v3"] .finance-card .meter{margin-top:4px}\n[data-radar-root="v3"] .score-card-v6 .score-data{margin-top:0}\n[data-radar-root="v3"] .score-card-v6 .score-parts{margin-top:10px}\n[data-radar-root="v3"] .score-card-v6.featured .card-static-head{background:transparent}\n[data-radar-root="v3"] .score-table-shell{background:#fff;border:1px solid var(--line);border-radius:20px;overflow:hidden;box-shadow:0 8px 26px rgba(27,36,74,.045)}\n[data-radar-root="v3"] .score-table-shell>summary{list-style:none;cursor:pointer;display:flex;align-items:center;justify-content:space-between;gap:16px;padding:16px 18px;background:#fff;font-weight:850}\n[data-radar-root="v3"] .score-table-shell>summary::-webkit-details-marker{display:none}\n[data-radar-root="v3"] .score-table-shell>summary:after{content:"+";display:grid;place-items:center;flex:0 0 28px;height:28px;border-radius:9px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:950}\n[data-radar-root="v3"] .score-table-shell[open]>summary:after{content:"−"}\n[data-radar-root="v3"] .score-summary-main{display:flex;gap:9px;align-items:center;flex-wrap:wrap}\n[data-radar-root="v3"] .score-summary-main strong{font-size:13px}\n[data-radar-root="v3"] .score-summary-meta{font-size:10px;color:var(--muted);font-weight:650}\n[data-radar-root="v3"] .score-winner-tag{display:inline-flex;align-items:center;gap:5px;background:#fffbd9;border:1px solid #ebe47f;color:#655f00;border-radius:999px;padding:5px 8px;font-size:9px;font-weight:850}\n[data-radar-root="v3"] .score-table-body{border-top:1px solid var(--line);padding:14px}\n[data-radar-root="v3"] .score-table-wrap{overflow-x:auto;border:1px solid var(--line);border-radius:14px;background:#fff}\n[data-radar-root="v3"] .score-table{min-width:760px}\n[data-radar-root="v3"] .score-table th,[data-radar-root="v3"] .score-table td{padding:11px 12px}\n[data-radar-root="v3"] .score-table thead th{background:#f8f9fc;top:0;z-index:1}\n[data-radar-root="v3"] .score-table tbody tr:last-child td{border-bottom:0}\n[data-radar-root="v3"] .score-table .theme-cell{min-width:220px;font-weight:800}\n[data-radar-root="v3"] .score-table .theme-cell small{display:block;margin-top:3px;color:var(--muted);font-size:9px;font-weight:650}\n[data-radar-root="v3"] .score-table .num{text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums}\n[data-radar-root="v3"] .score-table .final-cell{text-align:center;font-size:18px;font-weight:950;color:var(--bb-blue-deep)}\n[data-radar-root="v3"] .score-table .winner-row td{background:#fffef0}\n[data-radar-root="v3"] .score-table .winner-row td:first-child{box-shadow:inset 4px 0 0 var(--bb-yellow)}\n[data-radar-root="v3"] .score-inline-badge{display:inline-flex;margin-left:6px;background:#fff8c7;color:#645d00;border:1px solid #ebe47f;border-radius:999px;padding:3px 6px;font-size:8px;font-weight:900;vertical-align:middle;white-space:nowrap}\n[data-radar-root="v3"] .score-subhead{text-align:right!important}\n[data-radar-root="v3"] .score-subhead:first-child{text-align:left!important}\n@media(max-width:760px){[data-radar-root="v3"] .score-table-body{padding:10px}\n[data-radar-root="v3"] .score-table-shell>summary{padding:14px}\n[data-radar-root="v3"] .score-summary-meta{width:100%}}\n[data-radar-root="v3"] .topbar-inner{justify-content:space-between}\n[data-radar-root="v3"] .score-table-simple{min-width:0}\n[data-radar-root="v3"] .score-table-simple th:last-child,[data-radar-root="v3"] .score-table-simple td:last-child{width:140px;text-align:center}\n[data-radar-root="v3"] .score-table-simple .theme-cell{min-width:0}\n[data-radar-root="v3"] .score-table-simple .final-cell{font-size:20px}\n[data-radar-root="v3"] .score-table-static{padding:14px;overflow:hidden}\n[data-radar-root="v3"] .score-table-static .score-table-wrap{height:100%;margin:0}\n[data-radar-root="v3"] .score-table-complete{min-width:980px;width:100%}\n[data-radar-root="v3"] .score-table-complete th,[data-radar-root="v3"] .score-table-complete td{padding:12px 11px}\n[data-radar-root="v3"] .score-table-complete th{font-size:9px}\n[data-radar-root="v3"] .score-table-complete .theme-cell{min-width:230px}\n[data-radar-root="v3"] .score-table-complete .final-cell{font-size:18px;font-weight:900;text-align:center}\n[data-radar-root="v3"] .score-table-complete .winner-row td{background:#fffdf0}\n[data-radar-root="v3"] .score-table-complete .winner-row td:first-child{font-weight:900}\n[data-radar-root="v3"] .score-final-badge{display:inline-flex;align-items:center;justify-content:center;width:38px;height:38px;border-radius:999px;background:var(--bb-yellow);color:#173b67;font-weight:950;line-height:1;box-shadow:inset 0 0 0 1px #e7de58}\n@media(max-width:760px){[data-radar-root="v3"] .score-table-static{padding:10px}\n[data-radar-root="v3"] .score-table-complete{min-width:920px}}\n[data-radar-root="v3"] .top-actions{display:flex;align-items:center;gap:8px;white-space:nowrap}\n[data-radar-root="v3"] .export-btn{border:1px solid var(--bb-blue);background:var(--bb-blue);color:#fff;border-radius:999px;padding:8px 12px;font-weight:800;font-size:12px;cursor:pointer;white-space:nowrap}\n[data-radar-root="v3"] .export-btn:hover{background:var(--bb-blue-deep);border-color:var(--bb-blue-deep)}\n[data-radar-root="v3"] .export-btn:focus-visible,[data-radar-root="v3"] .privacy:focus-visible{outline:3px solid rgba(70,94,255,.22);outline-offset:2px}\n[data-radar-root="v3"] .export-toast{position:fixed;right:22px;bottom:22px;z-index:9999;max-width:360px;background:#171b2f;color:#fff;border-radius:12px;padding:11px 14px;box-shadow:0 12px 36px rgba(0,0,0,.22);font-size:11px;line-height:1.4;opacity:0;transform:translateY(8px);pointer-events:none;transition:.18s ease}\n[data-radar-root="v3"] .export-toast.show{opacity:1;transform:translateY(0)}\n@media(max-width:520px){[data-radar-root="v3"] .top-actions{gap:5px}\n[data-radar-root="v3"] .privacy,[data-radar-root="v3"] .export-btn{font-size:10px;padding:7px 9px}\n[data-radar-root="v3"] .export-toast{left:12px;right:12px;bottom:12px;max-width:none}}'
dashboard_root_id = f'radar-financeiro-v3-{CD_CLI}-{DATA_EXECUCAO.strftime("%Y%m%d")}'

html_dashboard_v3 = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Radar Financeiro — Experiência Final</title>
<style>{CSS_APROVADO_V3}{CSS_COMPLEMENTO_V5}</style>
<style>
[data-radar-root="v3"] [data-role="privacy-button"]{{display:inline-flex;align-items:center}}
[data-radar-root="v3"] [data-role="export-button"]{{display:inline-flex;align-items:center}}
[data-radar-root="v3"] [data-role="privacy-toggle"],
[data-radar-root="v3"] [data-role="export-confirm"]{{position:absolute;opacity:0;width:1px;height:1px;pointer-events:none}}
[data-radar-root="v3"] [data-role="privacy-hide"]{{display:none}}
[data-radar-root="v3"]:has([data-role="privacy-toggle"]:checked) [data-role="privacy-show"]{{display:none}}
[data-radar-root="v3"]:has([data-role="privacy-toggle"]:checked) [data-role="privacy-hide"]{{display:inline}}
[data-radar-root="v3"]:has([data-role="privacy-toggle"]:checked) [data-sensitive="true"]{{filter:none!important;user-select:auto!important}}
[data-radar-root="v3"]:has([data-role="privacy-toggle"]:checked) .tech tbody tr[data-private="true"] td:last-child{{filter:none!important;user-select:auto!important}}
[data-radar-root="v3"]:has([data-role="export-confirm"]:checked) [data-role="export-toast"]{{opacity:1;transform:translateY(0)}}
</style>
</head>
<body>
<section class="privacy-on" data-radar-root="v3" id="{dashboard_root_id}" tabindex="-1">
<a class="skip" href="#{dashboard_root_id}">Ir para o conteúdo</a>
<header class="topbar"><div class="topbar-inner">
<div class="brand"><span aria-hidden="true" class="brand-mark"></span><span>Radar Financeiro</span></div>
<div class="top-actions"><label class="privacy" data-role="privacy-button">
<input aria-label="Mostrar ou ocultar dados sensíveis" data-role="privacy-toggle" type="checkbox"/>
<span data-role="privacy-show">Mostrar dados</span><span data-role="privacy-hide">Ocultar dados</span>
</label>
<label class="export-btn" data-role="export-button" title="O HTML é salvo automaticamente no workdir">
<input aria-label="Confirmar localização do HTML exportado" data-role="export-confirm" type="checkbox"/>
<span>Exportar HTML</span>
</label></div>
</div></header>
<main class="shell" data-role="content">
<section aria-label="Radar Financeiro" class="run-header"><div class="run-title"><div class="eyebrow">Análise do ciclo</div><h1>Radar Financeiro</h1></div>
<div aria-label="Condições da execução" class="run-tags">
{tag_execucao('CPF único', res_dict.get('FL_CPF_UNICO'))}
{tag_execucao('Conta única', res_dict.get('FL_CONTA_ELEGIVEL_UNICA'))}
{tag_execucao('Somente BRL', res_dict.get('FL_SOMENTE_BRL'))}
{tag_execucao('Agro', res_dict.get('FL_TEM_MOV_AGRO'))}
<span data-v5-slot-html="tag_base_html">{payload_padrao_v5['tag_base_html']}</span>
<span data-v5-slot-html="tag_pontuacao_html">{payload_padrao_v5['tag_pontuacao_html']}</span>
</div></section>

<section class="section" data-section="contexto"><div class="section-head"><div><h2>Contexto</h2></div></div><div class="context-grid">
<article class="context-card identity-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Cliente</span><span aria-hidden="true" class="context-dot"></span></div><div class="card-summary-value" data-sensitive="true">{esc(res_dict.get('CD_CLI'))}</div></div></div>
<div class="card-body"><div class="context-labels"><span class="micro-label"><code>NR_AG_TITR</code> <b data-sensitive="true">{esc(res_cta['NR_AG_TITR'])}</b></span><span class="micro-label"><code>CD_CT_TITR</code> <b data-sensitive="true">{esc(res_cta['CD_CT_TITR'])}</b></span></div>
<div class="context-labels"><span class="micro-label"><code>CD_UOR_CC</code> <b data-sensitive="true">{esc(cta_norm_row['CD_UOR_CC_NORM'])}</b></span><span class="micro-label"><code>NR_CC</code> <b data-sensitive="true">{esc(cta_norm_row['NR_CC_NORM'])}</b></span></div></div></article>
<article class="context-card cycle-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Ciclo</span><span class="context-badge">Dia {fmt_inteiro(res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK'))}</span></div><div class="card-summary-value date-range">{fmt_data(res_dict.get('DT_REF_INI'), True)} <span>→</span> {fmt_data(res_dict.get('DT_REF_FIM'), True)}</div></div></div>
<div class="card-body"><div class="context-labels"><span class="micro-label">Início <b>{fmt_data(res_dict.get('DT_REF_INI'))}</b></span><span class="micro-label">Fim <b>{fmt_data(res_dict.get('DT_REF_FIM'))}</b></span><span class="micro-label">Fallback <b>{fallback_usado}</b></span></div><div class="context-data single"><div class="data-cell"><span>Referência</span><strong>{fmt_data(res_dict.get('TS_DD_INC_MM_CLC_BLC_REF'))}</strong></div></div></div></article>
<article class="context-card income-card" data-v5-component="base-financeira"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker" data-v5-slot="base_kicker">{payload_padrao_v5['base_kicker']}</span><span class="context-badge good">BRL</span></div><div aria-live="polite" class="card-summary-value" data-v5-slot="valor_base">{payload_padrao_v5['valor_base']}</div></div></div><div aria-label="Selecionar base financeira" class="base-switch" role="group"><button aria-pressed="true" class="base-switch-btn active" data-v5-mode="RENDA_PRESUMIDA" type="button">Renda Presumida</button><button aria-pressed="false" class="base-switch-btn" data-v5-mode="ENTRADAS_REALIZADAS" type="button">Entradas Realizadas</button></div><div class="card-body"><div class="context-data single"><div class="data-cell"><span data-v5-slot="referencia_rotulo">{payload_padrao_v5['referencia_rotulo']}</span><strong data-v5-slot="referencia_valor">{payload_padrao_v5['referencia_valor']}</strong></div></div><div class="context-foot"><span class="scenario-active-tag" data-v5-slot="base_badge">{payload_padrao_v5['base_badge']}</span></div></div></article>
<article class="context-card profile-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Perfil</span><span class="context-badge subtle">Ref. {fmt_data(res_dict.get('DT_REF_PRFL'), True)}</span></div><div class="card-summary-value">{esc(res_dict.get('NM_MAC_PRFL_CLI'))}</div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Microperfil <b>{esc(res_dict.get('NM_MIC_PRFL_CLI'))}</b></span></div><div class="context-foot">Referência: {fmt_data(res_dict.get('DT_REF_PRFL'))}</div></div></article>
<article class="context-card coverage-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Base analisada</span><span class="context-badge good">{('Somente BRL' if res_dict.get('FL_SOMENTE_BRL') == 'S' else 'Moedas verificadas')}</span></div><div class="card-summary-value">{fmt_inteiro(res_dict.get('QT_TRANS_TOTAL'))} <small>transações</small></div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Entradas <b>{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))}</b></span><span class="micro-label">Saídas <b>{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))}</b></span></div><div class="context-foot">Moeda utilizada nos cálculos: BRL</div></div></article>
<article class="context-card window-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Janela de contexto</span><span class="context-badge">±{DIAS_CONTEXTO_RECONCILIACAO} dias</span></div><div class="card-summary-value date-range">{fmt_data(dt_contexto_ini, True)} <span>→</span> {fmt_data(dt_contexto_fim, True)}</div></div></div><div class="card-body"><div class="context-data"><div class="data-cell"><span>Ciclo oficial</span><strong>{fmt_data(dt_ini_j, True)} → {fmt_data(dt_fim_j, True)}</strong></div><div class="data-cell"><span>Contexto</span><strong>{fmt_data(dt_contexto_ini, True)} → {fmt_data(dt_contexto_fim, True)}</strong></div></div></div></article>
</div></section>

<section class="section" data-section="resultado"><div class="section-head"><div><span class="section-label">Resultado</span><h2>Resumo financeiro</h2></div></div><div class="finance-grid">
<article class="finance-card finance-in"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker" data-v5-slot="entrada_rotulo">{payload_padrao_v5['entrada_rotulo']}</span><span class="card-badge">{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))} transações</span></div><div class="card-summary-value" data-v5-slot="entrada_valor">{payload_padrao_v5['entrada_valor']}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Base aplicada a toda a cadeia deste cenário</span></div></div></article>
<article class="finance-card finance-out"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saídas</span><span class="card-badge">{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))} transações</span></div><div class="card-summary-value">{fmt_moeda(res_dict.get('VL_TRANS_SAI'))}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Consideradas <b>{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))}</b></span></div></div></article>
<article class="finance-card finance-balance"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saldo do ciclo</span><span class="card-badge good" data-v5-slot="saldo_status">{payload_padrao_v5['saldo_status']}</span></div><div class="card-summary-value good" data-v5-slot="saldo_valor">{payload_padrao_v5['saldo_valor']}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Situação <b>pré-calculada pelo motor</b></span></div></div></article>
<article class="finance-card finance-ratio"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saídas / Base</span></div><div class="card-summary-value" data-v5-slot="razao_valor">{payload_padrao_v5['razao_valor']}</div></div></div><div class="card-body"><div aria-label="Relação Saídas sobre Base Financeira" class="meter"><span data-v5-meter="true" style="width:{payload_padrao_v5['largura_medidor']}"></span></div><div class="card-labels"><span class="card-label">Relação do ciclo</span></div></div></article>
</div></section>

<section class="section" data-section="composicao"><div class="section-head"><div><span class="section-label">Explicabilidade</span><h2>Composição</h2></div><div class="controls"><input aria-label="Buscar na composição" class="search" data-role="composition-search" placeholder="Buscar classe ou categoria" type="search"/><button class="control-btn" data-action="expand-composition" type="button">Expandir tudo</button><button class="control-btn" data-action="collapse-composition" type="button">Recolher</button></div></div><div class="explorer" data-role="composition-explorer">{render_composicao()}</div></section>

<section class="section" data-section="pontuacao"><div class="section-head"><div><span class="section-label">Motor</span><h2>Pontuação</h2></div><span class="scenario-active-tag" data-v5-slot="base_badge">{payload_padrao_v5['base_badge']}</span></div><div aria-label="Tabela completa de pontuação" class="score-table-shell score-table-static"><div class="score-table-wrap"><table class="score-table score-table-complete"><thead><tr><th>Tema</th><th class="num">Valor</th><th class="num">% base</th><th class="num">Referência</th><th class="num">Conc.</th><th class="num">Orç.</th><th class="num">Perfil</th><th class="score-subhead">Final</th></tr></thead><tbody data-v5-slot-html="pontuacao_html">{payload_padrao_v5['pontuacao_html']}</tbody></table></div></div></section>

<div data-v5-slot-html="fechamento_html">{payload_padrao_v5['fechamento_html']}</div>

<section class="section" data-section="reconciliacao"><div class="section-head"><div><span class="section-label">Controle</span><h2>Reconciliação</h2></div><div class="controls"><button class="control-btn" data-action="expand-reconciliation" type="button">Expandir eventos</button><button class="control-btn" data-action="collapse-reconciliation" type="button">Recolher</button></div></div><div class="recon"><div class="funnel"><div class="step"><b>{qt_raw}</b>oficiais</div><div>→</div><div class="step minus"><b>−{2 * qt_pares_exatos_oficiais}</b>pares exatos removidos</div><div>→</div><div class="step minus"><b>−{qt_pares_borda}</b>borda removida</div><div>→</div><div class="step"><b>{qt_efetivo}</b>efetivas</div></div>{render_eventos_reconciliacao()}{render_contexto_externo()}</div></section>

<section class="section" data-section="auditoria"><div class="section-head"><div><span class="section-label">Auditoria</span><h2>Dados técnicos</h2></div><input aria-label="Buscar atributo técnico" class="search" data-role="technical-search" placeholder="Buscar campo técnico" type="search"/></div><div class="scenario-audit"><div><strong data-v5-slot="audit_titulo">{payload_padrao_v5['audit_titulo']}</strong><span data-v5-slot="audit_texto">{payload_padrao_v5['audit_texto']}</span></div><span class="scenario-active-tag" data-v5-slot="base_badge">{payload_padrao_v5['base_badge']}</span></div><details class="tech"><summary>80 atributos · view oficial V4 preservada</summary><div class="scroll"><table><thead><tr><th>#</th><th>Campo</th><th>Tipo</th><th>Valor oficial</th></tr></thead><tbody>{linhas_auditoria}</tbody></table></div></details></section>
</main>
<div aria-live="polite" class="export-toast" data-role="export-toast" role="status">HTML já salvo no workdir: radar_financeiro_{res_dict.get('CD_CLI')}_{DATA_EXECUCAO.strftime('%Y%m%d')}.html</div>
<script>
(function(){{
  const root=document.getElementById('{dashboard_root_id}');
  if(!root||root.dataset.radarBound==='1')return;
  root.dataset.radarBound='1';
  const one=selector=>root.querySelector(selector);
  const all=selector=>root.querySelectorAll(selector);
  function bind(element,eventName,handler){{
    if(!element||element.dataset.radarEventBound==='1')return;
    element.dataset.radarEventBound='1';
    element.addEventListener(eventName,handler);
  }}
  const scenarioPayloads={payload_cenarios_json_v5};
  Object.values(scenarioPayloads).forEach(payload=>Object.freeze(payload));
  Object.freeze(scenarioPayloads);
  function applyScenario(mode){{
    const payload=scenarioPayloads[mode];
    if(!payload)return;
    all('[data-v5-slot]').forEach(element=>{{
      const name=element.dataset.v5Slot;
      element.textContent=String(payload[name]??'—');
    }});
    all('[data-v5-slot-html]').forEach(element=>{{
      const name=element.dataset.v5SlotHtml;
      element.innerHTML=payload[name]||'';
    }});
    const meter=one('[data-v5-meter]');
    if(meter)meter.style.width=payload.largura_medidor;
    root.dataset.v5Scenario=mode;
    all('[data-v5-mode]').forEach(button=>{{
      const active=button.dataset.v5Mode===mode;
      button.classList.toggle('active',active);
      button.setAttribute('aria-pressed',String(active));
    }});
  }}
  all('[data-v5-mode]').forEach(button=>bind(button,'click',()=>applyScenario(button.dataset.v5Mode)));
  applyScenario('RENDA_PRESUMIDA');
  const privacyToggle=one('[data-role="privacy-toggle"]');
  bind(privacyToggle,'change',()=>{{
    const ocultar=!privacyToggle.checked;
    root.classList.toggle('privacy-on',ocultar);
  }});
  all('[data-section="auditoria"] .tech tbody tr').forEach(tr=>{{const cells=tr.querySelectorAll('td');if(cells.length>2&&['CD_CLI','CD_CPF'].includes(cells[1].textContent.trim()))tr.dataset.private='true';}});
  all('details').forEach(d=>d.open=false);
  function setOpen(container,open){{if(container)container.querySelectorAll('details').forEach(d=>d.open=open);}}
  const composition=one('[data-role="composition-explorer"]');
  const reconciliation=one('[data-section="reconciliacao"]');
  bind(one('[data-action="expand-composition"]'),'click',()=>setOpen(composition,true));
  bind(one('[data-action="collapse-composition"]'),'click',()=>setOpen(composition,false));
  bind(one('[data-action="expand-reconciliation"]'),'click',()=>setOpen(reconciliation,true));
  bind(one('[data-action="collapse-reconciliation"]'),'click',()=>setOpen(reconciliation,false));
  const compSearch=one('[data-role="composition-search"]');
  bind(compSearch,'input',()=>{{const q=compSearch.value.trim().toLowerCase();composition.querySelectorAll('details.category').forEach(d=>{{const hit=!q||d.textContent.toLowerCase().includes(q);d.style.display=hit?'':'none';if(q&&hit){{d.open=true;let p=d.parentElement.closest('details');while(p&&root.contains(p)){{p.open=true;p=p.parentElement.closest('details');}}}}}});}});
  const techSearch=one('[data-role="technical-search"]');
  const technical=one('[data-section="auditoria"] .tech');
  bind(techSearch,'input',()=>{{const q=techSearch.value.trim().toLowerCase();technical.querySelectorAll('tbody tr').forEach(tr=>tr.style.display=(!q||tr.textContent.toLowerCase().includes(q))?'':'none');if(q)technical.open=true;}});
  const exportToast=one('[data-role="export-toast"]');
  function showExportToast(message){{if(!exportToast)return;exportToast.textContent=message;exportToast.classList.add('show');window.clearTimeout(showExportToast._timer);showExportToast._timer=window.setTimeout(()=>exportToast.classList.remove('show'),5000);}}
  const filename='radar_financeiro_{res_dict.get('CD_CLI')}_{DATA_EXECUCAO.strftime('%Y%m%d')}.html';
  const exportConfirm=one('[data-role="export-confirm"]');
  bind(exportConfirm,'change',()=>{{
    showExportToast('HTML já salvo no workdir: '+filename);
    if(exportConfirm.checked)window.setTimeout(()=>{{exportConfirm.checked=false;}},5000);
  }});
}})();
</script>
</section>
</body></html>"""

html_bytes_v3 = html_dashboard_v3.encode('utf-8')
tamanho_v3 = len(html_bytes_v3)
sha256_v3 = hashlib.sha256(html_bytes_v3).hexdigest()
if tamanho_v3 > LIMITE_PAYLOAD_BYTES:
    raise RuntimeError(f'BLOQUEADO: payload HTML ({tamanho_v3} bytes) excede 2 MiB.')

metadados_dashboard_v3 = {
    'tamanho_bytes': tamanho_v3,
    'sha256': sha256_v3,
    'cd_cli': CD_CLI,
    'data_execucao': DATA_EXECUCAO.strftime('%Y%m%d'),
    'filename': f'radar_financeiro_{CD_CLI}_{DATA_EXECUCAO.strftime("%Y%m%d")}.html',
    'root_id': dashboard_root_id,
}

print(f'[V3_DASHBOARD] HTML final gerado: {tamanho_v3} bytes (SHA-256: {sha256_v3[:12]}...).')


### Tabela final oficial antes do dashboard


In [ ]:
%%spark

print('[V5_DASHBOARD] Tabela final oficial (80 atributos):')
spark.table(VIEW_RESULTADO).show(truncate=False, vertical=True)


### Renderização da experiência final no kernel local


In [ ]:
from IPython.display import HTML, Javascript, display
from pathlib import Path
import hashlib
import re

html_dashboard_local = spark.get_from_spark("html_dashboard_v3")
metadados_local = spark.get_from_spark("metadados_dashboard_v3")

if not isinstance(html_dashboard_local, str):
    raise TypeError(f"Payload inválido transferido do Spark: {type(html_dashboard_local)}")

bytes_recebidos = html_dashboard_local.encode('utf-8')
if len(bytes_recebidos) != metadados_local['tamanho_bytes']:
    raise RuntimeError('Tamanho do HTML divergiu durante o transporte Spark → kernel local.')

hash_local = hashlib.sha256(bytes_recebidos).hexdigest()
if hash_local != metadados_local['sha256']:
    raise RuntimeError('SHA-256 do HTML divergiu durante o transporte Spark → kernel local.')

workdir_v3 = Path.cwd().resolve()
destino_exportacao_v3 = (workdir_v3 / metadados_local['filename']).resolve()
if destino_exportacao_v3.parent != workdir_v3:
    raise RuntimeError('Destino de exportação fora do workdir.')

def salvar_html_v3_workdir():
    """Grava o payload validado no workdir; nunca aceita destino externo."""
    destino_exportacao_v3.write_bytes(bytes_recebidos)
    return destino_exportacao_v3

def verificar_exportacao_v3_workdir():
    """Comprova caminho, tamanho e hash da gravação feita pelo kernel local."""
    if not destino_exportacao_v3.is_file():
        raise RuntimeError(f'HTML exportado não encontrado em {destino_exportacao_v3}.')
    exportado = destino_exportacao_v3.read_bytes()
    if len(exportado) != metadados_local['tamanho_bytes']:
        raise RuntimeError('Tamanho do HTML exportado divergiu do payload validado.')
    if hashlib.sha256(exportado).hexdigest() != metadados_local['sha256']:
        raise RuntimeError('SHA-256 do HTML exportado divergiu do payload validado.')
    return {
        'path': str(destino_exportacao_v3),
        'bytes': len(exportado),
        'sha256': metadados_local['sha256'],
        'status': 'OK',
    }

# A gravação é responsabilidade do kernel local. Isso funciona no VSCode
# remoto e não depende da URL nem da Contents API do JupyterLab.
salvar_html_v3_workdir()
comprovante_exportacao_v6 = verificar_exportacao_v3_workdir()

# O renderer do VSCode pode manter <script> inerte em uma saída HTML. A mesma
# rotina embutida no arquivo standalone é, por isso, ativada também como MIME JS.
scripts_dashboard_v6 = re.findall(
    r'<script>(.*?)</script>',
    html_dashboard_local,
    flags=re.S | re.I,
)
if len(scripts_dashboard_v6) != 1:
    raise RuntimeError(
        f'Contrato de interatividade inválido: esperado 1 script; obtido={len(scripts_dashboard_v6)}.'
    )

print(f"[V3_DASHBOARD] Integridade verificada ({len(bytes_recebidos)} bytes).")
print(
    f"[V3_DASHBOARD] HTML salvo e verificado: {comprovante_exportacao_v6['path']} "
    f"({comprovante_exportacao_v6['bytes']} bytes; SHA-256 "
    f"{comprovante_exportacao_v6['sha256'][:12]}...)."
)
display(HTML(html_dashboard_local))
display(Javascript(scripts_dashboard_v6[0]))


---
# Bloco 6 — Testes Determinísticos em Memória em SQL Puro (Sob Demanda)
Validações contratuais determinísticas executadas em consultas Spark SQL sem dependências externas.


### Teste 6.0 — Período e Calendário Financeiro


In [ ]:
%%spark

print('[TESTE 6.0] Iniciando testes de período e calendário financeiro...')

for periodo_valido in (1, 6):
    if validar_periodo(periodo_valido) != periodo_valido:
        raise RuntimeError(f'[TESTE 6.0 FALHOU] Período válido rejeitado: {periodo_valido}.')

for periodo_invalido in (0, 7, True, False, 1.0, '1', None):
    try:
        validar_periodo(periodo_invalido)
    except (TypeError, ValueError):
        pass
    else:
        raise RuntimeError(f'[TESTE 6.0 FALHOU] Período inválido aceito: {periodo_invalido!r}.')

def calcular_janela_caso_teste(dia_ciclo, ts_referencia, quantidade_ciclos):
    dia_mes_referencia = min(dia_ciclo, calendar.monthrange(ts_referencia.year, ts_referencia.month)[1])
    candidato_aberto = datetime.datetime(ts_referencia.year, ts_referencia.month, dia_mes_referencia)
    if ts_referencia >= candidato_aberto:
        inicio_ciclo_aberto = candidato_aberto.date()
    else:
        mes_anterior_total = ts_referencia.year * 12 + ts_referencia.month - 2
        ano_anterior, mes_anterior_zero = divmod(mes_anterior_total, 12)
        mes_anterior = mes_anterior_zero + 1
        inicio_ciclo_aberto = datetime.date(
            ano_anterior,
            mes_anterior,
            min(dia_ciclo, calendar.monthrange(ano_anterior, mes_anterior)[1]),
        )
    return (
        calcular_inicio_periodo_fechado(inicio_ciclo_aberto, dia_ciclo, quantidade_ciclos),
        inicio_ciclo_aberto - timedelta(days=1),
    )

casos_calendario = [
    ('dia_10_periodo_1', 10, datetime.datetime(2026, 8, 13, 18, 42, 10), 1, datetime.date(2026, 7, 10), datetime.date(2026, 8, 9)),
    ('dia_10_periodo_2', 10, datetime.datetime(2026, 8, 13, 18, 42, 10), 2, datetime.date(2026, 6, 10), datetime.date(2026, 8, 9)),
    ('dia_28', 28, datetime.datetime(2026, 3, 28, 12, 0, 0), 1, datetime.date(2026, 2, 28), datetime.date(2026, 3, 27)),
    ('dia_29_fevereiro_comum', 29, datetime.datetime(2026, 3, 29, 12, 0, 0), 1, datetime.date(2026, 2, 28), datetime.date(2026, 3, 28)),
    ('dia_31_fevereiro_comum', 31, datetime.datetime(2026, 3, 31, 12, 0, 0), 1, datetime.date(2026, 2, 28), datetime.date(2026, 3, 30)),
    ('dia_31_fevereiro_bissexto', 31, datetime.datetime(2024, 3, 31, 12, 0, 0), 1, datetime.date(2024, 2, 29), datetime.date(2024, 3, 30)),
    ('dia_31_periodo_2', 31, datetime.datetime(2026, 3, 31, 12, 0, 0), 2, datetime.date(2026, 1, 31), datetime.date(2026, 3, 30)),
    ('transicao_ano', 30, datetime.datetime(2026, 1, 31, 12, 0, 0), 2, datetime.date(2025, 11, 30), datetime.date(2026, 1, 29)),
    ('ancora_dia_original', 31, datetime.datetime(2026, 3, 15, 12, 0, 0), 1, datetime.date(2026, 1, 31), datetime.date(2026, 2, 27)),
]

for nome_caso, dia_ciclo_teste, ts_teste, periodo_teste, inicio_esperado, fim_esperado in casos_calendario:
    inicio_obtido, fim_obtido = calcular_janela_caso_teste(dia_ciclo_teste, ts_teste, periodo_teste)
    if (inicio_obtido, fim_obtido) != (inicio_esperado, fim_esperado):
        raise RuntimeError(
            f'[TESTE 6.0 FALHOU] {nome_caso}: obtido={(inicio_obtido, fim_obtido)}; '
            f'esperado={(inicio_esperado, fim_esperado)}.'
        )

for inicio_aberto_teste, dia_ciclo_teste in [
    (datetime.date(2026, 8, 10), 10),
    (datetime.date(2026, 3, 31), 31),
    (datetime.date(2024, 3, 31), 31),
    (datetime.date(2026, 1, 30), 30),
]:
    total_legado = inicio_aberto_teste.year * 12 + inicio_aberto_teste.month - 2
    ano_legado, mes_legado_zero = divmod(total_legado, 12)
    mes_legado = mes_legado_zero + 1
    inicio_legado = datetime.date(
        ano_legado,
        mes_legado,
        min(dia_ciclo_teste, calendar.monthrange(ano_legado, mes_legado)[1]),
    )
    inicio_periodo_1 = calcular_inicio_periodo_fechado(inicio_aberto_teste, dia_ciclo_teste, 1)
    if inicio_periodo_1 != inicio_legado:
        raise RuntimeError(
            f'[TESTE 6.0 FALHOU] Regressão periodo=1: obtido={inicio_periodo_1}; esperado={inicio_legado}.'
        )

print(f'[TESTE 6.0] OK — período atual={periodo}; limites e calendário aprovados.')

### Teste 6.1 — Reconciliação em SQL (Pares Exatos e Bordas)


In [ ]:
%%spark

print('[TESTE 6.1 SQL] Iniciando teste determinístico de reconciliação em SQL...')

# 1. Validação de Par Exato em SQL (mesmo dia, mesmo valor, naturezas C e D na mesma janela)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_t1_exato_raw AS
SELECT 101 AS NR_TRAN_INST_PCT, 1 AS CD_CLI, DATE('2026-07-20') AS DT_TRAN, 'C' AS CD_NTZ_CTB_TRAN, 1 AS CD_CTGR_TRAN_OGNL, 'BRL' AS CD_TIP_MOE_CRR, CAST(500.00 AS DECIMAL(15,2)) AS VL_TRAN, 'S' AS IN_JANELA
UNION ALL
SELECT 102 AS NR_TRAN_INST_PCT, 1 AS CD_CLI, DATE('2026-07-20') AS DT_TRAN, 'D' AS CD_NTZ_CTB_TRAN, 6 AS CD_CTGR_TRAN_OGNL, 'BRL' AS CD_TIP_MOE_CRR, CAST(500.00 AS DECIMAL(15,2)) AS VL_TRAN, 'S' AS IN_JANELA
""")

pares_t1 = spark.sql("""
SELECT
    LEAST(
        SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END),
        SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END)
    ) AS QT_PARES
FROM vw_t1_exato_raw
GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR
""").first()['QT_PARES']

if pares_t1 != 1:
    raise RuntimeError('[TESTE 6.1 SQL FALHOU] Par exato não identificado em SQL.')

# 2. Validação dos Limites Temporais de Borda (5 dias permitido, 6 dias rejeitado)
dentro_lim = [(datetime.date(2026, 7, 20), 1)]
fora_lim5 = [(datetime.date(2026, 7, 15), 2)]
fora_lim6 = [(datetime.date(2026, 7, 14), 3)]

p5 = parear_listas_residuais_sql_impl([{'DT_TRAN': d, 'NR_TRAN_INST_PCT': i} for d, i in dentro_lim], [{'DT_TRAN': d, 'NR_TRAN_INST_PCT': i} for d, i in fora_lim5])
p6 = parear_listas_residuais_sql_impl([{'DT_TRAN': d, 'NR_TRAN_INST_PCT': i} for d, i in dentro_lim], [{'DT_TRAN': d, 'NR_TRAN_INST_PCT': i} for d, i in fora_lim6])

if len(p5) != 1 or len(p6) != 0:
    raise RuntimeError('[TESTE 6.1 SQL FALHOU] Limites de borda 5/6 dias violados.')

print('[TESTE 6.1 SQL] OK — Reconciliação em SQL aprovada.')

### Teste 6.2 — Classificação e Moedas em SQL


In [ ]:
%%spark

print('[TESTE 6.2 SQL] Iniciando teste de classificação e moedas em SQL...')

# Validação das regras de exclusão/inclusão orçamentária:
# - Categoria 111 (Cartão de Crédito) deve ser N/N (fora de tema e fora de orçamento)
# - Categorias 448977/448978 (Investimentos) devem ser S/N (entram no tema, mas fora do orçamento)
regras_check = spark.sql("""
SELECT
    MAX(CASE WHEN CD_CATEGORIA = 111 AND TIPO = 'D' THEN IN_PARTICIPA_CALCULO END) AS P_CALC_111,
    MAX(CASE WHEN CD_CATEGORIA = 111 AND TIPO = 'D' THEN IN_PARTICIPA_ORCAMENTO END) AS P_ORC_111,
    MAX(CASE WHEN CD_CATEGORIA = 448977 AND TIPO = 'D' THEN IN_PARTICIPA_CALCULO END) AS P_CALC_448977,
    MAX(CASE WHEN CD_CATEGORIA = 448977 AND TIPO = 'D' THEN IN_PARTICIPA_ORCAMENTO END) AS P_ORC_448977
FROM vw_categorias
""").first()

if regras_check['P_CALC_111'] != 'N' or regras_check['P_ORC_111'] != 'N':
    raise RuntimeError('[TESTE 6.2 SQL FALHOU] Categoria 111 deve ser N/N.')
if regras_check['P_CALC_448977'] != 'S' or regras_check['P_ORC_448977'] != 'N':
    raise RuntimeError('[TESTE 6.2 SQL FALHOU] Categoria 448977 deve ser S/N.')

print('[TESTE 6.2 SQL] OK — Classificação e regras de participação em SQL aprovadas.')

### Teste 6.3 — Faixas Orçamentárias em SQL


In [ ]:
%%spark

print('[TESTE 6.3 SQL] Iniciando teste de faixas orçamentárias em SQL...')

# Validação dos pontos de corte das 5 faixas orçamentárias contratuais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_t3_faixas AS
SELECT 1000.0 AS SAI, 1000.0 AS ENT, 0 AS FAIXA_ESP
UNION ALL
SELECT 1100.0 AS SAI, 1000.0 AS ENT, 1 AS FAIXA_ESP
UNION ALL
SELECT 1300.0 AS SAI, 1000.0 AS ENT, 2 AS FAIXA_ESP
UNION ALL
SELECT 850.0 AS SAI, 1000.0 AS ENT, 3 AS FAIXA_ESP
UNION ALL
SELECT 600.0 AS SAI, 1000.0 AS ENT, 4 AS FAIXA_ESP
""")

faixas_check = spark.sql("""
SELECT
    SAI, ENT, FAIXA_ESP,
    CASE 
        WHEN SAI / ENT >= 0.950000 AND SAI / ENT <= 1.050000 THEN 0
        WHEN SAI / ENT > 1.050000 AND SAI / ENT <= 1.250000 THEN 1
        WHEN SAI / ENT > 1.250000 THEN 2
        WHEN SAI / ENT >= 0.750000 AND SAI / ENT < 0.950000 THEN 3
        ELSE 4
    END AS FAIXA_OBT
FROM vw_t3_faixas
""").collect()

for r in faixas_check:
    if r['FAIXA_ESP'] != r['FAIXA_OBT']:
        raise RuntimeError(f'[TESTE 6.3 SQL FALHOU] Faixa incorreta para SAI={r["SAI"]}, ENT={r["ENT"]}: obtido={r["FAIXA_OBT"]}, esperado={r["FAIXA_ESP"]}.')

print('[TESTE 6.3 SQL] OK — Faixas orçamentárias em SQL aprovadas.')

### Teste 6.4 — Pontuações, Regra Especial IND e Empate em SQL


In [ ]:
%%spark

print('[TESTE 6.4 SQL] Iniciando teste de pontuações e regra especial IND em SQL...')

# 1. Validação da Regra Especial de IND: IND_FIM = CONC_IND (ignora ORC e PRFL)
ind_check = spark.sql("""
SELECT
    NR_PONT_CONC_IND,
    NR_PONT_ORC_IND,
    NR_PONT_PRFL_IND,
    NR_PONT_IND_FIM
FROM vw_pontuacoes
""").first()

if ind_check['NR_PONT_IND_FIM'] != ind_check['NR_PONT_CONC_IND']:
    raise RuntimeError('[TESTE 6.4 SQL FALHOU] Regra especial IND violada (IND_FIM != CONC_IND).')

# 2. Validação da Lógica de Empate: se mais de um tema tiver a pontuação máxima, retorna código 9
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_t4_empate AS
SELECT 3 AS P1, 3 AS P2, 1 AS P3, 0 AS P4, 2 AS P5
""")

empate_res = spark.sql("""
WITH m AS (
    SELECT GREATEST(P1, P2, P3, P4, P5) AS MAX_P FROM vw_t4_empate
),
c AS (
    SELECT 
        (CASE WHEN P1 = MAX_P THEN 1 ELSE 0 END +
         CASE WHEN P2 = MAX_P THEN 1 ELSE 0 END +
         CASE WHEN P3 = MAX_P THEN 1 ELSE 0 END +
         CASE WHEN P4 = MAX_P THEN 1 ELSE 0 END +
         CASE WHEN P5 = MAX_P THEN 1 ELSE 0 END) AS QT_VENC
    FROM vw_t4_empate CROSS JOIN m
)
SELECT CASE WHEN QT_VENC > 1 THEN 9 ELSE 1 END AS CD_VENC FROM c
""").first()['CD_VENC']

if empate_res != 9:
    raise RuntimeError('[TESTE 6.4 SQL FALHOU] Empate deve produzir CD_TEMA_VENCEDOR = 9.')

print('[TESTE 6.4 SQL] OK — Pontuações, regra especial IND e empates em SQL aprovados.')

### Teste 6.5 — Consistência das 80 Colunas e Integridade Visual em SQL


In [ ]:
%%spark

print('[TESTE 6.5 SQL] Iniciando validação de consistência dos 80 atributos em SQL...')

# Validação das relações de igualdade matemática entre totalizadores
totais_res = spark.sql("""
SELECT
    QT_TRANS_TOTAL,
    QT_TRANS_ENT,
    QT_TRANS_SAI,
    VL_TRANS_ENT,
    VL_ENT_TOTAL,
    VL_TRANS_SAI,
    VL_SAI_TOTAL
FROM vw_resultado_80_colunas
""").first()

if totais_res['QT_TRANS_TOTAL'] is not None:
    if totais_res['QT_TRANS_TOTAL'] != (totais_res['QT_TRANS_ENT'] + totais_res['QT_TRANS_SAI']):
        raise RuntimeError('[TESTE 6.5 SQL FALHOU] QT_TRANS_TOTAL != QT_TRANS_ENT + QT_TRANS_SAI.')

if totais_res['VL_ENT_TOTAL'] is not None:
    if totais_res['VL_TRANS_ENT'] != totais_res['VL_ENT_TOTAL']:
        raise RuntimeError('[TESTE 6.5 SQL FALHOU] VL_TRANS_ENT != VL_ENT_TOTAL.')
    if totais_res['VL_TRANS_SAI'] != totais_res['VL_SAI_TOTAL']:
        raise RuntimeError('[TESTE 6.5 SQL FALHOU] VL_TRANS_SAI != VL_SAI_TOTAL.')

if metadados_dashboard_v3['tamanho_bytes'] > LIMITE_PAYLOAD_BYTES:
    raise RuntimeError('[TESTE 6.5 SQL FALHOU] Tamanho do payload HTML acima de 2 MiB.')

print('[TESTE 6.5 SQL] OK — Consistência das 80 colunas e integridade visual aprovadas.')
print('===================================================================')
print('     [V3_DASHBOARD] TODOS OS TESTES EM SQL PASSARAM!          ')
print('===================================================================')

### Teste V4.1 — Renda por período e gate estrito V3 × V4


In [ ]:
%%spark

from decimal import Decimal


CAMPOS_DIFERENCA_PERMITIDA_V4 = {
    'VL_REN_PRES',
    'PC_SAI_IND', 'PC_SAI_ESS', 'PC_SAI_NAO_ESS', 'PC_SAI_FUT', 'PC_SAI_OBR',
    'NR_PONT_CONC_IND', 'NR_PONT_CONC_ESS', 'NR_PONT_CONC_NAO_ESS',
    'NR_PONT_CONC_FUT', 'NR_PONT_CONC_OBR',
    'NR_PONT_IND_FIM', 'NR_PONT_ESS_FIM', 'NR_PONT_NAO_ESS_FIM',
    'NR_PONT_FUT_FIM', 'NR_PONT_OBR_FIM',
    'NR_PONT_MAX', 'QT_TEMAS_PONT_MAX', 'CD_TEMA_VENCEDOR', 'TX_TEMA_VENCEDOR',
}

if len(CAMPOS_DIFERENCA_PERMITIDA_V4) != 20:
    raise RuntimeError('[TESTE V4.1 FALHOU] Allowlist V3 x V4 deve conter exatamente 20 atributos.')


def _como_dict(linha):
    return linha.asDict(recursive=True) if hasattr(linha, 'asDict') else dict(linha)


def validar_gate_resultados_v3_v4(resultado_v3, resultado_v4):
    """Bloqueia qualquer divergência V3 x V4 fora da dependência contratual da renda."""
    v3 = _como_dict(resultado_v3)
    v4 = _como_dict(resultado_v4)
    if list(v3) != list(v4):
        raise RuntimeError('[GATE V3 x V4 FALHOU] Schema ou ordem de atributos divergente.')
    if len(v3) != 80:
        raise RuntimeError(f'[GATE V3 x V4 FALHOU] Esperados 80 atributos; encontrados {len(v3)}.')
    divergentes = {campo for campo in v3 if v3[campo] != v4[campo]}
    proibidos = sorted(divergentes - CAMPOS_DIFERENCA_PERMITIDA_V4)
    if proibidos:
        raise RuntimeError(f'[GATE V3 x V4 FALHOU] Diferenças não permitidas: {proibidos}.')
    return divergentes


# Exercita o gate estrito usando o schema real das 80 colunas.
linha_v4_atual = df_res_80.first()
controle_gate = linha_v4_atual.asDict(recursive=True)
permitido_gate = dict(controle_gate)
for campo in CAMPOS_DIFERENCA_PERMITIDA_V4:
    permitido_gate[campo] = ('__V4_TESTE__', campo)
divergencias_gate = validar_gate_resultados_v3_v4(controle_gate, permitido_gate)
if divergencias_gate != CAMPOS_DIFERENCA_PERMITIDA_V4:
    raise RuntimeError('[TESTE V4.1 FALHOU] O gate não reconheceu exatamente os 20 campos permitidos.')

proibido_gate = dict(controle_gate)
proibido_gate['DT_REN_PRES_REF'] = '__ALTERACAO_PROIBIDA__'
try:
    validar_gate_resultados_v3_v4(controle_gate, proibido_gate)
except RuntimeError:
    pass
else:
    raise RuntimeError('[TESTE V4.1 FALHOU] Gate aceitou alteração em DT_REN_PRES_REF.')


# Casos mínimos da regra, sem acesso a fonte externa.
casos_regra = spark.sql("""
SELECT periodo_teste, renda_mensal,
       CAST(renda_mensal * periodo_teste AS DECIMAL(17,2)) AS renda_periodo
FROM VALUES
    (1, CAST(1000.00 AS DECIMAL(17,2))),
    (2, CAST(1000.00 AS DECIMAL(17,2))),
    (3, CAST(1000.00 AS DECIMAL(17,2))),
    (3, CAST(NULL AS DECIMAL(17,2)))
AS casos(periodo_teste, renda_mensal)
ORDER BY periodo_teste, renda_mensal NULLS LAST
""").collect()
obtido_regra = [(r['periodo_teste'], r['renda_mensal'], r['renda_periodo']) for r in casos_regra]
esperado_regra = [
    (1, Decimal('1000.00'), Decimal('1000.00')),
    (2, Decimal('1000.00'), Decimal('2000.00')),
    (3, Decimal('1000.00'), Decimal('3000.00')),
    (3, None, None),
]
if obtido_regra != esperado_regra:
    raise RuntimeError(f'[TESTE V4.1 FALHOU] Casos de renda x período divergentes: {obtido_regra}.')


# Confere seleção, data de referência e valor contra a Q3 já materializada em memória.
renda_esperada = spark.sql(f"""
WITH selecionada AS (
    SELECT
        CAST(DT_INCL_REN_AVLD AS DATE) AS DT_REN_PRES_REF,
        CAST(VL_REN * {periodo} AS DECIMAL(17,2)) AS VL_REN_PRES,
        ROW_NUMBER() OVER (PARTITION BY NR_CPF ORDER BY DT_INCL_REN_AVLD DESC) AS RN
    FROM vw_q3_renda_raw
), final AS (
    SELECT DT_REN_PRES_REF, VL_REN_PRES FROM selecionada WHERE RN = 1
)
SELECT DT_REN_PRES_REF, VL_REN_PRES FROM final
UNION ALL
SELECT CAST(NULL AS DATE), CAST(NULL AS DECIMAL(17,2))
WHERE NOT EXISTS (SELECT 1 FROM final)
""").first()
renda_obtida = spark.sql(
    'SELECT DT_REN_PRES_REF, VL_REN_PRES FROM vw_renda_derivada'
).first()
if renda_obtida != renda_esperada:
    raise RuntimeError(
        f'[TESTE V4.1 FALHOU] Derivação da renda divergente: '
        f'esperado={renda_esperada}, obtido={renda_obtida}.'
    )


# Percentuais devem usar diretamente a renda ajustada pelo período.
percentuais_esperados = spark.sql("""
SELECT
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_IND / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_NAO_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_FUT / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_OBR / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
FROM vw_agregacoes_financeiras a
CROSS JOIN vw_renda_derivada r
""").first()
percentuais_obtidos = spark.sql("""
SELECT PC_SAI_IND, PC_SAI_ESS, PC_SAI_NAO_ESS, PC_SAI_FUT, PC_SAI_OBR
FROM vw_percentuais_renda
""").first()
if percentuais_obtidos != percentuais_esperados:
    raise RuntimeError('[TESTE V4.1 FALHOU] Percentuais não refletem VL_REN_PRES ajustada.')


def pontuacao_concentracao(campo, pc, renda, quantidade):
    if quantidade is None or quantidade == 0 or renda is None:
        return None
    if renda <= 0:
        return 0
    if campo == 'IND':
        return 99 if pc > Decimal('0.750000') else 0
    limites = {
        'ESS': (Decimal('0.500000'), Decimal('0.750000'), 'crescente'),
        'NAO_ESS': (Decimal('0.300000'), Decimal('0.450000'), 'crescente'),
        'FUT': (Decimal('0.200000'), Decimal('0.300000'), 'decrescente'),
        'OBR': (Decimal('0.300000'), Decimal('0.450000'), 'crescente'),
    }
    baixo, alto, direcao = limites[campo]
    if direcao == 'decrescente':
        return 0 if pc >= alto else (1 if pc >= baixo else 2)
    return 0 if pc < baixo else (1 if pc < alto else 2)


pontuacoes = spark.sql("""
SELECT
    QT_TRANS_TOTAL, VL_REN_PRES,
    PC_SAI_IND, PC_SAI_ESS, PC_SAI_NAO_ESS, PC_SAI_FUT, PC_SAI_OBR,
    NR_PONT_CONC_IND, NR_PONT_CONC_ESS, NR_PONT_CONC_NAO_ESS,
    NR_PONT_CONC_FUT, NR_PONT_CONC_OBR,
    NR_PONT_ORC_ESS, NR_PONT_ORC_NAO_ESS, NR_PONT_ORC_FUT, NR_PONT_ORC_OBR,
    NR_PONT_PRFL_ESS, NR_PONT_PRFL_NAO_ESS, NR_PONT_PRFL_FUT, NR_PONT_PRFL_OBR,
    NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM,
    NR_PONT_FUT_FIM, NR_PONT_OBR_FIM,
    FL_PONTUACAO_COMPLETA, NR_PONT_MAX, QT_TEMAS_PONT_MAX,
    CD_TEMA_VENCEDOR, TX_TEMA_VENCEDOR
FROM vw_tema_vencedor
""").first()

for sufixo in ['IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR']:
    esperado = pontuacao_concentracao(
        sufixo,
        pontuacoes[f'PC_SAI_{sufixo}'],
        pontuacoes['VL_REN_PRES'],
        pontuacoes['QT_TRANS_TOTAL'],
    )
    if pontuacoes[f'NR_PONT_CONC_{sufixo}'] != esperado:
        raise RuntimeError(f'[TESTE V4.1 FALHOU] Pontuação de concentração divergente para {sufixo}.')

finais_esperados = {'IND': pontuacoes['NR_PONT_CONC_IND']}
for sufixo in ['ESS', 'NAO_ESS', 'FUT', 'OBR']:
    parcelas = [
        pontuacoes[f'NR_PONT_CONC_{sufixo}'],
        pontuacoes[f'NR_PONT_ORC_{sufixo}'],
        pontuacoes[f'NR_PONT_PRFL_{sufixo}'],
    ]
    finais_esperados[sufixo] = None if any(v is None for v in parcelas) else sum(parcelas)
for sufixo, esperado in finais_esperados.items():
    if pontuacoes[f'NR_PONT_{sufixo}_FIM'] != esperado:
        raise RuntimeError(f'[TESTE V4.1 FALHOU] Pontuação final divergente para {sufixo}.')

finais = [finais_esperados[s] for s in ['IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR']]
completa_esperada = 'S' if all(valor is not None for valor in finais) else 'N'
if pontuacoes['FL_PONTUACAO_COMPLETA'] != completa_esperada:
    raise RuntimeError('[TESTE V4.1 FALHOU] FL_PONTUACAO_COMPLETA divergente da nulabilidade das pontuações finais.')

if completa_esperada == 'N':
    fechamento_esperado = (None, None, None, None)
else:
    pontuacao_maxima = max(finais)
    quantidade_maxima = sum(valor == pontuacao_maxima for valor in finais)
    if quantidade_maxima > 1:
        fechamento_esperado = (pontuacao_maxima, quantidade_maxima, 9, 'Empate')
    else:
        indice = finais.index(pontuacao_maxima)
        nomes = [
            'Categorização dos Gastos', 'Gestão de Orçamento', 'Consumo Planejado',
            'Formação de Reserva', 'Uso Consciente do Crédito',
        ]
        fechamento_esperado = (pontuacao_maxima, 1, indice + 1, nomes[indice])
fechamento_obtido = (
    pontuacoes['NR_PONT_MAX'], pontuacoes['QT_TEMAS_PONT_MAX'],
    pontuacoes['CD_TEMA_VENCEDOR'], pontuacoes['TX_TEMA_VENCEDOR'],
)
if fechamento_obtido != fechamento_esperado:
    raise RuntimeError(
        f'[TESTE V4.1 FALHOU] Máximo/vencedor não reflete as pontuações finais: '
        f'esperado={fechamento_esperado}, obtido={fechamento_obtido}.'
    )

print('[TESTE V4.1] OK — renda por período, NULL, propagação e gate estrito V3 x V4 validados.')


### Teste V5.1 — Base única, alternância e preservação contratual


In [ ]:
%%spark

from decimal import Decimal


print('[TESTE V5.1] Validando a alternância da base financeira...')

# Fixture local: cobre inclusões, flags ignoradas e exclusões obrigatórias.
fixture_v5 = spark.sql("""
WITH mov AS (
    SELECT * FROM VALUES
      (CAST(1 AS BIGINT), 'C', 1,      'BRL', CAST(1000.00 AS DECIMAL(25,2))),
      (CAST(2 AS BIGINT), 'C', 448978, 'BRL', CAST( 200.00 AS DECIMAL(25,2))),
      (CAST(3 AS BIGINT), 'C', 300,    'BRL', CAST( 300.00 AS DECIMAL(25,2))),
      (CAST(4 AS BIGINT), 'C', 999999, 'BRL', CAST( 400.00 AS DECIMAL(25,2))),
      (CAST(5 AS BIGINT), 'C', 4,      'USD', CAST( 500.00 AS DECIMAL(25,2))),
      (CAST(6 AS BIGINT), 'C', 5,      'BRL', CAST( 600.00 AS DECIMAL(25,2))),
      (CAST(7 AS BIGINT), 'D', 448977, 'BRL', CAST( 700.00 AS DECIMAL(25,2))),
      (CAST(8 AS BIGINT), 'D', 279,    'BRL', CAST( 100.00 AS DECIMAL(25,2)))
    AS mov(ID, NTZ, CATEGORIA, MOEDA, VL)
),
removidas AS (
    SELECT * FROM VALUES (CAST(6 AS BIGINT)) AS removidas(ID)
),
catalogo AS (
    SELECT * FROM VALUES
      (1,      'C', 'S', 'S'),
      (448978, 'C', 'S', 'N'),
      (300,    'C', 'N', 'N'),
      (4,      'C', 'S', 'S'),
      (5,      'C', 'S', 'S'),
      (448977, 'D', 'S', 'N'),
      (279,    'D', 'S', 'S')
    AS catalogo(CATEGORIA, NTZ, PARTICIPA_CALCULO, PARTICIPA_ORCAMENTO)
),
efetivo AS (
    SELECT m.* FROM mov m LEFT ANTI JOIN removidas r ON m.ID = r.ID
),
classificado AS (
    SELECT e.*, c.PARTICIPA_CALCULO, c.PARTICIPA_ORCAMENTO
    FROM efetivo e
    INNER JOIN catalogo c
      ON e.CATEGORIA = c.CATEGORIA AND e.NTZ = c.NTZ
)
SELECT
    CAST(COALESCE(SUM(CASE WHEN NTZ = 'C' AND MOEDA = 'BRL' THEN VL END), 0.00) AS DECIMAL(25,2)) AS ENTRADAS_REALIZADAS,
    CAST(COALESCE(SUM(CASE WHEN ID = 2 AND NTZ = 'C' AND MOEDA = 'BRL' THEN VL END), 0.00) AS DECIMAL(25,2)) AS RESGATE,
    CAST(COALESCE(SUM(CASE WHEN ID = 3 AND NTZ = 'C' AND MOEDA = 'BRL' THEN VL END), 0.00) AS DECIMAL(25,2)) AS AGRO,
    CAST(COALESCE(SUM(CASE WHEN NTZ = 'C' AND MOEDA = 'BRL' AND PARTICIPA_CALCULO = 'N' THEN VL END), 0.00) AS DECIMAL(25,2)) AS CALCULO_N,
    CAST(COALESCE(SUM(CASE WHEN NTZ = 'C' AND MOEDA = 'BRL' AND PARTICIPA_ORCAMENTO = 'N' THEN VL END), 0.00) AS DECIMAL(25,2)) AS ORCAMENTO_N,
    CAST(COALESCE(SUM(CASE WHEN ID = 4 AND NTZ = 'C' AND MOEDA = 'BRL' THEN VL END), 0.00) AS DECIMAL(25,2)) AS SEM_CLASSIFICACAO,
    CAST(COALESCE(SUM(CASE WHEN ID = 5 AND NTZ = 'C' AND MOEDA = 'BRL' THEN VL END), 0.00) AS DECIMAL(25,2)) AS NAO_BRL,
    CAST(COALESCE(SUM(CASE WHEN ID = 6 AND NTZ = 'C' AND MOEDA = 'BRL' THEN VL END), 0.00) AS DECIMAL(25,2)) AS RECONCILIADA,
    CAST(COALESCE(SUM(CASE WHEN NTZ = 'D' AND MOEDA = 'BRL' AND PARTICIPA_ORCAMENTO = 'S' THEN VL END), 0.00) AS DECIMAL(25,2)) AS SAIDA_ORCAMENTARIA,
    CAST(COALESCE(SUM(CASE WHEN ID = 7 AND PARTICIPA_ORCAMENTO = 'S' THEN VL END), 0.00) AS DECIMAL(25,2)) AS APLICACAO_COMO_GASTO
FROM classificado
""").first().asDict(recursive=True)

esperado_fixture_v5 = {
    'ENTRADAS_REALIZADAS': Decimal('1500.00'),
    'RESGATE': Decimal('200.00'),
    'AGRO': Decimal('300.00'),
    'CALCULO_N': Decimal('300.00'),
    'ORCAMENTO_N': Decimal('500.00'),
    'SEM_CLASSIFICACAO': Decimal('0.00'),
    'NAO_BRL': Decimal('0.00'),
    'RECONCILIADA': Decimal('0.00'),
    'SAIDA_ORCAMENTARIA': Decimal('100.00'),
    'APLICACAO_COMO_GASTO': Decimal('0.00'),
}
if fixture_v5 != esperado_fixture_v5:
    raise RuntimeError(f'[TESTE V5.1 FALHOU] Fixture divergente: {fixture_v5}.')

# Ausência contratual não pode ser confundida com janela válida sem créditos.
casos_janela_v5 = spark.sql("""
SELECT CASO,
       CASE
         WHEN DT_INI IS NULL OR DT_FIM IS NULL THEN CAST(NULL AS DECIMAL(25,2))
         ELSE CAST(COALESCE(SOMA, CAST(0.00 AS DECIMAL(25,2))) AS DECIMAL(25,2))
       END AS BASE
FROM VALUES
  ('SEM_JANELA', CAST(NULL AS DATE), CAST(NULL AS DATE), CAST(NULL AS DECIMAL(25,2))),
  ('JANELA_VAZIA', DATE('2026-07-01'), DATE('2026-07-31'), CAST(NULL AS DECIMAL(25,2))),
  ('JANELA_COM_ENTRADAS', DATE('2026-07-01'), DATE('2026-07-31'), CAST(150.00 AS DECIMAL(25,2)))
AS casos(CASO, DT_INI, DT_FIM, SOMA)
ORDER BY CASO
""").collect()
obtido_janela_v5 = {row['CASO']: row['BASE'] for row in casos_janela_v5}
esperado_janela_v5 = {
    'SEM_JANELA': None,
    'JANELA_VAZIA': Decimal('0.00'),
    'JANELA_COM_ENTRADAS': Decimal('150.00'),
}
if obtido_janela_v5 != esperado_janela_v5:
    raise RuntimeError(f'[TESTE V5.1 FALHOU] Nulabilidade da janela divergente: {obtido_janela_v5}.')

sql_soma_v5 = (SQL_ENTRADAS_REALIZADAS_V5 + SQL_TOTAL_ENTRADAS_REALIZADAS_V5).upper()
if 'IN_PARTICIPA_CALCULO' in sql_soma_v5 or 'IN_PARTICIPA_ORCAMENTO' in sql_soma_v5:
    raise RuntimeError('[TESTE V5.1 FALHOU] A soma aplicou flag proibida.')
if 'INNER JOIN VW_CATEGORIAS' not in sql_soma_v5 or "CD_TIP_MOE_CRR = 'BRL'" not in sql_soma_v5:
    raise RuntimeError('[TESTE V5.1 FALHOU] Classificação válida ou BRL ausente da soma.')

esperado_real_v5 = spark.sql("""
WITH total AS (
    SELECT CAST(SUM(m.VL_TRAN) AS DECIMAL(25,2)) AS SOMA
    FROM vw_mov_efetivo m
    INNER JOIN vw_categorias c
      ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
     AND m.CD_NTZ_CTB_TRAN = c.TIPO
    WHERE m.CD_NTZ_CTB_TRAN = 'C' AND m.CD_TIP_MOE_CRR = 'BRL'
)
SELECT CASE
    WHEN j.DT_REF_INI IS NULL OR j.DT_REF_FIM IS NULL THEN CAST(NULL AS DECIMAL(25,2))
    ELSE CAST(COALESCE(t.SOMA, CAST(0.00 AS DECIMAL(25,2))) AS DECIMAL(25,2))
END AS VALOR
FROM vw_ciclo_janela j CROSS JOIN total t
""").first()['VALOR']
if valor_entradas_realizadas_v5 != esperado_real_v5:
    raise RuntimeError(
        f'[TESTE V5.1 FALHOU] Entradas reais divergentes: esperado={esperado_real_v5}, '
        f'obtido={valor_entradas_realizadas_v5}.'
    )

regras_especiais_v5 = spark.sql("""
SELECT
    MAX(CASE WHEN CD_CATEGORIA = 448978 AND TIPO = 'C' THEN CONCAT(IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO) END) AS RESGATE,
    MAX(CASE WHEN CD_CATEGORIA = 300 AND TIPO = 'C' THEN CONCAT(IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO) END) AS AGRO,
    MAX(CASE WHEN CD_CATEGORIA = 448977 AND TIPO = 'D' THEN CONCAT(IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO) END) AS APLICACAO
FROM vw_categorias
""").first()
if tuple(regras_especiais_v5) != ('SN', 'NN', 'SN'):
    raise RuntimeError(f'[TESTE V5.1 FALHOU] Regras especiais divergentes: {regras_especiais_v5}.')

aplicacao_orcamentaria_v5 = spark.sql("""
SELECT CAST(COALESCE(SUM(CASE WHEN IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VALOR
FROM vw_mov_brl
WHERE CD_NTZ_CTB_TRAN = 'D' AND CD_CTGR_TRAN_OGNL = 448977
""").first()['VALOR']
if aplicacao_orcamentaria_v5 != Decimal('0.00'):
    raise RuntimeError('[TESTE V5.1 FALHOU] Aplicação entrou nos gastos do ciclo.')

# Os dois modos executam a mesma função e os mesmos SQLs V4, sem regra por cenário.
if execucoes_cenarios_v5 != ['RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS']:
    raise RuntimeError(f'[TESTE V5.1 FALHOU] Execuções da cadeia divergentes: {execucoes_cenarios_v5}.')
for etapa, sql_fonte in SQL_CADEIA_V4_V5.items():
    hash_obtido = hashlib.sha256(sql_fonte.encode('utf-8')).hexdigest()
    if hash_obtido != SHA256_SQL_CADEIA_V4_V5[etapa]:
        raise RuntimeError(f'[TESTE V5.1 FALHOU] SQL V4 da etapa {etapa} foi alterado.')
    if 'ENTRADAS_REALIZADAS' in sql_fonte or 'RENDA_PRESUMIDA' in sql_fonte:
        raise RuntimeError(f'[TESTE V5.1 FALHOU] Etapa {etapa} contém metodologia específica por modo.')

janela_v5 = spark.table('vw_ciclo_janela').first()
renda_v5 = spark.table('vw_renda_derivada').first()['VL_REN_PRES']
base_presumida_esperada_v5 = (
    None if janela_v5['DT_REF_INI'] is None or janela_v5['DT_REF_FIM'] is None
    else Decimal(str(renda_v5)) if renda_v5 is not None else None
)
if bases_cenarios_v5['RENDA_PRESUMIDA'] != base_presumida_esperada_v5:
    raise RuntimeError('[TESTE V5.1 FALHOU] BASE_FINANCEIRA presumida não corresponde a VL_REN_PRES.')

for chave, prefixo in PREFIXOS_CENARIOS_V5.items():
    base = bases_cenarios_v5[chave]
    agregado_adaptado = spark.table(
        _nome_view_cenario_v5(prefixo, 'vw_agregacoes_financeiras')
    ).first()['VL_ENT_TOTAL']
    renda_adaptada = spark.table(
        _nome_view_cenario_v5(prefixo, 'vw_renda_derivada')
    ).first()['VL_REN_PRES']
    if agregado_adaptado != base or renda_adaptada != base:
        raise RuntimeError(f'[TESTE V5.1 FALHOU] Cenário {chave} não consumiu uma base única.')
    for campo_independente in [
        'NR_PONT_ORC_IND', 'NR_PONT_PRFL_IND', 'NR_PONT_PRFL_ESS',
        'NR_PONT_PRFL_NAO_ESS', 'NR_PONT_PRFL_FUT', 'NR_PONT_PRFL_OBR',
    ]:
        if calculos_cenarios_v5[chave]['resultado'][campo_independente] != resultado_oficial_v4_v5[campo_independente]:
            raise RuntimeError(
                f'[TESTE V5.1 FALHOU] Regra independente {campo_independente} '
                f'divergiu no cenário {chave}.'
            )

# Saídas, fatos e view oficial permanecem inalterados.
oficial_pos_v5 = spark.table(VIEW_RESULTADO)
if oficial_pos_v5.first().asDict(recursive=True) != resultado_oficial_v4_v5:
    raise RuntimeError('[TESTE V5.1 FALHOU] A view oficial V4 foi modificada pelo sidecar.')
if len(oficial_pos_v5.columns) != 80 or 'BASE_FINANCEIRA' in oficial_pos_v5.columns:
    raise RuntimeError('[TESTE V5.1 FALHOU] Contrato oficial deixou de ter as 80 colunas originais.')

campos_invariantes_v5 = set(resultado_oficial_v4_v5) - CAMPOS_VARIAVEIS_CENARIO_V5
for chave, snapshot in resultados_cenarios_v5.items():
    if dataframes_cenarios_v5[chave].schema != df_res_80.schema:
        raise RuntimeError(f'[TESTE V5.1 FALHOU] Schema lateral divergente em {chave}.')
    for campo in campos_invariantes_v5:
        if snapshot[campo] != resultado_oficial_v4_v5[campo]:
            raise RuntimeError(f'[TESTE V5.1 FALHOU] Campo invariante {campo} mudou em {chave}.')
for campo_saida in ['QT_TRANS_SAI', 'VL_TRANS_SAI', 'VL_SAI_IND', 'VL_SAI_ESS', 'VL_SAI_NAO_ESS', 'VL_SAI_FUT', 'VL_SAI_OBR', 'VL_SAI_TOTAL']:
    if resultados_cenarios_v5['RENDA_PRESUMIDA'][campo_saida] != resultados_cenarios_v5['ENTRADAS_REALIZADAS'][campo_saida]:
        raise RuntimeError(f'[TESTE V5.1 FALHOU] Saída divergente entre cenários: {campo_saida}.')

# O navegador recebe dois payloads imutáveis e apenas os alterna.
if set(payload_cenarios_v5) != {'RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS'}:
    raise RuntimeError('[TESTE V5.1 FALHOU] Payload não contém exatamente os dois modos.')
if payload_cenarios_v5['RENDA_PRESUMIDA']['valor_base'] != fmt_moeda(bases_cenarios_v5['RENDA_PRESUMIDA']):
    raise RuntimeError('[TESTE V5.1 FALHOU] Valor visual presumido divergente da base Spark.')
if payload_cenarios_v5['ENTRADAS_REALIZADAS']['valor_base'] != fmt_moeda(valor_entradas_realizadas_v5):
    raise RuntimeError('[TESTE V5.1 FALHOU] Valor visual realizado divergente da base Spark.')

marcadores_v5 = [
    'data-v5-component="base-financeira"',
    'data-v5-mode="RENDA_PRESUMIDA"',
    'data-v5-mode="ENTRADAS_REALIZADAS"',
    'aria-pressed="true" class="base-switch-btn active" data-v5-mode="RENDA_PRESUMIDA"',
    'view oficial V4 preservada',
    "applyScenario('RENDA_PRESUMIDA')",
    'applyScenario(button.dataset.v5Mode)',
    'Object.freeze(scenarioPayloads)',
]
ausentes_v5 = [item for item in marcadores_v5 if item not in html_dashboard_v3]
if ausentes_v5:
    raise RuntimeError(f'[TESTE V5.1 FALHOU] Marcadores visuais ausentes: {ausentes_v5}.')
if not re.search(r'<article class="context-card income-card"[^>]*>.*data-v5-mode="ENTRADAS_REALIZADAS"', html_dashboard_v3, re.S):
    raise RuntimeError('[TESTE V5.1 FALHOU] Controle não está dentro do card da base.')

inicio_toggle_v5 = html_dashboard_v3.index('  function applyScenario(mode)')
fim_toggle_v5 = html_dashboard_v3.index("  const privacyBtn=one('[data-role=\"privacy-button\"]');", inicio_toggle_v5)
toggle_js_v5 = html_dashboard_v3[inicio_toggle_v5:fim_toggle_v5].lower()
proibidos_toggle_v5 = ['spark', 'db2', 'fetch(', 'xmlhttprequest', 'round(', 'greatest(', 'case when']
encontrados_toggle_v5 = [item for item in proibidos_toggle_v5 if item in toggle_js_v5]
if encontrados_toggle_v5:
    raise RuntimeError(f'[TESTE V5.1 FALHOU] JavaScript contém consulta/regra: {encontrados_toggle_v5}.')

# Aplicar novamente o snapshot presumido é uma restauração, não transformação incremental.
if "const payload=scenarioPayloads[mode]" not in html_dashboard_v3:
    raise RuntimeError('[TESTE V5.1 FALHOU] Alternância não seleciona snapshot pré-calculado.')

print('[TESTE V5.1] OK — cadeia única, bases, nulabilidade, saídas, contrato e UI validados.')


### Teste V6.1 — Normalização textual da conta elegível


In [ ]:
%%spark

from decimal import Decimal


print('[TESTE V6.1] Validando normalização textual da conta elegível...')

view_fixture_v6 = 'vw_v6_teste_conta_origem'
view_resultado_v6 = 'vw_v6_teste_conta_normalizada'

try:
    spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW {view_fixture_v6} AS
    SELECT * FROM VALUES
      ('S', ' 3242 ',     '0000047949   '),
      ('S', '7',          '00000000000'),
      ('S', '32A2',       '47949'),
      ('S', '2147483648', '47949'),
      ('S', '3242',       '123456789012'),
      ('N', '3242',       '47949')
    AS casos(FL_CONTA_ELEGIVEL_UNICA, NR_AG_TITR, CD_CT_TITR)
    """)

    criar_view_conta_normalizada_v6(view_fixture_v6, view_resultado_v6)

    linhas_v6 = spark.table(view_resultado_v6).collect()
    obtido_v6 = {
        (r['FL_CONTA_ELEGIVEL_UNICA'], r['NR_AG_TITR'], r['CD_CT_TITR']):
        (r['CD_UOR_CC_NORM'], r['NR_CC_NORM'])
        for r in linhas_v6
    }
    esperado_v6 = {
        ('S', ' 3242 ',     '0000047949   '): (3242, Decimal('47949')),
        ('S', '7',          '00000000000'):   (7, Decimal('0')),
        ('S', '32A2',       '47949'):         (None, None),
        ('S', '2147483648', '47949'):         (None, None),
        ('S', '3242',       '123456789012'):  (None, None),
        ('N', '3242',       '47949'):         (None, None),
    }

    if obtido_v6 != esperado_v6:
        raise RuntimeError(
            f'[TESTE V6.1 FALHOU] Normalização divergente: '
            f'esperado={esperado_v6}, obtido={obtido_v6}.'
        )

    colunas_oficiais_v6 = [
        'FL_CONTA_ELEGIVEL_UNICA', 'NR_AG_TITR', 'CD_CT_TITR',
        'CD_UOR_CC_NORM', 'NR_CC_NORM',
    ]
    if spark.table('vw_conta_normalizada').columns != colunas_oficiais_v6:
        raise RuntimeError('[TESTE V6.1 FALHOU] Schema lógico da conta normalizada foi alterado.')

    assinatura_base_v6 = assinatura_estrutural_q5(SCHEMA_Q5_FUNCIONAL_V2)
    schema_transporte_v6 = StructType([
        StructField(
            campo.name,
            campo.dataType,
            False,
            {'origem': 'jdbc-db2', 'posicao': posicao},
        )
        for posicao, campo in enumerate(SCHEMA_Q5_FUNCIONAL_V2.fields)
    ])
    if assinatura_estrutural_q5(schema_transporte_v6) != assinatura_base_v6:
        raise RuntimeError(
            '[TESTE V6.1 FALHOU] Nullable ou metadata alteraram a assinatura funcional.'
        )

    campos_base_v6 = list(SCHEMA_Q5_FUNCIONAL_V2.fields)
    schemas_invalidos_v6 = {
        'nome': StructType([
            StructField('NR_TRAN_INST_PCT_ALTERADO', campos_base_v6[0].dataType, True),
            *campos_base_v6[1:],
        ]),
        'ordem': StructType(list(reversed(campos_base_v6))),
        'tipo': StructType([
            campos_base_v6[0],
            StructField('CD_CLI', StringType(), True),
            *campos_base_v6[2:],
        ]),
        'precisao': StructType([
            *campos_base_v6[:-1],
            StructField('VL_TRAN', DecimalType(16, 2), True),
        ]),
        'escala': StructType([
            *campos_base_v6[:-1],
            StructField('VL_TRAN', DecimalType(15, 3), True),
        ]),
    }
    aceitos_incorretamente_v6 = [
        nome
        for nome, schema in schemas_invalidos_v6.items()
        if assinatura_estrutural_q5(schema) == assinatura_base_v6
    ]
    if aceitos_incorretamente_v6:
        raise RuntimeError(
            f'[TESTE V6.1 FALHOU] Divergências estruturais aceitas: '
            f'{aceitos_incorretamente_v6}.'
        )

    if len(spark.table(VIEW_RESULTADO).columns) != 80:
        raise RuntimeError('[TESTE V6.1 FALHOU] Contrato oficial deixou de ter 80 colunas.')

    print('[TESTE V6.1] OK — conta, gate estrutural da Q5 e contrato final validados.')
finally:
    spark.catalog.dropTempView(view_resultado_v6)
    spark.catalog.dropTempView(view_fixture_v6)


### Teste V3.1 — Isolamento funcional e contrato visual


In [ ]:
%%spark

# Guard obrigatório: uma reexecução isolada após o cleanup não pode tocar a linhagem JDBC.
nivel_q5_v31 = df_q5_contexto_apresentacao.storageLevel
q5_v31_persistida = nivel_q5_v31.useMemory or nivel_q5_v31.useDisk or nivel_q5_v31.useOffHeap
if dt_ini_j is not None and not q5_v31_persistida:
    raise RuntimeError('[TESTE V3.1 BLOQUEADO] Q5 já liberada; reexecute o notebook a partir da Q5.')

def executar_validacoes_v31():
    print('[TESTE V3.1] Validando isolamento funcional da apresentação...')

    if spark.table('vw_q5_mov_contexto').columns != COLUNAS_Q5_FUNCIONAIS:
        raise RuntimeError('[TESTE V3.1 FALHOU] Projeção funcional da Q5 divergiu da v2.')

    schema_q5_v31 = spark.table('vw_q5_mov_contexto').schema
    if assinatura_estrutural_q5(schema_q5_v31) != assinatura_estrutural_q5(SCHEMA_Q5_FUNCIONAL_V2):
        raise RuntimeError(
            '[TESTE V3.1 FALHOU] Schema funcional da Q5 divergiu da v2. '
            f'obtido={schema_q5_v31.json()}; esperado={SCHEMA_Q5_FUNCIONAL_V2.json()}.'
        )

    campos_apresentacao = set(spark.table('vw_q5_mov_contexto_apresentacao').columns)
    if not {'TX_DCR_TRAN_OGNL', 'NR_MCA_PCT_OPB'}.issubset(campos_apresentacao):
        raise RuntimeError('[TESTE V3.1 FALHOU] Campos explicativos ausentes da sidecar.')

    for view_funcional in ['vw_mov_marcado', 'vw_mov_raw', 'vw_mov_efetivo', 'vw_mov_classificado', 'vw_mov_brl', 'vw_resultado_80_colunas']:
        colunas = set(spark.table(view_funcional).columns)
        if {'TX_DCR_TRAN_OGNL', 'NR_MCA_PCT_OPB'} & colunas:
            raise RuntimeError(f'[TESTE V3.1 FALHOU] Campo explicativo vazou para {view_funcional}.')

    # A réplica explicativa das transações efetivas deve coincidir com o universo
    # funcional, desconsiderando apenas ID e os dois campos informativos.
    divergencias = spark.sql("""
    WITH funcional AS (
      SELECT CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN, CD_CTGR_TRAN_OGNL,
             CD_TIP_MOE_CRR, VL_TRAN, CD_GRUPO, TX_GRUPO, TX_CATEGORIA,
             CD_IR, TX_IR, CD_CLASS_RADAR, TX_CLASS_RADAR, IN_AGRO,
             IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO
      FROM vw_mov_brl
    ),
    explicativa AS (
      SELECT CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN, CD_CTGR_TRAN_OGNL,
             CD_TIP_MOE_CRR, VL_TRAN, CD_GRUPO, TX_GRUPO, TX_CATEGORIA,
             CD_IR, TX_IR,
             COALESCE(CD_CLASS_RADAR, 0) AS CD_CLASS_RADAR,
             COALESCE(TX_CLASS_RADAR, 'Outras Entradas') AS TX_CLASS_RADAR,
             IN_AGRO,
             IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO
      FROM vw_dashboard_transacoes_efetivas
    ),
    funcional_menos_explicativa AS (
      SELECT * FROM funcional EXCEPT ALL SELECT * FROM explicativa
    ),
    explicativa_menos_funcional AS (
      SELECT * FROM explicativa EXCEPT ALL SELECT * FROM funcional
    ),
    diff AS (
      SELECT * FROM funcional_menos_explicativa
      UNION ALL
      SELECT * FROM explicativa_menos_funcional
    )
    SELECT COUNT(1) AS QT FROM diff
    """).first()['QT']

    if divergencias != 0:
        raise RuntimeError(f'[TESTE V3.1 FALHOU] Universo explicativo divergiu do funcional em {divergencias} linhas.')

    # A sidecar preserva a classificação bruta. Apenas o motor mantém seu fallback
    # histórico; a apresentação jamais transforma ausência em classe 0.
    divergencias_classificacao = spark.sql("""
    SELECT COUNT(1) AS QT
    FROM vw_dashboard_transacoes d
    LEFT JOIN vw_categorias c
      ON d.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
     AND d.CD_NTZ_CTB_TRAN = c.TIPO
    WHERE NOT (d.CD_CLASS_RADAR <=> c.CD_CLASS_RADAR)
       OR NOT (d.TX_CLASS_RADAR <=> c.TX_CLASS_RADAR)
       OR d.IN_CLASSIFICADA_APRESENTACAO <> CASE
            WHEN c.CD_CLASS_RADAR IS NOT NULL AND c.TX_CLASS_RADAR IS NOT NULL THEN 'S'
            ELSE 'N'
          END
    """).first()['QT']
    if divergencias_classificacao != 0:
        raise RuntimeError('[TESTE V3.1 FALHOU] Sidecar artificializou ou perdeu classificações do catálogo.')

    classe_zero_artificial = spark.sql("""
    SELECT COUNT(1) AS QT
    FROM vw_dashboard_transacoes
    WHERE IN_CLASSIFICADA_APRESENTACAO = 'N'
      AND (CD_CLASS_RADAR = 0 OR TX_CLASS_RADAR = 'Outras Entradas')
    """).first()['QT']
    if classe_zero_artificial != 0:
        raise RuntimeError('[TESTE V3.1 FALHOU] Linha sem classificação apresentada como classe 0.')

    casos_tratamento = [
        ({'IN_PARTICIPA_CALCULO': 'S', 'IN_PARTICIPA_ORCAMENTO': 'S', 'IN_CLASSIFICADA_APRESENTACAO': 'S'}, 'Tema + Orçamento'),
        ({'IN_PARTICIPA_CALCULO': 'S', 'IN_PARTICIPA_ORCAMENTO': 'N', 'IN_CLASSIFICADA_APRESENTACAO': 'S'}, 'Tema / Fora do orçamento'),
        ({'IN_PARTICIPA_CALCULO': 'N', 'IN_PARTICIPA_ORCAMENTO': 'S', 'IN_CLASSIFICADA_APRESENTACAO': 'S'}, 'Fora do tema / Orçamento'),
        ({'IN_PARTICIPA_CALCULO': 'N', 'IN_PARTICIPA_ORCAMENTO': 'N', 'IN_CLASSIFICADA_APRESENTACAO': 'S'}, 'Classificada / Fora tema e orçamento'),
        ({'IN_PARTICIPA_CALCULO': 'N', 'IN_PARTICIPA_ORCAMENTO': 'N', 'IN_CLASSIFICADA_APRESENTACAO': 'N'}, 'Sem classificação / Fora tema e orçamento'),
    ]
    for entrada, esperado in casos_tratamento:
        if rotulo_tratamento(entrada) != esperado:
            raise RuntimeError(f'[TESTE V3.1 FALHOU] Tratamento visual divergente: {entrada}.')

    if sum(int(r['qt']) for r in lista_dashboard_pivot) != len(lista_dashboard_transacoes):
        raise RuntimeError('[TESTE V3.1 FALHOU] Quantidade do pivot divergiu do detalhe transacional.')
    valor_pivot = sum((Decimal(str(r['vl_mov'])) for r in lista_dashboard_pivot), Decimal('0'))
    valor_detalhe = sum((Decimal(str(r['VL_TRAN'])) for r in lista_dashboard_transacoes), Decimal('0'))
    if valor_pivot != valor_detalhe:
        raise RuntimeError('[TESTE V3.1 FALHOU] Valor movimentado do pivot divergiu do detalhe.')

    casos_fechamento = [
        (
            {
                'FL_PONTUACAO_COMPLETA': 'S', 'NR_PONT_MAX': 99,
                'QT_TEMAS_PONT_MAX': 1, 'CD_TEMA_VENCEDOR': 1,
                'TX_TEMA_VENCEDOR': 'Categorização dos Gastos',
            },
            'vencedor', 'Tema de destaque',
        ),
        (
            {
                'FL_PONTUACAO_COMPLETA': 'S', 'NR_PONT_MAX': 4,
                'QT_TEMAS_PONT_MAX': 2, 'CD_TEMA_VENCEDOR': 9,
                'TX_TEMA_VENCEDOR': 'Empate',
            },
            'empate', 'Empate',
        ),
        (
            {
                'FL_PONTUACAO_COMPLETA': 'N', 'NR_PONT_MAX': None,
                'QT_TEMAS_PONT_MAX': None, 'CD_TEMA_VENCEDOR': None,
                'TX_TEMA_VENCEDOR': None,
            },
            'incompleta', 'Pontuação incompleta',
        ),
    ]
    for entrada, estado, marcador in casos_fechamento:
        resolvido = resolver_fechamento_pontuacao(entrada)
        fechamento_html = render_fechamento_pontuacao(entrada)
        if resolvido['estado'] != estado or marcador not in fechamento_html:
            raise RuntimeError(f'[TESTE V3.1 FALHOU] Fechamento visual divergente para {estado}.')
        if estado != 'vencedor' and 'Tema de destaque' in fechamento_html:
            raise RuntimeError(f'[TESTE V3.1 FALHOU] Vencedor artificial no estado {estado}.')

    casos_fechamento_invalidos = [
        {'FL_PONTUACAO_COMPLETA': 'N', 'NR_PONT_MAX': 1, 'QT_TEMAS_PONT_MAX': None, 'CD_TEMA_VENCEDOR': None, 'TX_TEMA_VENCEDOR': None},
        {'FL_PONTUACAO_COMPLETA': 'S', 'NR_PONT_MAX': 4, 'QT_TEMAS_PONT_MAX': 2, 'CD_TEMA_VENCEDOR': 1, 'TX_TEMA_VENCEDOR': 'Categorização dos Gastos'},
        {'FL_PONTUACAO_COMPLETA': 'S', 'NR_PONT_MAX': 4, 'QT_TEMAS_PONT_MAX': 1, 'CD_TEMA_VENCEDOR': 1, 'TX_TEMA_VENCEDOR': 'Empate'},
    ]
    for entrada in casos_fechamento_invalidos:
        try:
            resolver_fechamento_pontuacao(entrada)
        except RuntimeError:
            pass
        else:
            raise RuntimeError(f'[TESTE V3.1 FALHOU] Estado final inconsistente foi aceito: {entrada}.')

    marcadores_html = [
        'data-radar-root="v3"', 'class="privacy-on"',
        'data-role="privacy-button"', 'data-role="export-button"',
        'data-section="composicao"', 'data-section="pontuacao"',
        'data-section="fechamento-pontuacao"',
        'data-section="reconciliacao"', 'data-section="auditoria"',
        'TX_DCR_TRAN_OGNL', 'NR_MCA_PCT_OPB',
        'FL_PONTUACAO_COMPLETA', 'NR_PONT_MAX', 'QT_TEMAS_PONT_MAX',
        'CD_TEMA_VENCEDOR', 'TX_TEMA_VENCEDOR',
        'IND: Final = Concentração',
        'Demais temas: Final = Concentração + Orçamento + Perfil',
        'HTML já salvo no workdir', 'dataset.radarBound',
        '80 atributos',
    ]
    ausentes = [m for m in marcadores_html if m not in html_dashboard_v3]
    if ausentes:
        raise RuntimeError(f'[TESTE V3.1 FALHOU] Marcadores ausentes do HTML: {ausentes}')

    if lista_dashboard_transacoes and '<th>Tratamento</th>' not in html_dashboard_v3:
        raise RuntimeError('[TESTE V3.1 FALHOU] Coluna Tratamento ausente do detalhe transacional.')

    ids_html = re.findall(r'\bid="[^"]+"', html_dashboard_v3)
    if len(ids_html) != 1 or f'id="{dashboard_root_id}"' not in html_dashboard_v3:
        raise RuntimeError(f'[TESTE V3.1 FALHOU] Contrato de raiz exclusiva violado: {ids_html}.')

    proibidos_html = [
        'new Blob', 'showSaveFilePicker', ' download=', 'document.querySelector',
        'eval(', 'new Function', '<script src=', '<link href=', '@import', ':root{',
    ]
    encontrados = [item for item in proibidos_html if item in html_dashboard_v3]
    if encontrados:
        raise RuntimeError(f'[TESTE V3.1 FALHOU] Marcadores globais/externos proibidos: {encontrados}.')

    if not CSS_APROVADO_V3.startswith('[data-radar-root="v3"]'):
        raise RuntimeError('[TESTE V3.1 FALHOU] CSS não inicia no escopo exclusivo da raiz.')

    print('[TESTE V3.1] OK — motor isolado, Q5 única, classificação, tratamento, fechamento, raiz, eventos e HTML validados.')

try:
    executar_validacoes_v31()
finally:
    if dt_ini_j is not None and df_q5_contexto_apresentacao is not None:
        df_q5_contexto_apresentacao.unpersist(blocking=True)
        print('[TESTE V3.1] Cache materializado da Q5 liberado após a última validação.')
